# Reversible Data Hiding in Encrypted Images via Multi-Band Integer Wavelet Embedding and a Bounded Lozi–Chebyshev Chaotic Map — all results of the paper

This notebook regenerates **every table (1–22) and every figure (1–22)** of the paper, plus the numbers quoted in the text (key space, Lyapunov scan, cycle search, attack results, ordering gain, …). The scheme itself is the reference implementation used for the paper, embedded unchanged.

**Input — choose one in Section 1**

| `INPUT_MODE` | What is processed |
|---|---|
| `'paper'` | the 79 test images of the paper (5 USC-SIPI, 10 standard, 24 Kodak, 40 chest X-ray/CT), downloaded from their public sources at pinned versions and checked against checksums, so every number is reproduced exactly |
| `'single'` | one image: `IMAGE_PATH` (or upload it from your computer) |
| `'folder'` | every image in `FOLDER_PATH`; each sub-folder is treated as one image group (e.g. `images/Natural/`, `images/Medical/`), like the four groups of the paper |

Colour images are converted to 8-bit grey (ITU-R BT.601). Tables that the paper reports "for the USC-SIPI images" are computed on the USC-SIPI group in `'paper'` mode and on your images otherwise.

**Run:** *Runtime → Run all*. Every table is shown and saved as CSV, every figure as PNG (300 dpi), all under `OUTPUT_DIR`; the last cell zips everything.

**Run time on a standard Colab CPU** (`'paper'` mode, all switches on): about 2–3 h, of which NIST SP 800-22 ≈ 45 min and the 79-image capacity/rate–distortion study ≈ 1 h. A single 512 × 512 image takes ≈ 5 min without NIST. Switch parts off in `RUN`, or set `QUICK = True` for a fast check (fewer levels, rates and keys).

**Reproduction status.** All results of the scheme (capacity, rate–distortion, comparisons, security statistics, key sensitivity, key separation, attacks, cycles, NIST) use the paper's fixed seeds, keys and nonces and reproduce the published values exactly in `'paper'` mode. Three quantities are statistical and are recomputed with a documented protocol: the two χ²/df values of Fig. 10 (they vary with the orbit by about ±0.14 and ±0.09) and the equivalent-key count *y₀ ≡ y₀ + 2* (224 of 300 random keys in the paper). Table 3 and Fig. 11 are computed from the same 201-point sweeps (ranges listed in the table), and the values printed in the revised manuscript are checked in Section 15.

**Additions for the revision (reviewer comments 1 and 2, and the revised Section 3.3.7)**
- *Sec. 4.2, fall-off-boundary correction (comment 1)*: Section 7 contains a check that the stored correction is the non-negative magnitude r = |I − I_p|, that its direction follows from I_p = D or I_p = 255 − D, and that the cover is restored exactly, including on a synthetic image with large black and white regions.
- *Eq. (20), key-byte conversion (comment 2)*: Section 7 contains a check that the single function used for the cipher key-streams, the round seeds, the data-hiding mask and the NIST sequences equals ⌊|p_n|·2³²⌋ mod 256.
- *Sec. 3.3.7*: Section 5 prints the bound λ₁ ≥ ½ ln|b| that explains why the 2D-LCM has no non-chaotic window, and the entropy rate of the map in bits per iteration (Table 2).

**Two points where the notebook differs from the manuscript on purpose**
- *Fig. 3(b)*: (a, b) = (1.5, 0.5) lies exactly on the border b = a − 1 between the period-2 and chaotic regions, so the orbit there is periodic. The V-shaped attractor is the one at (1.7, 0.5), the parameters of Fig. 6(c) and Table 2; the revised manuscript now states (1.7, 0.5) in the caption and in Section 3.2, and this is what the notebook draws.
- *Table 19, local entropy*: the text of Sec. 5.7 specifies 30 random **non-overlapping** 44 × 44 blocks. The revised manuscript tabulates the non-overlapping values (mean 7.9027); the notebook reports both that column and the earlier one computed with random block positions that may overlap (mean 7.9029).

## 1. Settings

In [ ]:
# ================================ INPUT ================================
INPUT_MODE  = 'paper'                     # 'paper' | 'single' | 'folder'
IMAGE_PATH  = '/content/Boat.png'         # used when INPUT_MODE == 'single'
FOLDER_PATH = '/content/images'           # used when INPUT_MODE == 'folder' (sub-folders = image groups)
RECURSIVE   = True                        # 'folder': also read sub-folders (each one becomes a group)
EXTENSIONS  = ('.png', '.bmp', '.tif', '.tiff', '.jpg', '.jpeg', '.pgm', '.ppm')
MOUNT_GOOGLE_DRIVE   = False              # True: mount Drive, then use paths such as '/content/drive/MyDrive/...'
UPLOAD_FROM_COMPUTER = False              # True with 'single': opens a file picker
RESIZE_TO_512 = False                     # True: centre-square crop and resize to 512 x 512 as in the paper
                                          # (always applied in 'paper' mode); otherwise the size is kept (odd sizes cropped by 1)
OUTPUT_DIR  = '/content/rdhei_paper_results'

# ================================ SECRET IMAGE ================================
SECRET_IMAGE_PATH = None                  # None: the paper's 256 x 256 binarised Cameraman (65,536 bits, downloaded)

# ================================ KEYS (paper) ================================
KE_TUPLE = (0.3, 0.6, 5.0, 20.7, 2.3)     # encryption key  K_e = (x0, y0, a, b, w)
KD_TUPLE = (-0.41, 0.27, 6.1, 18.3, 3.7)  # data-hiding key K_d

# ================================ WHAT TO RUN ================================
RUN = dict(
    iwt=True,           # Sec. 3.1: Figs. 1-2, Table 1
    chaos=True,         # Sec. 3.2-3.3: Figs. 3-11, Tables 2-3, admissible domain, equivalent keys
    nist=True,          # Sec. 3.4: Tables 4-5 (NIST STS 2.1.2, three runs of 100 x 10^6 bits, ~45 min)
    scheme=True,        # Sec. 4: Fig. 12, Tables 6-7, ordering gain
    keyspace=True,      # Sec. 5.2: Tables 8-10, Eq. (30), Fig. 14
    capacity=True,      # Sec. 5.3: Tables 12-13, Fig. 15
    rate_distortion=True,  # Sec. 5.4: Table 14, Figs. 16-17
    comparison=True,    # Sec. 5.5: Tables 11, 15, 16, 17, Figs. 18-19
    security=True,      # Sec. 5.6-5.7: Tables 18-22, Figs. 13, 20, 21
    attacks=True,       # Sec. 5.7.6: Fig. 22, affinity and key-reuse tests
    cycles=True,        # Sec. 5.7.7: Brent cycle search (float64 / float32)
)
QUICK = False               # True: fewer levels/rates/keys/iterations for a fast check (numbers then differ from the paper)
CYCLE_ITERATIONS = 2 ** 31  # paper: 2.15e9 iterations per orbit (about 2-3 min per orbit); QUICK uses 2**26
N_JOBS = None               # parallel workers for the per-image experiments (None = number of CPU cores)

## 2. Environment and reference implementation

In [ ]:
import os, sys, glob, json, math, time, zlib, ctypes, subprocess, hashlib, warnings, itertools, tracemalloc, io, shutil
warnings.filterwarnings('ignore')
def _pip(*p): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *p], check=False)
try: import skimage, imageio, scipy, tifffile
except ImportError: _pip('scikit-image', 'imageio', 'scipy', 'tifffile')
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown
IN_COLAB = 'google.colab' in sys.modules
if MOUNT_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
if QUICK: CYCLE_ITERATIONS = min(CYCLE_ITERATIONS, 2 ** 26)
N_JOBS = N_JOBS or os.cpu_count() or 1
OUT = os.path.abspath(OUTPUT_DIR); FIGD = f'{OUT}/figures'; TABD = f'{OUT}/tables'; RES = f'{OUT}/raw'
for d in (OUT, FIGD, TABD, RES): os.makedirs(d, exist_ok=True)
PKG = os.path.abspath(os.path.join(OUT, '..', 'rdhei_pkg')); os.makedirs(PKG, exist_ok=True)
plt.rcParams.update({'font.family': 'serif', 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.6, 'lines.linewidth': 1.6,
                     'savefig.dpi': 300, 'savefig.bbox': 'tight', 'legend.frameon': False, 'figure.max_open_warning': 0})
COL = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']; MK = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']
print('Python', sys.version.split()[0], '| NumPy', np.__version__, '| CPU cores', os.cpu_count(), '| output ->', OUT)

In [ ]:
PACKAGE_FILES = {'lcm_core.c': '/* Fast kernels for the 2D Lozi-Chebyshev map (2D-LCM) and the chained cipher.\n * Bounded ("fold") realisation used throughout the revised manuscript:\n *     x_{n+1} = W( 1 - a*|cos(w*acos(x_n))| + y_n ),   W(u) = u+1 - 2*floor((u+1)/2) - 1  (fold onto [-1,1))\n *     y_{n+1} = b * x_n\n *     p_{n+1} = W( x_{n+1} + y_{n+1} )\n * Compile: gcc -O2 -shared -fPIC -o liblcm.so lcm_core.c -lm      (no -ffast-math: IEEE-754 semantics required)\n */\n#include <math.h>\n#include <stdint.h>\n#include <string.h>\n\nstatic inline double wrap11(double u) { double v = u + 1.0; return v - 2.0 * floor(v / 2.0) - 1.0; }\n\nstatic inline void step(double *x, double *y, double a, double b, double w) {\n    double xc = *x, yc = *y;\n    double xg = xc > 1.0 ? 1.0 : (xc < -1.0 ? -1.0 : xc);\n    double xn = wrap11(1.0 - a * fabs(cos(w * acos(xg))) + yc);\n    *y = b * xc; *x = xn;\n}\n\n/* Key schedule: absorb an optional nonce (nb bytes) into the initial state, then discard `transient`\n * iterates. Absorption of byte u: x <- W(x + (u+1)/257), followed by 8 map iterations. */\nvoid lcm_init(double *x, double *y, double a, double b, double w,\n              const uint8_t *nonce, int nb, int64_t transient) {\n    for (int j = 0; j < nb; j++) {\n        *x = wrap11(*x + ((double)nonce[j] + 1.0) / 257.0);\n        for (int r = 0; r < 8; r++) step(x, y, a, b, w);\n    }\n    for (int64_t k = 0; k < transient; k++) step(x, y, a, b, w);\n}\n\n/* Generate n iterates (x, y, p) starting from state (x0,y0) (state is NOT advanced past the output). */\nvoid lcm_gen(double x0, double y0, double a, double b, double w, int64_t n,\n             double *xs, double *ys, double *ps) {\n    double x = x0, y = y0;\n    for (int64_t k = 0; k < n; k++) {\n        step(&x, &y, a, b, w);\n        xs[k] = x; ys[k] = y; ps[k] = wrap11(x + y);\n    }\n}\n\n/* Byte keystream from p: floor(|p| * 2^32) mod 256 (bits 25..32 below the binary point). */\nvoid bytes_from_p(const double *p, int64_t n, uint8_t *out) {\n    for (int64_t k = 0; k < n; k++) out[k] = (uint8_t)(((uint64_t)floor(fabs(p[k]) * 4294967296.0)) & 0xFF);\n}\n\n/* One cipher round: c_t = ((v_t + c_{t-1}) mod 256) XOR k_t,   c_{-1} = s. */\nvoid chain_fwd(const uint8_t *v, const uint8_t *k, uint8_t s, int64_t n, uint8_t *c) {\n    uint8_t prev = s;\n    for (int64_t t = 0; t < n; t++) { prev = (uint8_t)((uint8_t)(v[t] + prev) ^ k[t]); c[t] = prev; }\n}\nvoid chain_inv(const uint8_t *c, const uint8_t *k, uint8_t s, int64_t n, uint8_t *v) {\n    uint8_t prev = s;\n    for (int64_t t = 0; t < n; t++) { v[t] = (uint8_t)((c[t] ^ k[t]) - prev); prev = c[t]; }\n}\n/* Legacy (first-submission code) round: c_t = (v_t + k_t + c_{t-1}) mod 256  -- affine over Z_256. */\nvoid chain_fwd_legacy(const uint8_t *v, const uint8_t *k, uint8_t s, int64_t n, uint8_t *c) {\n    uint8_t prev = s;\n    for (int64_t t = 0; t < n; t++) { prev = (uint8_t)(v[t] + k[t] + prev); c[t] = prev; }\n}\nvoid chain_inv_legacy(const uint8_t *c, const uint8_t *k, uint8_t s, int64_t n, uint8_t *v) {\n    uint8_t prev = s;\n    for (int64_t t = 0; t < n; t++) { v[t] = (uint8_t)(c[t] - k[t] - prev); prev = c[t]; }\n}\n\n/* Brent cycle detection on the exact double-precision state (x,y), up to `maxit` iterations\n * after `transient`. Returns cycle length lambda (0 if none found) and writes mu-estimate bound. */\nint64_t lcm_cycle(double x0, double y0, double a, double b, double w, int64_t transient, int64_t maxit) {\n    double x = x0, y = y0;\n    for (int64_t k = 0; k < transient; k++) step(&x, &y, a, b, w);\n    double tx = x, ty = y;            /* tortoise */\n    double hx = x, hy = y; step(&hx, &hy, a, b, w);\n    int64_t power = 1, lam = 1, it = 0;\n    while (!(tx == hx && ty == hy)) {\n        if (power == lam) { tx = hx; ty = hy; power <<= 1; lam = 0; }\n        step(&hx, &hy, a, b, w); lam++; it++;\n        if (it > maxit) return 0;\n    }\n    return lam;\n}\n\n/* Same map evaluated in single precision (float32) to illustrate finite-precision degradation. */\nstatic inline float wrap11f(float u) { float v = u + 1.0f; return v - 2.0f * floorf(v / 2.0f) - 1.0f; }\nint64_t lcm_cycle_f32(float x0, float y0, float a, float b, float w, int64_t transient, int64_t maxit) {\n    float x = x0, y = y0, xn;\n    #define STEPF(X,Y) { float xc=X, xg = xc>1.f?1.f:(xc<-1.f?-1.f:xc); xn = wrap11f(1.f - a*fabsf(cosf(w*acosf(xg))) + Y); Y = b*xc; X = xn; }\n    for (int64_t k = 0; k < transient; k++) STEPF(x, y);\n    float tx = x, ty = y, hx = x, hy = y; STEPF(hx, hy);\n    int64_t power = 1, lam = 1, it = 0;\n    while (!(tx == hx && ty == hy)) {\n        if (power == lam) { tx = hx; ty = hy; power <<= 1; lam = 0; }\n        STEPF(hx, hy); lam++; it++;\n        if (it > maxit) return 0;\n    }\n    return lam;\n}\n\n/* Largest + second Lyapunov exponents of the fold map via QR (Gram-Schmidt) on the analytic Jacobian.\n * d/dx |cos(w acos x)| = sign(cos(w acos x)) * w sin(w acos x) / sqrt(1-x^2).  J = [[-a*that, 1],[b, 0]]. */\nvoid lcm_lyap(double x0, double y0, double a, double b, double w, int64_t n, int64_t transient, double *l1, double *l2) {\n    double x = x0, y = y0;\n    for (int64_t k = 0; k < transient; k++) step(&x, &y, a, b, w);\n    double q11 = 1, q21 = 0, q12 = 0, q22 = 1, s1 = 0, s2 = 0;\n    for (int64_t k = 0; k < n; k++) {\n        double xg = x > 1.0 ? 1.0 : (x < -1.0 ? -1.0 : x);\n        double th = acos(xg), c = cos(w * th);\n        double den = sqrt(fmax(1e-300, 1.0 - xg * xg));\n        double dT = (c >= 0 ? 1.0 : -1.0) * w * sin(w * th) / den;\n        double j11 = -a * dT, j12 = 1.0, j21 = b, j22 = 0.0;\n        double z11 = j11 * q11 + j12 * q21, z21 = j21 * q11 + j22 * q21;\n        double z12 = j11 * q12 + j12 * q22, z22 = j21 * q12 + j22 * q22;\n        double r11 = sqrt(z11 * z11 + z21 * z21); q11 = z11 / r11; q21 = z21 / r11;\n        double r12 = q11 * z12 + q21 * z22; z12 -= r12 * q11; z22 -= r12 * q21;\n        double r22 = sqrt(z12 * z12 + z22 * z22); q12 = z12 / r22; q22 = z22 / r22;\n        s1 += log(r11); s2 += log(r22);\n        step(&x, &y, a, b, w);\n    }\n    *l1 = s1 / n; *l2 = s2 / n;\n}\n', 'rdhei.py': '"""\nReference implementation of the REVISED scheme:\n  Reversible data hiding in encrypted images via multi-band integer wavelet embedding and a bounded\n  Lozi-Chebyshev chaotic map (2D-LCM).\n\nPipeline (content owner / sender)\n  1. FOB pre-processing : Ip = clip(I, D, 255-D); corrections of the "ambiguous" pixels {Ip==D or Ip==255-D}\n                          are arithmetic-coded into the FOB map B_F (embedded in the image, never out-of-band).\n  2. 5/3 IWT            : (ll, hl, lh, hh) = IWT(Ip).\n  3. Embedding order    : detail coefficients of HL, LH, HH ordered by the local complexity of the (unchanged) LL band.\n  4. Multi-level expansion/shift (level L, blind & reversible), header first at level 1:\n                          c in [-L, L-1] -> 2c + b ;  c >= L -> c + L ;  c < -L -> c - L.\n                          Stage 1 (level 1): H (76 bits) || B_E;  stage 2 (level L): B_F || message masked with K_d.\n  5. Marked image       : Im = IIWT(ll, hl\', lh\', hh\')  (in [0,255] by construction; checked).\n  6. Encryption (Ke, public 128-bit nonce N): 3 rounds of  c_t = ((v_t + c_{t-1}) mod 256) XOR k_t  with two chaotic\n                          rank permutations in between; all key-streams from the nonce-keyed 2D-LCM.\nReceiver: exact inverse. Ke alone -> marked image AND exact cover; Ke+Kd -> message as well.\n"""\nimport ctypes, os, numpy as np\n\n_lib = ctypes.CDLL(os.path.join(os.path.dirname(os.path.abspath(__file__)), \'liblcm.so\'))\n_D = ctypes.c_double; _I64 = ctypes.c_int64\n_PD = np.ctypeslib.ndpointer(np.float64, flags=\'C\'); _PU8 = np.ctypeslib.ndpointer(np.uint8, flags=\'C\')\n_lib.lcm_init.argtypes = [ctypes.POINTER(_D), ctypes.POINTER(_D), _D, _D, _D, ctypes.c_char_p, ctypes.c_int, _I64]\n_lib.lcm_gen.argtypes = [_D, _D, _D, _D, _D, _I64, _PD, _PD, _PD]\n_lib.bytes_from_p.argtypes = [_PD, _I64, _PU8]\nfor f in (\'chain_fwd\', \'chain_inv\', \'chain_fwd_legacy\', \'chain_inv_legacy\'):\n    getattr(_lib, f).argtypes = [_PU8, _PU8, ctypes.c_uint8, _I64, _PU8]\n_lib.lcm_cycle.argtypes = [_D, _D, _D, _D, _D, _I64, _I64]; _lib.lcm_cycle.restype = _I64\n_lib.lcm_cycle_f32.argtypes = [ctypes.c_float] * 5 + [_I64, _I64]; _lib.lcm_cycle_f32.restype = _I64\n_lib.lcm_lyap.argtypes = [_D, _D, _D, _D, _D, _I64, _I64, ctypes.POINTER(_D), ctypes.POINTER(_D)]\n\nTRANSIENT = 2000\n\n# ----------------------------------------------------------------------------- 2D-LCM\nclass Key:\n    """Secret key K = (x0, y0, a, b, w) of the bounded 2D-LCM."""\n    def __init__(self, x0, y0, a, b, w): self.x0, self.y0, self.a, self.b, self.w = map(float, (x0, y0, a, b, w))\n    def tup(self): return (self.x0, self.y0, self.a, self.b, self.w)\n    def replace(self, **kw):\n        d = dict(zip((\'x0\', \'y0\', \'a\', \'b\', \'w\'), self.tup())); d.update(kw); return Key(**d)\n\ndef lcm_sequence(K, n, nonce=b\'\', transient=TRANSIENT):\n    x = _D(K.x0); y = _D(K.y0)\n    _lib.lcm_init(ctypes.byref(x), ctypes.byref(y), K.a, K.b, K.w, nonce, len(nonce), transient)\n    xs = np.empty(n); ys = np.empty(n); ps = np.empty(n)\n    _lib.lcm_gen(x.value, y.value, K.a, K.b, K.w, n, xs, ys, ps)\n    return xs, ys, ps\n\ndef p_bytes(p):\n    out = np.empty(len(p), np.uint8); _lib.bytes_from_p(np.ascontiguousarray(p), len(p), out); return out\n\ndef lyapunov(K, n=20000, transient=TRANSIENT):\n    l1 = _D(); l2 = _D(); _lib.lcm_lyap(K.x0, K.y0, K.a, K.b, K.w, n, transient, ctypes.byref(l1), ctypes.byref(l2))\n    return l1.value, l2.value\n\n# ----------------------------------------------------------------------------- 5/3 integer lifting (JPEG2000 reversible)\ndef _lift53(X):                      # along axis 0\n    Xe = X[0::2]; Xo = X[1::2]\n    Xe_n = np.concatenate([Xe[1:], Xe[-1:]], 0)\n    D = Xo - np.floor_divide(Xe + Xe_n, 2)\n    D_p = np.concatenate([D[:1], D[:-1]], 0)\n    S = Xe + np.floor_divide(D_p + D + 2, 4)\n    return S, D\n\ndef _ilift53(S, D):\n    D_p = np.concatenate([D[:1], D[:-1]], 0)\n    Xe = S - np.floor_divide(D_p + D + 2, 4)\n    Xe_n = np.concatenate([Xe[1:], Xe[-1:]], 0)\n    Xo = D + np.floor_divide(Xe + Xe_n, 2)\n    X = np.empty((2 * S.shape[0],) + S.shape[1:], dtype=np.int64); X[0::2] = Xe; X[1::2] = Xo\n    return X\n\ndef iwt53(X):\n    """Returns ll, hl, lh, hh (columns first, then rows; identical ordering to the released MATLAB code)."""\n    S, D = _lift53(np.asarray(X, np.int64))\n    ll, hl = (t.T for t in _lift53(S.T)); lh, hh = (t.T for t in _lift53(D.T))\n    return ll, hl, lh, hh\n\ndef iiwt53(ll, hl, lh, hh):\n    S = _ilift53(ll.T, hl.T).T; D = _ilift53(lh.T, hh.T).T\n    return _ilift53(S, D)\n\n# ----------------------------------------------------------------------------- adaptive arithmetic coder (32-bit, WNC)\nclass _AC:\n    TOP = (1 << 32) - 1; HALF = 1 << 31; Q1 = 1 << 30; Q3 = 3 << 30\n\ndef ac_encode(symbols, nsym):\n    lo, hi, pend, out = 0, _AC.TOP, 0, []\n    freq = [1] * nsym; tot = nsym\n    def emit(b):\n        nonlocal pend\n        out.append(b); out.extend([1 - b] * pend); pend = 0\n    for s in symbols:\n        rng = hi - lo + 1; cum = sum(freq[:s])\n        hi = lo + rng * (cum + freq[s]) // tot - 1; lo = lo + rng * cum // tot\n        while True:\n            if hi < _AC.HALF: emit(0)\n            elif lo >= _AC.HALF: emit(1); lo -= _AC.HALF; hi -= _AC.HALF\n            elif lo >= _AC.Q1 and hi < _AC.Q3: pend += 1; lo -= _AC.Q1; hi -= _AC.Q1\n            else: break\n            lo = 2 * lo; hi = 2 * hi + 1\n        freq[s] += 1; tot += 1\n        if tot > 65535: freq = [(f + 1) // 2 for f in freq]; tot = sum(freq)\n    pend += 1; emit(0 if lo < _AC.Q1 else 1)\n    return np.array(out, np.uint8)\n\ndef ac_decode(bits, n, nsym):\n    bits = list(map(int, bits)); pos = 0\n    def nb():\n        nonlocal pos\n        b = bits[pos] if pos < len(bits) else 0; pos += 1; return b\n    lo, hi, val = 0, _AC.TOP, 0\n    for _ in range(32): val = 2 * val + nb()\n    freq = [1] * nsym; tot = nsym; out = []\n    for _ in range(n):\n        rng = hi - lo + 1; tgt = ((val - lo + 1) * tot - 1) // rng\n        cum = 0; s = 0\n        while cum + freq[s] <= tgt: cum += freq[s]; s += 1\n        hi = lo + rng * (cum + freq[s]) // tot - 1; lo = lo + rng * cum // tot\n        while True:\n            if hi < _AC.HALF: pass\n            elif lo >= _AC.HALF: lo -= _AC.HALF; hi -= _AC.HALF; val -= _AC.HALF\n            elif lo >= _AC.Q1 and hi < _AC.Q3: lo -= _AC.Q1; hi -= _AC.Q1; val -= _AC.Q1\n            else: break\n            lo = 2 * lo; hi = 2 * hi + 1; val = 2 * val + nb()\n        out.append(s); freq[s] += 1; tot += 1\n        if tot > 65535: freq = [(f + 1) // 2 for f in freq]; tot = sum(freq)\n    return np.array(out, np.int64)\n\n# ----------------------------------------------------------------------------- helpers\ndef int2bits(v, n): return np.array([(int(v) >> (n - 1 - k)) & 1 for k in range(n)], np.uint8)\ndef bits2int(b): return int(\'\'.join(str(int(x)) for x in b), 2) if len(b) else 0\n\n# ----------------------------------------------------------------------------- FOB handling\n# Coefficient (m,n) of any detail band can only alter pixels in rows/cols [2m-1, 2m+3] x [2n-1, 2n+3] (measured\n# support of the 5/3 synthesis); one pixel of safety margin is added on each side.\nINF_LO, INF_HI = -2, 4\nBLK = 8                       # exclusion-map granularity: 8x8 coefficient positions = 16x16 pixels\nSAT_T = 16                    # ordering only: positions whose 3x3 LL neighbourhood touches [0,16] or [239,255] go last\n\ndef influence(active, shape):\n    """Pixel mask R that can be altered by modifying coefficients at the active positions."""\n    M, N = shape; R = np.zeros((M, N), bool); ms, ns = np.nonzero(active)\n    for dy in range(INF_LO, INF_HI + 1):\n        for dx in range(INF_LO, INF_HI + 1):\n            r = 2 * ms + dy; c = 2 * ns + dx; ok = (r >= 0) & (r < M) & (c >= 0) & (c < N); R[r[ok], c[ok]] = True\n    return R\n\ndef blocks_to_positions(E, pshape):\n    return np.kron(E.astype(np.uint8), np.ones((BLK, BLK), np.uint8)).astype(bool)[:pshape[0], :pshape[1]]\n\ndef choose_exclusion(I, D, rho):\n    """Sender\'s (free) choice of excluded 16x16 pixel blocks: blocks whose fraction of near-boundary pixels\n    (<= D or >= 255-D) exceeds rho. rho > 1 disables exclusion."""\n    M, N = I.shape; nb = ((M // 2 + BLK - 1) // BLK, (N // 2 + BLK - 1) // BLK)\n    if rho > 1: return np.zeros(nb, bool)\n    near = ((I <= D) | (I >= 255 - D)).astype(np.float64)\n    s = 2 * BLK; E = np.zeros(nb, bool)\n    for i in range(nb[0]):\n        for j in range(nb[1]):\n            E[i, j] = near[i * s:(i + 1) * s, j * s:(j + 1) * s].mean() > rho\n    return E\n\ndef fob_map(I, Ip, R, D):\n    amb = R & ((Ip == D) | (Ip == 255 - D))\n    corr = np.abs(np.asarray(I, np.int64) - Ip)[amb]\n    return (ac_encode(corr.tolist(), D + 1) if corr.size else np.zeros(0, np.uint8)), int(amb.sum())\n\ndef fob_restore(Ip, D, BF, R):\n    amb = R & ((Ip == D) | (Ip == 255 - D)); n = int(amb.sum()); I = Ip.copy()\n    if n:\n        corr = ac_decode(BF, n, D + 1); sign = np.where(Ip[amb] == D, -1, 1); I[amb] = Ip[amb] + sign * corr\n    return I\n\n# ----------------------------------------------------------------------------- data hiding (plaintext IWT domain)\nHDR_BITS = 76                 # L(4) | D(8) | len_E(16) | len_F(24) | len_M(24)\nBAND_ORDER = {\'all3\': (\'hh\', \'hl\', \'lh\'), \'hh\': (\'hh\',), \'hl+lh\': (\'hl\', \'lh\')}\n\ndef embedding_order(ll, bands=\'all3\'):\n    """All positions sorted by (saturation flag, local LL complexity): smooth, non-saturated first. A function of LL\n    only; LL is never modified, so the receiver recomputes exactly the same order."""\n    P = np.pad(ll, 1, mode=\'edge\').astype(np.int64); h, w = ll.shape\n    nb8 = [P[1 + dy:h + 1 + dy, 1 + dx:w + 1 + dx] for dy in (-1, 0, 1) for dx in (-1, 0, 1)]\n    cx = sum(np.abs(P[1:-1, 1:-1] - q) for q in nb8)\n    sat = (np.min(nb8, axis=0) <= SAT_T) | (np.max(nb8, axis=0) >= 255 - SAT_T)\n    pos = np.lexsort((np.arange(cx.size), cx.ravel(), sat.ravel()))\n    names = BAND_ORDER[bands]; nb = len(names)\n    band_id = np.tile(np.arange(nb), len(pos)); flat = np.repeat(pos, nb)\n    return names, band_id, flat\n\ndef _gather(bands_dict, names, band_id, flat):\n    stack = np.stack([bands_dict[n].ravel() for n in names]); return stack[band_id, flat]\n\ndef _scatter(bands_dict, names, band_id, flat, vals):\n    for k, n in enumerate(names):\n        m = band_id == k; arr = bands_dict[n].ravel().copy(); arr[flat[m]] = vals[m]; bands_dict[n] = arr.reshape(bands_dict[n].shape)\n\ndef _expand(c, bits, L):\n    """Embed bits sequentially into coefficient vector c at level L. Returns (c\', n_used_coeffs)."""\n    c = c.copy(); nbits = len(bits)\n    if nbits == 0: return c, 0\n    emb = (c >= -L) & (c <= L - 1); cs = np.cumsum(emb)\n    if len(cs) == 0 or cs[-1] < nbits: raise ValueError(\'capacity\')\n    stop = int(np.searchsorted(cs, nbits)) + 1        # number of coefficients touched\n    seg = c[:stop]; e = emb[:stop]; out = seg.copy()\n    out[e] = 2 * seg[e] + bits\n    out[(~e) & (seg >= L)] = seg[(~e) & (seg >= L)] + L\n    out[(~e) & (seg < -L)] = seg[(~e) & (seg < -L)] - L\n    c[:stop] = out\n    return c, stop\n\ndef _extract(cw, nbits, L):\n    """Inverse of _expand on the prefix; returns (bits, restored prefix, n_used_coeffs)."""\n    if nbits == 0: return np.zeros(0, np.uint8), cw[:0], 0\n    emb = (cw >= -2 * L) & (cw <= 2 * L - 1); cs = np.cumsum(emb)\n    if len(cs) == 0 or cs[-1] < nbits: raise ValueError(\'stream shorter than header\')\n    stop = int(np.searchsorted(cs, nbits)) + 1\n    seg = cw[:stop]; e = emb[:stop]\n    bits = (seg[e] & 1).astype(np.uint8)\n    rest = seg.copy(); rest[e] = seg[e] >> 1               # floor(c\'/2)\n    rest[(~e) & (seg >= 2 * L)] -= L; rest[(~e) & (seg < -2 * L)] += L\n    return bits, rest, stop\n\ndef _embed_once(I, payload_bits, L, D, E, bands):\n    """One attempt with margin D and exclusion map E. Returns (Im or None, info, E_violation_blocks)."""\n    I = np.asarray(I, np.int64); M, N = I.shape; pshape = (M // 2, N // 2)\n    Epos = blocks_to_positions(E, pshape)\n    R = influence(~Epos, I.shape)\n    Ip = np.where(R, np.clip(I, D, 255 - D), I)\n    BF, namb = fob_map(I, Ip, R, D)\n    BE = ac_encode(E.ravel().astype(int).tolist(), 2) if E.any() else np.zeros(0, np.uint8)\n    ll, hl, lh, hh = iwt53(Ip); B = {\'hl\': hl, \'lh\': lh, \'hh\': hh}\n    names, bid, flat = embedding_order(ll, bands)\n    c = _gather(B, names, bid, flat)\n    hdr = np.concatenate([int2bits(L, 4), int2bits(D, 8), int2bits(len(BE), 16), int2bits(len(BF), 24),\n                          int2bits(len(payload_bits), 24)])\n    # stage 1: header + exclusion map at level 1 along the full order (receiver does not know E yet)\n    c1, s1 = _expand(c, np.concatenate([hdr, BE]).astype(np.int64), 1)\n    viol = Epos.ravel()[flat[:s1]]\n    if viol.any():                                   # stage-1 coefficients must lie outside excluded blocks\n        bad = np.unique(flat[:s1][viol]); bm = np.zeros(E.shape, bool)\n        bm[(bad // pshape[1]) // BLK, (bad % pshape[1]) // BLK] = True\n        return None, None, bm\n    # stage 2: FOB map + payload at level L along the order restricted to non-excluded positions\n    keep = np.nonzero(~Epos.ravel()[flat[s1:]])[0] + s1\n    tail, s2 = _expand(c1[keep], np.concatenate([BF, payload_bits]).astype(np.int64), L)\n    cw = c1.copy(); cw[keep] = tail\n    _scatter(B, names, bid, flat, cw)\n    Im = iiwt53(ll, B[\'hl\'], B[\'lh\'], B[\'hh\'])\n    info = dict(L=L, D=D, len_E=len(BE), n_excl_blocks=int(E.sum()), len_F=len(BF), n_amb=namb,\n                len_M=len(payload_bits), coeffs_used=int(s1 + s2), total_coeffs=len(c), R_frac=float(R.mean()))\n    if Im.min() < 0 or Im.max() > 255: return None, info, None\n    assert np.array_equal(Im[~R], Ip[~R])                 # nothing outside R is ever modified\n    return Im.astype(np.uint8), info, None\n\ndef embed(I, payload_bits, L, bands=\'all3\', D=None, rho=2.0, max_D=127):\n    """Returns marked image Im and info. payload_bits are already Kd-encrypted message bits.\n    D starts at 3L and is enlarged until the sender-side range check passes (D is carried in the header).\n    rho controls the optional exclusion of saturated blocks (rho > 1: none)."""\n    payload_bits = np.asarray(payload_bits, np.uint8)\n    D = 3 * L if D is None else D\n    I = np.asarray(I, np.int64)\n    while D <= max_D:\n        E = choose_exclusion(I, D, rho)\n        for _ in range(64):\n            Im, info, bm = _embed_once(I, payload_bits, L, D, E, bands)\n            if bm is None: break\n            E = E & ~bm                                      # un-exclude blocks hit by stage 1 (monotone)\n        if Im is not None: return Im, info\n        D += 1\n    raise RuntimeError(\'no valid margin\')\n\ndef extract(Im, bands=\'all3\'):\n    """Blind extraction + exact recovery. Returns (payload_bits, recovered cover, header dict)."""\n    Im = np.asarray(Im, np.int64); M, N = Im.shape; pshape = (M // 2, N // 2)\n    ll, hl, lh, hh = iwt53(Im); B = {\'hl\': hl, \'lh\': lh, \'hh\': hh}\n    names, bid, flat = embedding_order(ll, bands)\n    cw = _gather(B, names, bid, flat)\n    hb, _, _ = _extract(cw, HDR_BITS, 1)\n    L = bits2int(hb[:4]); D = bits2int(hb[4:12]); lE = bits2int(hb[12:28]); lF = bits2int(hb[28:52]); lM = bits2int(hb[52:76])\n    sb, r1, s1 = _extract(cw, HDR_BITS + lE, 1)\n    nbk = ((pshape[0] + BLK - 1) // BLK, (pshape[1] + BLK - 1) // BLK)\n    E = ac_decode(sb[HDR_BITS:], nbk[0] * nbk[1], 2).reshape(nbk).astype(bool) if lE else np.zeros(nbk, bool)\n    Epos = blocks_to_positions(E, pshape)\n    keep = np.nonzero(~Epos.ravel()[flat[s1:]])[0] + s1\n    rb, r2, s2 = _extract(cw[keep], lF + lM, L)\n    c = cw.copy(); c[:s1] = r1; c[keep[:s2]] = r2\n    _scatter(B, names, bid, flat, c)\n    Ip = iiwt53(ll, B[\'hl\'], B[\'lh\'], B[\'hh\'])\n    R = influence(~Epos, Ip.shape)\n    I = fob_restore(Ip, D, rb[:lF], R)\n    return rb[lF:], I.astype(np.uint8), dict(L=L, D=D, len_E=lE, len_F=lF, len_M=lM)\n\nRHOS = (2.0, 0.5, 0.25, 0.1, 0.05)\n\ndef max_net_capacity(I, L, bands=\'all3\', rhos=RHOS):\n    """Exact maximal NET payload (bits) at level L, every overhead deducted, maximised over the exclusion\n    threshold rho, and verified by an actual full-load embedding."""\n    I = np.asarray(I, np.int64)\n    best = dict(net=0, gross=0, D=3 * L, len_E=0, len_F=0, n_amb=0, rho=None, n_excl_blocks=0,\n                gross_ceiling=(3 if bands == \'all3\' else 1) * I.size // 4)\n    for rho in rhos:\n        D = 3 * L\n        while D <= 127:\n            E = choose_exclusion(I, D, rho); pshape = (I.shape[0] // 2, I.shape[1] // 2)\n            try:\n                for _ in range(64):\n                    Im, info, bm = _embed_once(I, np.zeros(0, np.uint8), L, D, E, bands)\n                    if bm is None: break\n                    E = E & ~bm\n            except ValueError:\n                break\n            if info is None: break\n            # count capacity of stage 2 exactly\n            Epos = blocks_to_positions(E, pshape)\n            R = influence(~Epos, I.shape); Ip = np.where(R, np.clip(I, D, 255 - D), I)\n            ll, hl, lh, hh = iwt53(Ip); Bd = {\'hl\': hl, \'lh\': lh, \'hh\': hh}\n            names, bid, flat = embedding_order(ll, bands); c = _gather(Bd, names, bid, flat)\n            n1 = HDR_BITS + info[\'len_E\']; e1 = (c >= -1) & (c <= 0); cs = np.cumsum(e1)\n            if cs[-1] < n1: break\n            s1 = int(np.searchsorted(cs, n1)) + 1\n            keep = np.nonzero(~Epos.ravel()[flat[s1:]])[0] + s1\n            cap2 = int(((c[keep] >= -L) & (c[keep] <= L - 1)).sum())\n            net = cap2 - info[\'len_F\']\n            if net <= 0: break\n            try:\n                Im2, inf2 = embed(I, np.zeros(net, np.uint8), L, bands, D=D, rho=rho, max_D=D)\n                if inf2[\'len_F\'] + net > cap2: break\n                if net > best[\'net\']:\n                    best = dict(net=net, gross=n1 + cap2, D=D, len_E=inf2[\'len_E\'], len_F=inf2[\'len_F\'], n_amb=inf2[\'n_amb\'],\n                                rho=rho, n_excl_blocks=inf2[\'n_excl_blocks\'], gross_ceiling=best[\'gross_ceiling\'])\n                break\n            except (RuntimeError, ValueError):\n                D += 1\n    return best\n\n# ----------------------------------------------------------------------------- image cipher (Ke, nonce)\ndef _cipher_material(K, MN, nonce, legacy=False):\n    xs, ys, ps = lcm_sequence(K, 3 * MN + 3, nonce)\n    ks = [p_bytes(ps[r * MN:(r + 1) * MN]) for r in range(3)]\n    if legacy: ks = [ks[0]] * 3                                     # first-submission code reused one stream\n    perms = [np.argsort(xs[r * MN:(r + 1) * MN], kind=\'stable\') for r in range(2)]   # rank permutation: bijective\n    seeds = p_bytes(ps[3 * MN:3 * MN + 3]) if not legacy else np.array([173, 91, 47], np.uint8)\n    return ks, perms, seeds\n\ndef encrypt(Im, K, nonce=b\'\', legacy=False):\n    shp = Im.shape; MN = Im.size; ks, perms, seeds = _cipher_material(K, MN, nonce, legacy)\n    fwd = _lib.chain_fwd_legacy if legacy else _lib.chain_fwd\n    v = np.ascontiguousarray(Im.ravel().astype(np.uint8)); out = np.empty(MN, np.uint8)\n    for r in range(3):\n        fwd(v, ks[r], int(seeds[r]), MN, out); v = out.copy()\n        if r < 2: v = np.ascontiguousarray(v[perms[r]])\n    return v.reshape(shp)\n\ndef decrypt(E, K, nonce=b\'\', legacy=False):\n    shp = E.shape; MN = E.size; ks, perms, seeds = _cipher_material(K, MN, nonce, legacy)\n    inv = _lib.chain_inv_legacy if legacy else _lib.chain_inv\n    c = np.ascontiguousarray(E.ravel().astype(np.uint8)); out = np.empty(MN, np.uint8)\n    for r in (2, 1, 0):\n        inv(c, ks[r], int(seeds[r]), MN, out); c = out.copy()\n        if r > 0:\n            ip = np.empty(MN, np.int64); ip[perms[r - 1]] = np.arange(MN); c = np.ascontiguousarray(c[ip])\n    return c.reshape(shp)\n\ndef payload_mask(Kd, n, nonce):\n    """Data-hiding key stream (bits) used to encrypt the message: one bit per iterate."""\n    _, _, ps = lcm_sequence(Kd, n, nonce)\n    return (p_bytes(ps) & 1).astype(np.uint8)\n\n# ----------------------------------------------------------------------------- full protocol\ndef sender(I, message_bits, L, Ke, Kd, nonce, bands=\'all3\', rho=2.0):\n    m = np.asarray(message_bits, np.uint8)\n    em = m ^ payload_mask(Kd, len(m), nonce)\n    Im, info = embed(I, em, L, bands, rho=rho)\n    return encrypt(Im, Ke, nonce), Im, info\n\ndef receiver(E, Ke, Kd, nonce, bands=\'all3\'):\n    Im = decrypt(E, Ke, nonce)\n    em, I, hdr = extract(Im, bands)\n    m = em ^ payload_mask(Kd, len(em), nonce) if Kd is not None else None\n    return m, I, Im, em, hdr\n\n# ----------------------------------------------------------------------------- metrics\ndef psnr(A, B):\n    mse = np.mean((A.astype(np.float64) - B.astype(np.float64)) ** 2)\n    return float(\'inf\') if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)\n\ndef mse(A, B): return float(np.mean((A.astype(np.float64) - B.astype(np.float64)) ** 2))\n\ndef ssim(A, B):\n    from skimage.metrics import structural_similarity\n    return float(structural_similarity(A, B, data_range=255, gaussian_weights=True, sigma=1.5, use_sample_covariance=False))\n\ndef entropy(I):\n    h = np.bincount(I.ravel(), minlength=256) / I.size; h = h[h > 0]; return float(-(h * np.log2(h)).sum())\n', 'baselines.py': '"""Re-implemented RDH-EI baselines, evaluated on the same images / payloads as the proposed scheme.\n  * Zhang (2011) [32]  : XOR stream encryption + block-wise 3-LSB flipping, fluctuation-based extraction (joint).\n  * Hong et al. (2012) [11]: same embedding, full-block side-match fluctuation + confidence-ordered decoding (joint).\n  * Puteaux & Puech (2018) [19] (EPE-HCRDH principle): MSB prediction, prediction-error map stored in the image,\n    MSB substitution by the data hider (separable). Error map arithmetic-coded (our implementation choice).\nAll three are re-implementations from the published descriptions (no author code was available offline).\n"""\nimport heapq, numpy as np\nfrom rdhei import ac_encode, ac_decode, int2bits, bits2int\n\ndef _bits_stream(seed, shape):\n    return np.random.default_rng(seed).integers(0, 256, shape, dtype=np.uint8)\n\n# ---------------------------------------------------------------------------------------------- Zhang 2011 / Hong 2012\ndef zh_encrypt(I, ke): return (I.astype(np.uint8) ^ _bits_stream(ke, I.shape))\n\ndef _block_partition(s, kd, nblk):\n    """For every block, a pseudo-random split of its s*s pixels into S0 (mask True) and S1 (equal sizes)."""\n    rng = np.random.default_rng(kd); masks = np.zeros((nblk, s * s), bool)\n    for b in range(nblk): masks[b, rng.permutation(s * s)[: s * s // 2]] = True\n    return masks.reshape(nblk, s, s)\n\ndef zh_embed(E, bits, s, kd):\n    M, N = E.shape; nb = (M // s) * (N // s); assert len(bits) <= nb\n    masks = _block_partition(s, kd, nb); out = E.copy()\n    for b, bit in enumerate(bits):\n        i, j = divmod(b, N // s); blk = out[i * s:(i + 1) * s, j * s:(j + 1) * s]\n        sel = masks[b] if bit == 0 else ~masks[b]\n        blk[sel] ^= 0b111\n    return out\n\ndef _flip(blk, m): b = blk.copy(); b[m] ^= 0b111; return b\n\ndef _fluct_zhang(b):\n    b = b.astype(np.float64)\n    return np.abs(b[1:-1, 1:-1] - (b[:-2, 1:-1] + b[2:, 1:-1] + b[1:-1, :-2] + b[1:-1, 2:]) / 4).sum()\n\ndef zhang_extract(Em, n, s, ke, kd):\n    D = zh_encrypt(Em, ke); M, N = D.shape; masks = _block_partition(s, kd, (M // s) * (N // s))\n    bits = np.zeros(n, np.uint8); R = D.copy()\n    for b in range(n):\n        i, j = divmod(b, N // s); blk = D[i * s:(i + 1) * s, j * s:(j + 1) * s]\n        h0 = _flip(blk, masks[b]); h1 = _flip(blk, ~masks[b])\n        f0, f1 = _fluct_zhang(h0), _fluct_zhang(h1)\n        bits[b] = 0 if f0 < f1 else 1; R[i * s:(i + 1) * s, j * s:(j + 1) * s] = h0 if f0 < f1 else h1\n    return bits, R, D\n\ndef _fluct_side(img, i, j, s):\n    """Hong et al.: fluctuation over ALL pixels of block (i,j), neighbours taken from the current image."""\n    M, N = img.shape; r0, r1 = max(i * s - 1, 0), min((i + 1) * s + 1, M); c0, c1 = max(j * s - 1, 0), min((j + 1) * s + 1, N)\n    P = img[r0:r1, c0:c1].astype(np.float64); P = np.pad(P, 1, mode=\'edge\')\n    oi, oj = i * s - r0 + 1, j * s - c0 + 1\n    C = P[oi:oi + s, oj:oj + s]\n    nb = (P[oi - 1:oi - 1 + s, oj:oj + s] + P[oi + 1:oi + 1 + s, oj:oj + s] + P[oi:oi + s, oj - 1:oj - 1 + s] + P[oi:oi + s, oj + 1:oj + 1 + s]) / 4\n    return np.abs(C - nb).sum()\n\ndef hong_extract(Em, n, s, ke, kd):\n    D = zh_encrypt(Em, ke); M, N = D.shape; nbx = N // s; masks = _block_partition(s, kd, (M // s) * nbx)\n    cur = D.copy(); decided = np.zeros(n, bool); bits = np.zeros(n, np.uint8); ver = np.zeros(n, int)\n    def score(b):\n        i, j = divmod(b, nbx); blk = D[i * s:(i + 1) * s, j * s:(j + 1) * s]\n        h = [_flip(blk, masks[b]), _flip(blk, ~masks[b])]; f = []\n        for k in (0, 1):\n            cur[i * s:(i + 1) * s, j * s:(j + 1) * s] = h[k]; f.append(_fluct_side(cur, i, j, s))\n        cur[i * s:(i + 1) * s, j * s:(j + 1) * s] = blk\n        return f\n    heap = []\n    for b in range(n):\n        f = score(b); heapq.heappush(heap, (-abs(f[0] - f[1]), b, 0, int(f[1] < f[0])))\n    while heap:\n        _, b, v, choice = heapq.heappop(heap)\n        if decided[b] or v != ver[b]: continue\n        i, j = divmod(b, nbx); blk = D[i * s:(i + 1) * s, j * s:(j + 1) * s]\n        cur[i * s:(i + 1) * s, j * s:(j + 1) * s] = _flip(blk, masks[b] if choice == 0 else ~masks[b])\n        bits[b] = choice; decided[b] = True\n        for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):           # side match: re-score undecided neighbours\n            ii, jj = i + di, j + dj\n            if 0 <= ii < M // s and 0 <= jj < nbx:\n                nbk = ii * nbx + jj\n                if nbk < n and not decided[nbk]:\n                    ver[nbk] += 1; f = score(nbk); heapq.heappush(heap, (-abs(f[0] - f[1]), nbk, ver[nbk], int(f[1] < f[0])))\n    return bits, cur, D\n\n# ---------------------------------------------------------------------------------------------- Puteaux & Puech 2018\ndef _pred(img, i, j):\n    if i == 0: return int(img[0, j - 1])\n    if j == 0: return int(img[i - 1, 0])\n    return (int(img[i - 1, j]) + int(img[i, j - 1])) // 2\n\ndef pp_error_map(I):\n    I = I.astype(np.int64); M, N = I.shape; err = np.zeros((M, N), bool); err[0, 0] = True\n    up = np.vstack([I[:1], I[:-1]]); left = np.hstack([I[:, :1], I[:, :-1]])\n    pred = (up + left) // 2; pred[0, :] = left[0, :]; pred[:, 0] = up[:, 0]\n    inv = I ^ 128\n    err |= np.abs(pred - I) >= np.abs(pred - inv)\n    err[0, 0] = True\n    return err\n\nPP_HDR = 24\n\ndef pp_sender_prepare(I, ke):\n    """Content owner: compute error map, encrypt (XOR), write [len|map] into the MSBs of the first K pixels and\n    keep their encrypted MSBs (to be appended by the data hider)."""\n    err = pp_error_map(I); BE = ac_encode(err.ravel().astype(int).tolist(), 2)\n    E = I.astype(np.uint8) ^ _bits_stream(ke, I.shape)\n    side = np.concatenate([int2bits(len(BE), PP_HDR), BE]); K = len(side)\n    flat = E.ravel().copy(); savedMSB = (flat[:K] >> 7).astype(np.uint8)\n    flat[:K] = (flat[:K] & 0x7F) | (side.astype(np.uint8) << 7)\n    return flat.reshape(I.shape), err, K, savedMSB\n\ndef pp_capacity(err, K):\n    usable = ~err.ravel(); usable[:K] = False\n    return int(usable.sum()) - K                                    # net payload (saved MSBs deducted)\n\ndef pp_embed(Eprep, err, K, savedMSB, bits, kd):\n    flat = Eprep.ravel().copy(); usable = np.nonzero(~err.ravel())[0]; usable = usable[usable >= K]\n    stream = np.concatenate([savedMSB, np.asarray(bits, np.uint8) ^ _bits_stream(kd, len(bits)) % 2]).astype(np.uint8)\n    assert len(stream) <= len(usable)\n    idx = usable[:len(stream)]; flat[idx] = (flat[idx] & 0x7F) | (stream << 7)\n    return flat.reshape(Eprep.shape)\n\ndef pp_read_side(Em):\n    flat = Em.ravel(); n = bits2int((flat[:PP_HDR] >> 7) & 1); K = PP_HDR + n\n    BE = ((flat[PP_HDR:K] >> 7) & 1).astype(np.uint8)\n    err = ac_decode(BE, Em.size, 2).astype(bool).reshape(Em.shape)\n    return err, K\n\ndef pp_extract(Em, nbits, kd):\n    err, K = pp_read_side(Em); flat = Em.ravel()\n    usable = np.nonzero(~err.ravel())[0]; usable = usable[usable >= K]\n    stream = ((flat[usable[:K + nbits]] >> 7) & 1).astype(np.uint8)\n    return stream[K:] ^ (_bits_stream(kd, nbits) % 2).astype(np.uint8)\n\ndef pp_recover(Em, nbits, ke):\n    """Receiver with the encryption key: exact recovery by MSB prediction."""\n    err, K = pp_read_side(Em); M, N = Em.shape; flat = Em.ravel().copy()\n    usable = np.nonzero(~err.ravel())[0]; usable = usable[usable >= K]\n    stream = ((flat[usable[:K]] >> 7) & 1).astype(np.uint8)        # saved original encrypted MSBs of first K pixels\n    flat[:K] = (flat[:K] & 0x7F) | (stream << 7)\n    ks = _bits_stream(ke, (M, N)).ravel()\n    low = (flat ^ ks) & 0x7F                                        # decrypted 7 LSBs (always correct)\n    used = np.zeros(M * N, bool); used[usable[:K + nbits]] = True\n    msb_ok = (~used)                                                # pixels whose encrypted MSB was not replaced\n    rec = np.zeros((M, N), np.int64); dec = (flat ^ ks).astype(np.int64)\n    um = used.reshape(M, N); lowm = low.reshape(M, N).astype(np.int64)\n    for i in range(M):\n        for j in range(N):\n            if not um[i, j]: rec[i, j] = dec[i * N + j]; continue\n            p = _pred(rec, i, j); a = lowm[i, j]; b = a + 128\n            rec[i, j] = a if abs(p - a) < abs(p - b) else b\n    return rec.astype(np.uint8), (flat ^ ks).reshape(M, N)\n'}
for name, text in PACKAGE_FILES.items():
    open(os.path.join(PKG, name), 'w').write(text)
# IEEE-754 double semantics are part of the specification: no -ffast-math
r = subprocess.run(['gcc', '-O2', '-shared', '-fPIC', '-o', os.path.join(PKG, 'liblcm.so'), os.path.join(PKG, 'lcm_core.c'), '-lm'],
                   capture_output=True, text=True)
assert r.returncode == 0, r.stderr
sys.path.insert(0, PKG)
for m in ('rdhei', 'baselines'): sys.modules.pop(m, None)
import rdhei
from rdhei import *
from rdhei import _lib, _embed_once, _gather, _scatter
from baselines import *
KE = Key(*KE_TUPLE); KD = Key(*KD_TUPLE)
def nonce_of(tag): return np.random.default_rng(zlib.crc32(tag.encode())).integers(0, 256, 16, dtype=np.uint8).tobytes()
print('reference implementation compiled and imported from', PKG)

In [ ]:
# ---------------------------------------------------------------- helpers: tables, figures, metrics
SAVED = []
def table(df, num, title, note=None, index=False):
    fn = f'{TABD}/Table{num:02d}.csv' if isinstance(num, int) else f'{TABD}/{num}.csv'
    df.to_csv(fn, index=index); SAVED.append(fn)
    display(Markdown(f'**Table {num}: {title}**' if isinstance(num, int) else f'**{title}**'))
    display(df if index else df.style.hide(axis='index'))
    if note: display(Markdown(f'<small>{note}</small>'))
def savefig(fig, num, name, title=''):
    fn = f'{FIGD}/Fig{num:02d}_{name}.png' if isinstance(num, int) else f'{FIGD}/{num}_{name}.png'
    fig.savefig(fn); SAVED.append(fn)
    if title: print(f'Fig. {num}: {title}')
    plt.show(); plt.close(fig)
def npcr_uaci(A, B):
    A = A.astype(np.float64); B = B.astype(np.float64)
    return 100 * np.mean(A != B), 100 * np.mean(np.abs(A - B)) / 255
def corr_all(I):
    I = I.astype(np.float64)
    pairs = {'H': (I[:, :-1], I[:, 1:]), 'V': (I[:-1, :], I[1:, :]), 'D': (I[:-1, :-1], I[1:, 1:])}
    return {k: float(np.corrcoef(a.ravel(), b.ravel())[0, 1]) for k, (a, b) in pairs.items()}
def chi2_hist(I):
    h = np.bincount(I.ravel(), minlength=256); e = I.size / 256; return float(((h - e) ** 2 / e).sum())
def local_entropy(I, seed=1, k=30, s=44):
    # Wu et al. (2013): k non-overlapping s x s blocks chosen at random
    rng = np.random.default_rng(seed); gy, gx = I.shape[0] // s, I.shape[1] // s
    idx = rng.choice(gy * gx, min(k, gy * gx), replace=False)
    return float(np.mean([entropy(I[(q // gx) * s:(q // gx + 1) * s, (q % gx) * s:(q % gx + 1) * s]) for q in idx]))
def local_entropy_random_pos(I, k=30, T=1936, seed=0):
    # variant used for the published Table 19 column: k blocks at random positions (blocks may overlap)
    rng = np.random.default_rng(seed); M, N = I.shape; s = int(np.sqrt(T)); vals = []
    for _ in range(k):
        i = rng.integers(0, M - s + 1); j = rng.integers(0, N - s + 1); vals.append(entropy(I[i:i + s, j:j + s]))
    return float(np.mean(vals))
def complexity(I):
    I = I.astype(np.float64); return (np.abs(np.diff(I, axis=1)).mean() + np.abs(np.diff(I, axis=0)).mean()) / 2
def pmap(fn, items):
    # parallel map over images (fork); falls back to serial
    if N_JOBS > 1 and len(items) > 1:
        try:
            import multiprocessing as mp
            with mp.get_context('fork').Pool(N_JOBS) as pool: return pool.map(fn, items)
        except Exception as ex: print('parallel run failed, running serially:', ex)
    return [fn(x) for x in items]

## 3. Input images

In [ ]:
PAPER_MANIFEST = [["SIPI5", "Airplane", "https://raw.githubusercontent.com/semnan-university-ai/image-processing-benchmark/bf86b771a920d11ec57aafd084cb88616a1c555c/airplane.png", "e0dcaaa5731ede03211997b464367b89"], ["SIPI5", "Baboon", "https://raw.githubusercontent.com/semnan-university-ai/image-processing-benchmark/bf86b771a920d11ec57aafd084cb88616a1c555c/baboon.png", "43e679e456da4eb04bf18c052271698f"], ["SIPI5", "Boat", "https://raw.githubusercontent.com/semnan-university-ai/image-processing-benchmark/bf86b771a920d11ec57aafd084cb88616a1c555c/boat.png", "49fff060920d71878383806593fc1922"], ["SIPI5", "Peppers", "https://raw.githubusercontent.com/semnan-university-ai/image-processing-benchmark/bf86b771a920d11ec57aafd084cb88616a1c555c/peppers.png", "b4cc612cb7307bf52c661039bc9d6676"], ["SIPI5", "Goldhill", "https://raw.githubusercontent.com/semnan-university-ai/image-processing-benchmark/bf86b771a920d11ec57aafd084cb88616a1c555c/goldhill.png", "6e87fcdaf27769c3d857ec18cd41da8c"], ["STD10", "Barbara", "https://raw.githubusercontent.com/semnan-university-ai/image-processing-benchmark/bf86b771a920d11ec57aafd084cb88616a1c555c/barbara.png", "349443e3f26bb4aceba5e6b6d97be6ee"], ["STD10", "Zelda", "https://raw.githubusercontent.com/semnan-university-ai/image-processing-benchmark/bf86b771a920d11ec57aafd084cb88616a1c555c/zelda.png", "bebbe156a2e8667f81c9176f09511309"], ["STD10", "House", "https://raw.githubusercontent.com/mohammadimtiazz/standard-test-images-for-Image-Processing/e14c591b5188b695f0b47555542bcd46d853916a/standard_test_images/house.tif", "d77562f89cf4bb53e003c5a598dc1880"], ["STD10", "Lake", "https://raw.githubusercontent.com/mohammadimtiazz/standard-test-images-for-Image-Processing/e14c591b5188b695f0b47555542bcd46d853916a/standard_test_images/lake.tif", "b6a22400e841948c755043af58c10794"], ["STD10", "Walkbridge", "https://raw.githubusercontent.com/mohammadimtiazz/standard-test-images-for-Image-Processing/e14c591b5188b695f0b47555542bcd46d853916a/standard_test_images/walkbridge.tif", "0f92a9589627dbff2d0a50fc382ee45c"], ["STD10", "Pirate", "https://raw.githubusercontent.com/mohammadimtiazz/standard-test-images-for-Image-Processing/e14c591b5188b695f0b47555542bcd46d853916a/standard_test_images/pirate.tif", "0ef56fa42aad03b08795f37d84c3e4df"], ["STD10", "Livingroom", "https://raw.githubusercontent.com/mohammadimtiazz/standard-test-images-for-Image-Processing/e14c591b5188b695f0b47555542bcd46d853916a/standard_test_images/livingroom.tif", "368f48b0002692fe2b78e7eb39eba73c"], ["STD10", "WomanBlonde", "https://raw.githubusercontent.com/mohammadimtiazz/standard-test-images-for-Image-Processing/e14c591b5188b695f0b47555542bcd46d853916a/standard_test_images/woman_blonde.tif", "8cc3cc37e19d566303e9d1049c9e69e6"], ["STD10", "WomanDarkhair", "https://raw.githubusercontent.com/mohammadimtiazz/standard-test-images-for-Image-Processing/e14c591b5188b695f0b47555542bcd46d853916a/standard_test_images/woman_darkhair.tif", "695335ad230bef7210a242f4469ea172"], ["STD10", "Cameraman", "https://raw.githubusercontent.com/mohammadimtiazz/standard-test-images-for-Image-Processing/e14c591b5188b695f0b47555542bcd46d853916a/standard_test_images/cameraman.tif", "028c5b08b9ccbba722f2cf8322cbae52"], ["KODAK24", "kodim01", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/01.png", "b133345bcf171df9d7cabe0f7a5430e1"], ["KODAK24", "kodim02", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/02.png", "c668a19c99751663457423c24e2f275e"], ["KODAK24", "kodim03", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/03.png", "609d340b0b06e9a805bb23c2227798f7"], ["KODAK24", "kodim04", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/04.png", "ff229d77abf779dc358fbd9e098d9b7c"], ["KODAK24", "kodim05", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/05.png", "e51788fecbfe85eb7af4db78c99fa1b0"], ["KODAK24", "kodim06", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/06.png", "f1e50a38497905ef05fdf3ae78f8da20"], ["KODAK24", "kodim07", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/07.png", "6c24bb1d8b361fb809663f8048a11d17"], ["KODAK24", "kodim08", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/08.png", "04222f567182bdd4f8c755e9deac0b28"], ["KODAK24", "kodim09", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/09.png", "4c27b7170ee9d466fc6281cdcaf6b7fe"], ["KODAK24", "kodim10", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/10.png", "9ef3653cdeba753248aecb8ca781f190"], ["KODAK24", "kodim11", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/11.png", "2f9b9f3fe6846a6b6398014d6c50703c"], ["KODAK24", "kodim12", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/12.png", "afd8e6d9bc73cc92253098058777ca01"], ["KODAK24", "kodim13", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/13.png", "d104f2872edb9ad8278f5a5b3a90d49c"], ["KODAK24", "kodim14", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/14.png", "0b5e277b009d1346fde78b91351d84f8"], ["KODAK24", "kodim15", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/15.png", "2ff2be45f0f7c57e4a12628fa71140b3"], ["KODAK24", "kodim16", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/16.png", "44eb33209ced278c0e1690e77748cc36"], ["KODAK24", "kodim17", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/17.png", "f5838c05fc31248ba71b81a578c416aa"], ["KODAK24", "kodim18", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/18.png", "3e510edc7099c7144e7d952bdd1334af"], ["KODAK24", "kodim19", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/19.png", "88531e9a60fbbd0169de83bf63a2f013"], ["KODAK24", "kodim20", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/20.png", "c019d3f74db161f4546591dd3c402f5d"], ["KODAK24", "kodim21", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/21.png", "93e2ccc22a749429f953d0bcbab405f9"], ["KODAK24", "kodim22", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/22.png", "acd7dc9cb40b47cdfe5852ddbd660e5b"], ["KODAK24", "kodim23", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/23.png", "2b9b2bbcd74f609b49387bc74270bdf8"], ["KODAK24", "kodim24", "https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/dd2e1105fe8d9afe1120cd7e4c0e5e41f509a81b/PhotoCD_PCD0992/24.png", "f9cec46a0d2e6673f1d06c8f7a9bac45"], ["MED", "CXR01", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/000001-1.png", "3c6c31cf794c8e0138d24aeef051f6bf"], ["MED", "CXR02", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/000001-4.png", "adb924ebae2c50eef8dc3133288319e1"], ["MED", "CXR03", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/000001-5.png", "0b319f3bec0721518d9a8f1fecf6edff"], ["MED", "CXR04", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/000001-6.png", "d3f5b39c93557df3c3651249b94541ec"], ["MED", "CXR05", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/000004.png", "969e6b7e0ed959d124b2636764c5ca21"], ["MED", "CXR06", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/000006-2.png", "594e7a4a758d8fc3437de7184a47c774"], ["MED", "CXR07", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/000007-2.png", "6d393bf3f9ef0276728275330a75aea8"], ["MED", "CXR08", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/000010-2.png", "f5d4a56e8c13c79772645100a6e53337"], ["MED", "CXR09", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1-s2.0-S2214250920300834-gr1_lrg-b.png", "aa5f1898b5c196afd0fefe2391c75f1c"], ["MED", "CXR10", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1-s2.0-S2214250920300834-gr1_lrg-c.png", "84ddfeee8118a74e6ca3d48166d7fc94"], ["MED", "CXR11", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1-s2.0-S2214250920300834-gr1_lrg-d.png", "2ae6b610b9d724a62af64f0a02986068"], ["MED", "CXR12", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1-s2.0-S2387020620301959-gr4_lrg-b.png", "caf58be036fd7294e0776aa74311b78c"], ["MED", "CXR13", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1-s2.0-S2387020620301959-gr4_lrg-c.png", "19c4a11519630c5dc5574a15a0a25d38"], ["MED", "CXR14", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1.CXRCTThoraximagesofCOVID-19fromSingapore.pdf-000-fig1a.png", "51254f397cf402a6bea9ab124c0a7efe"], ["MED", "CXR15", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1.CXRCTThoraximagesofCOVID-19fromSingapore.pdf-000-fig1b.png", "bff76288ba214e23944a6a3042ac51be"], ["MED", "CXR16", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1.CXRCTThoraximagesofCOVID-19fromSingapore.pdf-001-fig2a.png", "50d71c2016725d6c81f0fd1f3bb0a09a"], ["MED", "CXR17", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1.CXRCTThoraximagesofCOVID-19fromSingapore.pdf-001-fig2b.png", "2df0d3d0d860f92908245b49d3b793eb"], ["MED", "CXR18", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1.CXRCTThoraximagesofCOVID-19fromSingapore.pdf-002-fig3a.png", "a8f9fae19e26499a182c2221f163a1b2"], ["MED", "CXR19", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1.CXRCTThoraximagesofCOVID-19fromSingapore.pdf-002-fig3b.png", "8efc68889a77868eb1b475a6b446fdb4"], ["MED", "CXR20", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1.CXRCTThoraximagesofCOVID-19fromSingapore.pdf-003-fig4a.png", "0bcffef84f0af7c8d6c7e68e93e14c10"], ["MED", "CXR21", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1.CXRCTThoraximagesofCOVID-19fromSingapore.pdf-003-fig4b.png", "1a61be165d40e076adc132a1b54203b1"], ["MED", "CXR22", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/10.1016-slash-j.anl.2020.04.002-a.png", "fe5478b331a25636fdd5015688107c5f"], ["MED", "CXR23", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/10.1016-slash-j.crad.2020.04.002-a.png", "8cb8f16518143b7343697fb1e80c61ff"], ["MED", "CXR24", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/10.1016-slash-j.crad.2020.04.002-b.png", "d096fc5f1ec29e2d09ac3ab53ebcbaea"], ["MED", "CXR25", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/16497_1_1.png", "bf67e35c944086c884a2c0673a1683d3"], ["MED", "CT01", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1-s2.0-S0929664620300449-gr3_lrg-a.jpg", "6b3296f9c8f2c067b8b2c2ca8172d149"], ["MED", "CT02", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1-s2.0-S0929664620300449-gr3_lrg-b.jpg", "145035b461eafdc96f77d588be226272"], ["MED", "CT03", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1-s2.0-S0929664620300449-gr3_lrg-c.jpg", "51ee83c4974965163fac0867991bc745"], ["MED", "CT04", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/1-s2.0-S0929664620300449-gr3_lrg-d.jpg", "dead2af598548c311870757c72d2ea4f"], ["MED", "CT05", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/16630_1_1.jpg", "73685936977091b565cd1dd360143599"], ["MED", "CT06", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/16630_2_1.jpg", "5d2a94e26f218e6b289d9e6f75da124e"], ["MED", "CT07", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/16630_2_2.jpg", "8d90195d1b2a4f70ab8758d2de259a86"], ["MED", "CT08", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/16630_3_1.jpg", "6580b3e373530fcf82a6d0b2ef5b5f6d"], ["MED", "CT09", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/16630_4_1.jpg", "7578ff1e544be2ef337442280a59c089"], ["MED", "CT10", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/16630_5_1.jpg", "48a393a25e2f77b31e5f7d0b09875648"], ["MED", "CT11", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/16630_6_1.jpg", "05be768078057bfebdb6e0ac6be47412"], ["MED", "CT12", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/16631_1_1.jpg", "277b8e5223c667b35206a5eb46e275de"], ["MED", "CT13", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/16631_1_2.jpg", "2e95b0b6a4395ee8bffacc66a93036f5"], ["MED", "CT14", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/16631_1_3.jpg", "3909b4b0b2c96aa0c79e3df32c9e0f6e"], ["MED", "CT15", "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/78543292f8b01d5e0ed1d0e15dce71949f0657bb/images/16631_1_4.jpg", "dfbebbaaff6955bd1c2cd502c4ae7028"]]
GROUP_LABEL = {'SIPI5': 'USC-SIPI', 'STD10': 'Standard', 'KODAK24': 'Kodak', 'MED': 'Medical'}
def gray(src):
    try: im = Image.open(src)
    except Exception:
        import imageio.v3 as iio; a = iio.imread(src)
        if a.ndim == 3: a = a[..., 0]
        return np.clip(a, 0, 255).astype(np.uint8)
    if im.mode in ('RGB', 'RGBA', 'P', 'CMYK', 'YCbCr', 'LA'): a = np.asarray(im.convert('RGB'), dtype=np.float64)
    else: a = np.asarray(im.convert('L') if im.mode not in ('L', 'I;16', 'I') else im, dtype=np.float64)
    if a.ndim == 3: a = np.round(0.299 * a[..., 0] + 0.587 * a[..., 1] + 0.114 * a[..., 2])      # ITU-R BT.601
    if a.max() > 255: a = np.round(a / a.max() * 255)
    return np.clip(a, 0, 255).astype(np.uint8)
def to512(a):
    h, w = a.shape; s = min(h, w); a = a[(h - s) // 2:(h - s) // 2 + s, (w - s) // 2:(w - s) // 2 + s]
    return np.asarray(Image.fromarray(a).resize((512, 512), Image.LANCZOS)) if s != 512 else a
def even(a): return a[:a.shape[0] // 2 * 2, :a.shape[1] // 2 * 2]
def fetch(url):
    import urllib.request
    for k in range(4):
        try:
            with urllib.request.urlopen(url, timeout=60) as r: return io.BytesIO(r.read())
        except Exception as ex:
            if k == 3: raise
            time.sleep(2 + 3 * k)

IMGS = []                      # list of dict(group, name, I)
if INPUT_MODE == 'paper':
    cache = os.path.join(OUT, '..', 'paper_images'); ok = 0
    for g, n, url, md5 in PAPER_MANIFEST:
        fn = os.path.join(cache, g, n + '.png')
        if os.path.exists(fn): a = np.asarray(Image.open(fn))
        else:
            raw = os.path.join(cache, 'original', g, os.path.basename(url)); os.makedirs(os.path.dirname(raw), exist_ok=True)
            if not os.path.exists(raw): open(raw, 'wb').write(fetch(url).getvalue())
            a = to512(gray(raw)); os.makedirs(os.path.dirname(fn), exist_ok=True); Image.fromarray(a).save(fn)
        ok += hashlib.md5(a.tobytes()).hexdigest() == md5; IMGS.append(dict(group=g, name=n, I=a))
    print(f'{len(IMGS)} paper images, {ok} identical to the images used in the paper (checksum)')
else:
    if INPUT_MODE == 'single':
        if UPLOAD_FROM_COMPUTER and IN_COLAB:
            from google.colab import files as _f; up = _f.upload(); IMAGE_PATH = os.path.abspath(next(iter(up)))
        paths = [(os.path.splitext(os.path.basename(os.path.dirname(os.path.abspath(IMAGE_PATH))))[0] or 'input', IMAGE_PATH)]
    elif INPUT_MODE == 'folder':
        paths = []
        for root, dirs, fs in os.walk(FOLDER_PATH):
            dirs.sort()
            for f in sorted(fs):
                if f.lower().endswith(EXTENSIONS):
                    rel = os.path.relpath(root, FOLDER_PATH); paths.append((os.path.basename(FOLDER_PATH.rstrip('/')) if rel == '.' else rel, os.path.join(root, f)))
            if not RECURSIVE: break
        assert paths, f'no images found in {FOLDER_PATH}'
    else: raise ValueError(INPUT_MODE)
    for g, p in paths:
        a = gray(p); a = to512(a) if RESIZE_TO_512 else even(a)
        if min(a.shape) < 64: print('skipped (smaller than 64 x 64):', p); continue
        IMGS.append(dict(group=g, name=os.path.splitext(os.path.basename(p))[0], I=a))
    print(len(IMGS), 'image(s):', ', '.join(f"{d['group']}/{d['name']} {d['I'].shape[0]}x{d['I'].shape[1]}" for d in IMGS[:20]), '...' if len(IMGS) > 20 else '')
GROUPS = list(dict.fromkeys(d['group'] for d in IMGS))
glabel = lambda g: GROUP_LABEL.get(g, g)
# reference set for the tables the paper reports "for the USC-SIPI images"
REF = [d for d in IMGS if d['group'] == 'SIPI5'] or IMGS[:max(1, min(len(IMGS), 10))]
BYNAME = {d['name'].lower(): d for d in IMGS}
pick = lambda name: BYNAME.get(name.lower(), REF[0])
print('groups:', [f'{glabel(g)} ({sum(d["group"] == g for d in IMGS)})' for g in GROUPS], '| reference set:', [d['name'] for d in REF])

# 65,536-bit secret: 256 x 256 binarised Cameraman (threshold 128), exactly as in the paper
if SECRET_IMAGE_PATH: S512 = to512(gray(SECRET_IMAGE_PATH))
elif 'cameraman' in BYNAME and INPUT_MODE == 'paper': S512 = BYNAME['cameraman']['I']
else:
    S512 = to512(gray(fetch('https://raw.githubusercontent.com/mohammadimtiazz/standard-test-images-for-Image-Processing/e14c591b5188b695f0b47555542bcd46d853916a/standard_test_images/cameraman.tif')))
SECRET = (np.asarray(Image.fromarray(S512).resize((256, 256), Image.LANCZOS)) >= 128).astype(np.uint8).ravel()
print('secret:', SECRET.size, 'bits')

In [ ]:
# Fig. 13: examples of the test images of each group
ng = len(GROUPS); nc = min(5, max(sum(d['group'] == g for d in IMGS) for g in GROUPS))
fig, axs = plt.subplots(ng, nc, figsize=(1.45 * nc + 0.4, 1.55 * ng + 0.2), squeeze=False)
for r, g in enumerate(GROUPS):
    fs = [d for d in IMGS if d['group'] == g]; sel = fs[:nc] if len(fs) <= nc or g == 'SIPI5' else [fs[int(i)] for i in np.linspace(0, len(fs) - 1, nc)]
    for c in range(nc):
        ax = axs[r, c]; ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if c < len(sel):
            ax.imshow(sel[c]['I'], cmap='gray', vmin=0, vmax=255); ax.set_xlabel(sel[c]['name'], fontsize=7)
        else: ax.axis('off')
        if c == 0: ax.set_ylabel(f'{glabel(g)}\n({len(fs)} images)', fontsize=8)
fig.tight_layout(); savefig(fig, 13, 'test_images', 'Examples of the test images in each group')

## 4. Section 3.1 — integer wavelet transform (Figs. 1–2, Table 1)

In [ ]:
if RUN['iwt']:
    # Fig. 1: the 5 x 5 pixel block centred at (2m, 2n)
    off = ['-2', '-1', '', '+1', '+2']
    fig, ax = plt.subplots(figsize=(7.6, 2.7)); ax.set_xlim(0, 5); ax.set_ylim(0, 5); ax.axis('off')
    for i in range(5):
        for j in range(5):
            ax.add_patch(plt.Rectangle((j, 4 - i), 1, 1, fill=False, lw=1.6))
            ax.text(j + 0.5, 4.5 - i, f'I(2m{off[i]}, 2n{off[j]})', ha='center', va='center', fontsize=9, style='italic')
    savefig(fig, 1, 'block5x5', 'An image block of size 5 x 5 centred at pixel (2m, 2n)')
    # Fig. 2: column outputs of the 5/3 IWT for this block (rounding-free form)
    J = ['2n-2', '2n-1', '2n', '2n+1', '2n+2']
    for k, (lab, rows) in enumerate((('(a) low-pass S(m, j)', [rf'$S(m,{j}) \approx \frac{{1}}{{8}}[-I(2m-2,{j})+2I(2m-1,{j})+6I(2m,{j})+2I(2m+1,{j})-I(2m+2,{j})]$' for j in J]),
                                      ('(b) high-pass D(m, j)', [rf'$D(m,{j}) \approx \frac{{1}}{{2}}[-I(2m,{j})+2I(2m+1,{j})-I(2m+2,{j})]$' for j in J]))):
        fig = plt.figure(figsize=(6.6, 1.75))
        for i, s in enumerate(rows):
            a = fig.add_axes([0.005, 1 - (i + 1) / 5 + 0.02, 0.99, 0.17]); a.set_xticks([]); a.set_yticks([]); a.grid(False)
            a.text(0.5, 0.5, s, ha='center', va='center', fontsize=9.5)
        savefig(fig, 2, 'abcd'[k] + '_column_outputs', 'Column outputs of the 5/3 IWT for the block of Fig. 1: ' + lab)

    # Table 1: linear bounds (Eq. 13), guaranteed bounds of the integer lifting, exact extremes (MILP, 8 x 8 incl. borders)
    from fractions import Fraction as Fr
    from scipy.optimize import milp, LinearConstraint, Bounds
    from scipy.sparse import coo_matrix
    a5 = [Fr(c, 8) for c in (-1, 2, 6, 2, -1)]; b3 = [Fr(c, 2) for c in (-1, 2, -1)]
    filt = {'ll': (a5, a5), 'hl': (a5, b3), 'lh': (b3, a5), 'hh': (b3, b3)}       # (column filter, row filter)
    lin, wsum = {}, {}
    for kk, (fc, fr) in filt.items():
        w = [p * q for p in fc for q in fr]; ps = sum(x for x in w if x > 0); ng = sum(x for x in w if x < 0)
        wsum[kk] = (ps, ng); lin[kk] = (255 * ng, 255 * ps)
    def lin_range(w, lo, hi): return (sum(c * (lo if c > 0 else hi) for c in w), sum(c * (hi if c > 0 else lo) for c in w))
    S_off = (Fr(-1, 4), Fr(3, 4)); D_off = (Fr(0), Fr(1, 2)); g = {}
    for kk, (w, inoff, own) in {'ll': (a5, S_off, S_off), 'hl': (b3, S_off, D_off), 'lh': (a5, D_off, S_off), 'hh': (b3, D_off, D_off)}.items():
        r_ = lin_range(w, *inoff); lo = lin[kk][0] + r_[0] + own[0]; hi = lin[kk][1] + r_[1] + own[1]
        g[kk] = (-((-lo.numerator) // lo.denominator), hi.numerator // hi.denominator)
    class Model:
        def __init__(s): s.lb, s.ub, s.rows, s.rl, s.ru = [], [], [], [], []
        def var(s, lo, hi): s.lb.append(lo); s.ub.append(hi); return {len(s.lb) - 1: 1}
        def cons(s, expr, lo, hi): e = dict(expr); c = e.pop('c', 0); s.rows.append(e); s.rl.append(lo - c); s.ru.append(hi - c)
        def floordiv(s, expr, q): zv = s.var(-10 ** 6, 10 ** 6); s.cons(add(expr, mul(zv, -q)), 0, q - 1); return zv
        def solve(s, obj, sense):
            n = len(s.lb); c = np.zeros(n)
            for i, w in obj.items():
                if i != 'c': c[i] = -w if sense == 'max' else w
            rr, cc, vv = [], [], []
            for r_, e in enumerate(s.rows):
                for i, w in e.items(): rr.append(r_); cc.append(i); vv.append(w)
            A = coo_matrix((vv, (rr, cc)), shape=(len(s.rows), n)).tocsr()
            res = milp(c, constraints=LinearConstraint(A, s.rl, s.ru), integrality=np.ones(n), bounds=Bounds(s.lb, s.ub), options={'mip_rel_gap': 0})
            return round(sum(w * res.x[i] for i, w in obj.items() if i != 'c') + obj.get('c', 0)), res.x
    def add(*es):
        r_ = {}
        for e in es:
            for kk, v in e.items(): r_[kk] = r_.get(kk, 0) + v
        return r_
    def mul(e, c): return {kk: v * c for kk, v in e.items()}
    def fwd_model(Mo, x):
        K = len(x) // 2; e = [x[2 * k] for k in range(K)]; o = [x[2 * k + 1] for k in range(K)]; e.append(e[K - 1])
        d = [add(o[k], mul(Mo.floordiv(add(e[k], e[k + 1]), 2), -1)) for k in range(K)]
        s = [add(e[k], Mo.floordiv(add(d[k - 1] if k else d[0], d[k], {'c': 2}), 4)) for k in range(K)]
        return s, d
    def iwt_model(Mo, Iv):
        M_, N_ = len(Iv), len(Iv[0]); Sc, Dc = [], []
        for j in range(N_):
            s, d = fwd_model(Mo, [Iv[i][j] for i in range(M_)]); Sc.append(s); Dc.append(d)
        S = [[Sc[j][i] for j in range(N_)] for i in range(M_ // 2)]; D = [[Dc[j][i] for j in range(N_)] for i in range(M_ // 2)]
        out = {nm: tuple(zip(*[fwd_model(Mo, row) for row in Arr])) for nm, Arr in (('S', S), ('D', D))}
        return {'ll': out['S'][0], 'hl': out['S'][1], 'lh': out['D'][0], 'hh': out['D'][1]}
    Ms = 8; ext = {kk: [10 ** 9, -10 ** 9] for kk in filt}; t0 = time.time()
    for kk in filt:
        for mm in range(Ms // 2):
            for nn in range(Ms // 2):
                for sense in ('min', 'max'):
                    Mo = Model(); Iv = [[Mo.var(0, 255) for _ in range(Ms)] for _ in range(Ms)]
                    val, xs = Mo.solve(iwt_model(Mo, Iv)[kk][mm][nn], sense)
                    Img = np.array([[round(xs[list(Iv[i][j])[0]]) for j in range(Ms)] for i in range(Ms)])
                    assert iwt53(Img)[['ll', 'hl', 'lh', 'hh'].index(kk)][mm, nn] == val      # optimum re-checked with the real transform
                    ext[kk] = [min(ext[kk][0], val), ext[kk][1]] if sense == 'min' else [ext[kk][0], max(ext[kk][1], val)]
    f = lambda q: ('%d' % q) if q.denominator == 1 else ('%.1f' % float(q))
    DEN = {'ll': 64, 'hl': 16, 'lh': 16, 'hh': 4}
    fwk = lambda q, kk: f'{int(q * DEN[kk])}/{DEN[kk]}'
    t1 = pd.DataFrame([[kk, f'{fwk(wsum[kk][0], kk)}; −{fwk(-wsum[kk][1], kk)}', f'[{f(lin[kk][0])}, {f(lin[kk][1])}]', f'[{g[kk][0]}, {g[kk][1]}]',
                        f'[{ext[kk][0]}, {ext[kk][1]}]'] for kk in filt],
                      columns=['Sub-band', 'Sum of positive; negative weights', 'Linear bound, Eq. (13)', 'Guaranteed bound, integer lifting', 'Exact extremes, integer lifting'])
    table(t1, 1, 'Ranges of the 5/3 IWT sub-bands for 8-bit images', f'Exact extremes by mixed-integer programming over all positions of an 8 x 8 image incl. borders ({time.time() - t0:.0f} s).')
    # worst-case pixel change when every detail coefficient changes by at most L (sign chosen adversarially per pixel),
    # measured with the real integer inverse transform; this is the bound behind the margin D = 3L of Sec. 4.2
    rng = np.random.default_rng(31); worst = {}
    for L in (1, 2, 3):
        w = 0
        for _ in range(40):
            X = rng.integers(64, 192, (16, 16)); ll, hl, lh, hh = iwt53(X); i0, j0 = rng.integers(4, 12, 2)
            resp = {}
            for B, A in (('hl', hl), ('lh', lh), ('hh', hh)):
                for (i, j) in itertools.product(range(A.shape[0]), range(A.shape[1])):
                    Bd = dict(hl=hl.copy(), lh=lh.copy(), hh=hh.copy()); Bd[B][i, j] += 1
                    r_ = int(iiwt53(ll, Bd['hl'], Bd['lh'], Bd['hh'])[i0, j0] - X[i0, j0])
                    if r_: resp[(B, i, j)] = np.sign(r_)
            for sgn in (1, -1):
                Bd = dict(hl=hl.copy(), lh=lh.copy(), hh=hh.copy())
                for (B, i, j), sg in resp.items(): Bd[B][i, j] += sgn * L * sg
                w = max(w, abs(int(iiwt53(ll, Bd['hl'], Bd['lh'], Bd['hh'])[i0, j0] - X[i0, j0])))
        worst[L] = w
    print('largest pixel change with every detail coefficient changed by at most L (adversarial signs):',
          ', '.join(f'L = {L}: {w} (3L = {3 * L})' for L, w in worst.items()))

## 5. Sections 3.2–3.3 — the bounded 2D-LCM and its dynamics (Figs. 3–11, Tables 2–3)

In [ ]:
# ---- six maps, vectorised over a vector of control-parameter values P (state arrays x, y have the shape of P)
PI = np.pi
def Wfold(u): return u + 1 - 2 * np.floor((u + 1) / 2) - 1
def _cheb(x, y, w):
    t = w * np.arccos(np.clip(x, -1, 1)); xn = np.cos(t)
    d = w * np.sin(t) / np.sqrt(np.maximum(1 - x * x, 1e-300))
    return xn, y, d, 0 * x, 0 * x, 0 * x
def _henon(x, y, a, b=0.3): return 1 - a * x * x + y, b * x, -2 * a * x, 1 + 0 * x, b + 0 * x, 0 * x
def _lozi(x, y, a, b=0.5):  return 1 - a * np.abs(x) + y, b * x, -a * np.sign(x), 1 + 0 * x, b + 0 * x, 0 * x
def _slmm(x, y, a, b=3.0):
    x1 = a * (np.sin(PI * y) + b) * x * (1 - x); y1 = a * (np.sin(PI * x1) + b) * y * (1 - y)
    d11 = a * (np.sin(PI * y) + b) * (1 - 2 * x); d12 = a * PI * np.cos(PI * y) * x * (1 - x)
    g = a * (np.sin(PI * x1) + b) * (1 - 2 * y); h = a * PI * np.cos(PI * x1) * y * (1 - y)
    return x1, y1, d11, d12, h * d11, h * d12 + g
def _lasm(x, y, mu):
    A = PI * mu * (y + 3) * x * (1 - x); x1 = np.sin(A); cA = np.cos(A) * PI * mu
    d11 = cA * (y + 3) * (1 - 2 * x); d12 = cA * x * (1 - x)
    B = PI * mu * (x1 + 3) * y * (1 - y); y1 = np.sin(B); cB = np.cos(B) * PI * mu
    return x1, y1, cB * y * (1 - y) * d11, cB * (y * (1 - y) * d12 + (x1 + 3) * (1 - 2 * y)), 0, 0
def _lasm_full(x, y, mu):
    x1, y1, d21, d22, _, _ = _lasm(x, y, mu)
    A = PI * mu * (y + 3) * x * (1 - x); cA = np.cos(A) * PI * mu
    return x1, y1, cA * (y + 3) * (1 - 2 * x), cA * x * (1 - x), d21, d22
def _lcm(x, y, a, b=5.7, w=5.3):
    t = w * np.arccos(np.clip(x, -1, 1)); c = np.cos(t)
    d = -a * np.sign(c) * (-np.sin(t)) * (-w / np.sqrt(np.maximum(1 - x * x, 1e-300)))
    return Wfold(1 - a * np.abs(c) + y), b * x, d, 1 + 0 * x, b + 0 * x, 0 * x
# name, map, parameter label, range of the control parameter, initial state, 1-D?   (ranges as in Fig. 5)
MAPS = [('Chebyshev', _cheb, 'w', (0.0, 10.3), (0.3, 0.0), True),
        ('Hénon, b = 0.3', _henon, 'a', (0.0, 1.4), (0.0, 0.0), False),
        ('Lozi, b = 0.5', _lozi, 'a', (0.5, 1.75), (0.1, 0.1), False),
        ('2D-SLMM, b = 3', _slmm, 'a', (0.85, 1.0), (0.3, 0.6), False),
        ('2D-LASM', _lasm_full, 'a', (0.0, 1.0), (0.3, 0.6), False),
        ('2D-LCM, b = 5.7, w = 5.3', _lcm, 'a', (-10.0, 10.0), (0.3, 0.6), False)]

def iterate(f, P, s0, n, keep=0):
    x = np.full(P.shape, s0[0], float); y = np.full(P.shape, s0[1], float); out = []
    with np.errstate(all='ignore'):
        for i in range(n):
            x, y = f(x, y, P)[:2]
            if i >= n - keep: out.append(x.copy())
    return (np.array(out) if keep else None), x, y

def lyap_vec(f, P, s0, n=3000, tr=500, oned=False):
    """Lyapunov spectrum for every parameter value in P (QR / Gram-Schmidt on the tangent map); diverged orbits -> nan."""
    _, x, y = iterate(f, P, s0, tr)
    v1 = np.stack([np.ones_like(P), 0 * P]); v2 = np.stack([0 * P, np.ones_like(P)]); S1 = 0 * P; S2 = 0 * P
    with np.errstate(all='ignore'):
        for _ in range(n):
            r = f(x, y, P); a11, a12, a21, a22 = [np.broadcast_to(q, P.shape) for q in r[2:]]
            if oned:
                S1 += np.log(np.abs(a11)); x, y = r[0], r[1]; continue
            u1 = np.stack([a11 * v1[0] + a12 * v1[1], a21 * v1[0] + a22 * v1[1]])
            u2 = np.stack([a11 * v2[0] + a12 * v2[1], a21 * v2[0] + a22 * v2[1]])
            n1 = np.sqrt((u1 ** 2).sum(0)); u1 = u1 / n1; u2 = u2 - (u1 * u2).sum(0) * u1; n2 = np.sqrt((u2 ** 2).sum(0)); u2 = u2 / n2
            S1 += np.log(n1); S2 += np.log(n2); v1, v2 = u1, u2; x, y = r[0], r[1]
    l1, l2 = S1 / n, (np.full_like(P, np.nan) if oned else S2 / n)
    bad = ~np.isfinite(x) | (np.abs(x) > 1e6)
    l1[bad] = np.nan; l2[bad] = np.nan
    return l1, l2

def bif(ax, f, P, s0, n_tr=600, keep=80, title='', xl='a'):
    xs, _, _ = iterate(f, P, s0, n_tr + keep, keep)
    with np.errstate(all='ignore'):
        PP = np.broadcast_to(P, xs.shape); ok = np.isfinite(xs) & (np.abs(xs) < 1e3)
    ax.plot(PP[ok], xs[ok], ',', color='k', alpha=0.35, rasterized=True); ax.set_title(title, fontsize=9); ax.set_xlabel(xl); ax.set_ylabel('$x_i$'); ax.grid(False)
NP = 400 if QUICK else 1200

In [ ]:
if RUN['chaos']:
    # Fig. 3: Lozi map, (a) behaviour over the (a, b) plane, (b) attractor at (1.5, 0.5)
    A, B = np.meshgrid(np.linspace(1, 2, 241), np.linspace(-1.1, 0.9, 241)); P = A.ravel(); Bv = B.ravel()
    lozi_ab = lambda x, y, a: _lozi(x, y, a, Bv)
    xs, x, y = iterate(lozi_ab, P, (0.1, 0.1), 1200, 64)
    l1, _ = lyap_vec(lozi_ab, P, (0.1, 0.1), n=1500, tr=300)
    cls = np.full(P.shape, 4)                                   # 4 = other periodic
    ub = ~np.isfinite(xs).all(0) | (np.abs(xs) > 1e3).any(0); cls[ub] = 0
    fin = ~ub
    per = lambda k: fin & (np.abs(xs[-1] - xs[-1 - k]) < 1e-7)
    cls[per(1)] = 1; cls[per(2) & ~per(1)] = 2; cls[fin & (l1 > 1e-3)] = 3
    from matplotlib.colors import ListedColormap
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.4))
    axs[0].imshow(cls.reshape(A.shape), origin='lower', extent=(1, 2, -1.1, 0.9), aspect='auto',
                  cmap=ListedColormap(['#bdbdbd', '#2a78d6', '#1baf7a', '#e34948', '#eda100']), vmin=0, vmax=4, interpolation='nearest')
    from matplotlib.patches import Patch
    axs[0].legend(handles=[Patch(color=c_, label=l_) for c_, l_ in zip(['#bdbdbd', '#2a78d6', '#1baf7a', '#e34948', '#eda100'],
                  ['UB (unbounded)', 'P0 (fixed point)', 'P2 (period 2)', 'CH (chaos)', 'other periodic'])], fontsize=6.5, loc='lower left'); axs[0].set_xlabel('a'); axs[0].set_ylabel('b'); axs[0].set_title('(a) Lozi map over the (a, b) plane'); axs[0].grid(False)
    def lozi_orbit(a, b, n=40000, x_=0.1, y_=0.1):
        X_, Y_ = [], []
        for i in range(n + 1000):
            x_, y_ = 1 - a * abs(x_) + y_, b * x_
            if i >= 1000: X_.append(x_); Y_.append(y_)
        return np.array(X_), np.array(Y_)
    X15, _ = lozi_orbit(1.5, 0.5); X_, Y_ = lozi_orbit(1.7, 0.5)
    print(f'(a, b) = (1.5, 0.5) lies on the border b = a - 1 between P2 and CH: the orbit settles on a cycle with {len(np.unique(np.round(X15, 9)))} points; '
          'panel (b) therefore shows the strange attractor at (1.7, 0.5), the parameters of Fig. 6(c) and Table 2')
    axs[1].plot(X_, Y_, ',', color='#1b7a3a'); axs[1].set_title('(b) Lozi attractor, (a, b) = (1.7, 0.5)'); axs[1].set_xlabel('x'); axs[1].set_ylabel('y')
    savefig(fig, 3, 'lozi', 'Lozi map: (a) behaviour over the (a, b) plane; (b) strange attractor at (a, b) = (1.7, 0.5)')

    # Fig. 4: Chebyshev map, (a) chaotic sequence (w = 4), (b) bifurcation diagram
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.0))
    xs, _, _ = iterate(_cheb, np.array([4.0]), (0.3, 0), 200, 200); axs[0].plot(range(1, 201), xs[:, 0], lw=0.8, marker='.', ms=2)
    axs[0].set_title('(a) Chebyshev sequence, w = 4'); axs[0].set_xlabel('n'); axs[0].set_ylabel('$x_n$')
    bif(axs[1], _cheb, np.linspace(0, 10.3, NP), (0.3, 0), title='(b) Bifurcation diagram', xl='w')
    savefig(fig, 4, 'chebyshev', 'Chebyshev map: (a) chaotic sequence; (b) bifurcation diagram')

    # Fig. 5: bifurcation diagrams of the six maps
    fig, axs = plt.subplots(2, 3, figsize=(11, 6.2))
    for ax, (nm, f, pl, rg, s0, od), t in zip(axs.ravel(), MAPS, 'abcdef'):
        bif(ax, f, np.linspace(*rg, NP), s0, title=f'({t}) {nm}', xl=pl)
    fig.tight_layout(); savefig(fig, 5, 'bifurcation', 'Bifurcation diagrams of the six maps')

    # Fig. 6: phase portraits (2 x 10^4 iterates after a transient)
    PARS = [10.3, 1.4, 1.7, 1.0, 0.9, 5.0]; TIT = ['Chebyshev, w = 10.3', 'Hénon, (1.4, 0.3)', 'Lozi, (1.7, 0.5)', '2D-SLMM, (1, 3)', '2D-LASM, 0.9', '2D-LCM, (5, 5.7, 5.3)']
    fig, axs = plt.subplots(2, 3, figsize=(11, 6.4))
    for ax, (nm, f, pl, rg, s0, od), p, t, lt in zip(axs.ravel(), MAPS, PARS, TIT, 'abcdef'):
        x = np.array([s0[0]]); y = np.array([s0[1]]); X = []; Y = []
        with np.errstate(all='ignore'):
            for i in range(21000):
                xp = x[0]; x, y = f(x, y, np.array([p]))[:2]
                if i >= 1000: X.append(x[0]); Y.append(xp if od else y[0])
        ax.plot(X, Y, ',', color='k', alpha=0.5, rasterized=True); ax.set_title(f'({lt}) {t}', fontsize=9)
        ax.set_xlabel('$x_n$'); ax.set_ylabel('$x_{n-1}$' if od else '$y_n$'); ax.grid(False)
    fig.tight_layout(); savefig(fig, 6, 'phase', 'Phase portraits of the six maps')

    # Fig. 7: Lyapunov spectra
    fig, axs = plt.subplots(2, 3, figsize=(11, 6.2)); SPEC = {}
    for ax, (nm, f, pl, rg, s0, od), t in zip(axs.ravel(), MAPS, 'abcdef'):
        P = np.linspace(*rg, 401 if not QUICK else 151); l1, l2 = lyap_vec(f, P, s0, n=2000 if not QUICK else 800, oned=od); SPEC[nm] = (P, l1, l2)
        ax.plot(P, l1, color=COL[0], lw=1, label='$\\lambda_1$')
        if not od: ax.plot(P, l2, color=COL[1], lw=1, label='$\\lambda_2$')
        ax.axhline(0, color='k', lw=0.6, ls='--'); ax.set_title(f'({t}) {nm}', fontsize=9); ax.set_xlabel(pl); ax.set_ylabel('LE'); ax.legend(fontsize=7)
    fig.tight_layout(); savefig(fig, 7, 'lyapunov_spectra', 'Lyapunov spectra of the six maps')

    # Fig. 8: Lyapunov exponents of the 2D-LCM over the (a, b) plane (exact bounded map, C kernel)
    na, nb_ = (41, 41) if QUICK else (81, 81)
    Ag = np.linspace(-20, 20, na); Bg = np.linspace(-25, 25, nb_); SURF = {}
    fig = plt.figure(figsize=(12, 10))
    for k, w in enumerate((2.3, 5.3, 10.3)):
        L1 = np.zeros((nb_, na)); L2 = np.zeros((nb_, na))
        for i, b in enumerate(Bg):
            for j, a in enumerate(Ag): L1[i, j], L2[i, j] = lyapunov(Key(0.3, 0.6, a, b, w), n=2000 if QUICK else 4000, transient=500)
        SURF[w] = (L1, L2)
        ax = fig.add_subplot(4, 3, 1 + k, projection='3d'); AA, BB = np.meshgrid(Ag, Bg)
        ax.plot_surface(AA, BB, L1, cmap='viridis', lw=0, alpha=0.9); ax.plot_surface(AA, BB, L2, cmap='magma', lw=0, alpha=0.6)
        ax.set_title(f'({"ace"[k]}) w = {w}', fontsize=9); ax.set_xlabel('a'); ax.set_ylabel('b'); ax.set_zlabel('LE')
        ax = fig.add_subplot(4, 3, 4 + k)
        for i in range(0, nb_, max(1, nb_ // 40)): ax.plot(Bg[i] * np.ones(na), L1[i], '.', ms=1.5, color=COL[0]); ax.plot(Bg[i] * np.ones(na), L2[i], '.', ms=1.5, color=COL[1])
        ax.axhline(0, color='k', lw=0.6, ls='--'); ax.set_xlabel('b'); ax.set_ylabel('LE'); ax.set_title(f'({"bdf"[k]}) projection along a, w = {w}', fontsize=9)
    ag = np.linspace(-20, 20, 801 if not QUICK else 201); lg = np.array([lyapunov(Key(0.3, 0.6, a, 20.7, 2.3), n=20000 if not QUICK else 5000) for a in ag])
    ax = fig.add_subplot(4, 1, 3); ax.plot(ag, lg[:, 0], color=COL[0], label='$\\lambda_1$'); ax.plot(ag, lg[:, 1], color=COL[1], label='$\\lambda_2$')
    ax.axhline(0, color='k', lw=0.6, ls='--'); ax.axhline(np.log(np.sqrt(20.7)), color='gray', lw=0.6, ls=':'); ax.axvline(5, color='gray', lw=0.6)
    ax.set_xlabel('a'); ax.set_ylabel('LE'); ax.set_title('(g) w = 2.3, b = 20.7 (vertical line: working key a = 5)', fontsize=9); ax.legend(fontsize=7)
    fig.tight_layout(); savefig(fig, 8, 'lcm_lyapunov_ab', 'Lyapunov exponents of the 2D-LCM over the (a, b) plane')
    hyp = ag[lg[:, 1] > 0]
    print(f'(g): lambda1 in [{lg[:, 0].min():.2f}, {lg[:, 0].max():.2f}]; ln sqrt(20.7) = {np.log(np.sqrt(20.7)):.4f}; '
          f'lambda2 > 0 for a in [{hyp.min():.1f}, {hyp.max():.1f}] (width {hyp.max() - hyp.min():.1f})')
    for w, (L1, L2) in SURF.items():
        nc_ = Bg[(L1 <= 0).any(axis=1)]
        print(f'w = {w}: lambda1 > 0 at {100 * np.mean(L1 > 0):.1f} % of the (a,b) grid; non-chaotic b band {nc_.min() if len(nc_) else 0:+.2f} .. {nc_.max() if len(nc_) else 0:+.2f}; hyperchaos at {100 * np.mean((L1 > 0) & (L2 > 0)):.1f} %')

In [ ]:
if RUN['chaos']:
    # Fig. 9: sensitivity and correlation at the working key
    K2 = KE.replace(x0=KE.x0 + 1e-15); assert KE.x0 + 1e-15 != KE.x0
    xa, _, _ = lcm_sequence(KE, 65536, b'', transient=0); xb, _, _ = lcm_sequence(K2, 65536, b'', transient=0)
    div = int(np.argmax(np.abs(xa - xb) > 1e-3)) + 1
    xs, _, _ = lcm_sequence(KE, 65536, b''); xs2, _, _ = lcm_sequence(K2, 65536, b'')
    z = (xs - xs.mean()) / xs.std(); z2 = (xs2 - xs2.mean()) / xs2.std(); N = len(z)
    lags = np.arange(0, 101); ac = np.array([np.dot(z[:N - k], z[k:]) / (N - k) for k in lags])
    cl = np.arange(-50, 51); cc = np.array([np.dot(z[max(0, -k):N - max(0, k)], z2[max(0, k):N - max(0, -k)]) / (N - abs(k)) for k in cl])
    fig, axs = plt.subplots(1, 3, figsize=(10, 2.8)); n = 40
    axs[0].plot(range(1, n + 1), xa[:n], 'o-', ms=3, lw=1, label='$x_0=0.3$'); axs[0].plot(range(1, n + 1), xb[:n], 's-', ms=3, lw=1, label='$x_0=0.3+10^{-15}$')
    axs[0].axvline(div, color='gray', ls='--', lw=0.8); axs[0].set_xlabel('iteration n'); axs[0].set_ylabel('$x_n$'); axs[0].set_title('(a) Sensitivity to $x_0$'); axs[0].legend(fontsize=7, loc='lower left')
    axs[1].stem(lags, ac, basefmt=' ', markerfmt='.'); axs[1].set_xlabel('lag k'); axs[1].set_ylabel('autocorrelation'); axs[1].set_title('(b) Autocorrelation of $x_n$')
    axs[2].plot(cl, cc, lw=1, color=COL[7]); axs[2].axhline(0, color='k', lw=0.6); axs[2].set_ylim(-0.1, 0.1); axs[2].set_xlabel('lag k'); axs[2].set_ylabel('cross-correlation'); axs[2].set_title('(c) Cross-correlation')
    fig.tight_layout(); savefig(fig, 9, 'lcm_sensitivity', 'Sensitivity and correlation of the 2D-LCM at the working key')
    print(f'orbits separate after {div} iterations; max |autocorrelation| at lags 1-100: {np.abs(ac[1:]).max():.3f}; max |cross-correlation|: {np.abs(cc).max():.3f}')

    # Fig. 10: ergodicity (2 x 10^5 iterates at the working key)
    xs, ys, ps = lcm_sequence(KE, 200000, nonce_of('ergodicity'))
    kb = p_bytes(ps); hx, _ = np.histogram(xs, 100, range=(-1, 1)); e = len(xs) / 100; chix = ((hx - e) ** 2 / e).sum() / 99
    hk = np.bincount(kb, minlength=256); e2 = len(kb) / 256; chik = ((hk - e2) ** 2 / e2).sum() / 255
    fig, axs = plt.subplots(1, 3, figsize=(10.5, 3.0))
    H2, _, _ = np.histogram2d(xs, ys, 128, range=[[-1, 1], [-20.7, 20.7]]); im = axs[0].imshow(H2.T, origin='lower', extent=(-1, 1, -20.7, 20.7), aspect='auto', cmap='viridis')
    fig.colorbar(im, ax=axs[0], fraction=0.046); axs[0].set_title('phase-plane density (128 x 128)'); axs[0].set_xlabel('x'); axs[0].set_ylabel('y'); axs[0].grid(False)
    axs[1].bar(np.linspace(-0.99, 0.99, 100), hx / len(xs) / 0.02, width=0.02, color=COL[0]); axs[1].axhline(0.5, color='k', ls='--', lw=0.8)
    axs[1].set_title(f'marginal density of x (χ²/df = {chix:.3f})'); axs[1].set_xlabel('x')
    axs[2].bar(np.arange(256), hk, width=1, color=COL[1]); axs[2].axhline(e2, color='k', ls='--', lw=0.8); axs[2].set_title(f'key-stream bytes (χ²/df = {chik:.3f})'); axs[2].set_xlabel('byte value')
    fig.tight_layout(); savefig(fig, 10, 'ergodicity', 'Ergodicity of the 2D-LCM at the working key')
    print(f'chi2/df: x on 100 bins = {chix:.3f} (expected 1 +- {np.sqrt(2 / 99):.2f}); key-stream on 256 levels = {chik:.3f} (expected 1 +- {np.sqrt(2 / 255):.2f})')

In [ ]:
if RUN['chaos']:
    # Table 2: chaotic performance indicators, one protocol for all maps (2e5 iterates after 2000, QR of analytic Jacobians,
    # KS = sum of positive exponents, K = 0-1 test (median over 100 c), chi2/df on 100 equal bins of x, r1 = lag-1 autocorrelation)
    rng01 = np.random.default_rng(1)
    def Wf(u): return u + 1 - 2 * math.floor((u + 1) / 2) - 1
    def m_cheb(w): return (lambda s: (math.cos(w * math.acos(s[0])),), lambda s: np.array([[w * math.sin(w * math.acos(s[0])) / math.sqrt(max(1 - s[0] ** 2, 1e-300))]]), (0.3,))
    def m_henon(a, b): return (lambda s: (1 - a * s[0] ** 2 + s[1], b * s[0]), lambda s: np.array([[-2 * a * s[0], 1], [b, 0]]), (0.3, 0.2))
    def m_lozi(a, b): return (lambda s: (1 - a * abs(s[0]) + s[1], b * s[0]), lambda s: np.array([[-a * np.sign(s[0]), 1], [b, 0]]), (0.3, 0.2))
    def m_slmm(a, b):
        def f(s):
            x, y = s; x1 = a * (math.sin(PI * y) + b) * x * (1 - x); return (x1, a * (math.sin(PI * x1) + b) * y * (1 - y))
        def J(s):
            x, y = s; x1 = a * (math.sin(PI * y) + b) * x * (1 - x)
            dx1 = np.array([a * (math.sin(PI * y) + b) * (1 - 2 * x), a * PI * math.cos(PI * y) * x * (1 - x)])
            g_ = a * (math.sin(PI * x1) + b) * (1 - 2 * y); h = a * PI * math.cos(PI * x1) * y * (1 - y)
            return np.array([dx1, h * dx1 + np.array([0, g_])])
        return f, J, (0.3, 0.6)
    def m_lasm(mu):
        def f(s):
            x, y = s; x1 = math.sin(PI * mu * (y + 3) * x * (1 - x)); return (x1, math.sin(PI * mu * (x1 + 3) * y * (1 - y)))
        def J(s):
            x, y = s; A_ = PI * mu * (y + 3) * x * (1 - x); x1 = math.sin(A_)
            dx1 = math.cos(A_) * PI * mu * np.array([(y + 3) * (1 - 2 * x), x * (1 - x)]); B_ = PI * mu * (x1 + 3) * y * (1 - y)
            return np.array([dx1, math.cos(B_) * PI * mu * (y * (1 - y) * dx1 + np.array([0, (x1 + 3) * (1 - 2 * y)]))])
        return f, J, (0.3, 0.6)
    def m_lcm(a, b, w):
        f = lambda s: (Wf(1 - a * abs(math.cos(w * math.acos(s[0]))) + s[1]), b * s[0])
        def J(s):
            x = s[0]; t = w * math.acos(x); c = math.cos(t)
            return np.array([[-a * np.sign(c) * (-math.sin(t)) * (-w / math.sqrt(max(1 - x * x, 1e-300))), 1], [b, 0]])
        return f, J, (0.3, 0.6)
    def zero_one(phi, nc=100):
        phi = phi - phi.mean(); N = len(phi); n = np.arange(1, N + 1); ncut = N // 10; Ks = []
        for c in rng01.uniform(PI / 5, 4 * PI / 5, nc):
            p = np.cumsum(phi * np.cos(c * n)); q = np.cumsum(phi * np.sin(c * n))
            M = np.array([np.mean((p[j:] - p[:-j]) ** 2 + (q[j:] - q[:-j]) ** 2) for j in range(1, ncut + 1)])
            D = M - (phi.mean() ** 2) * (1 - np.cos(c * np.arange(1, ncut + 1))) / (1 - np.cos(c))
            Ks.append(np.corrcoef(np.arange(1, ncut + 1), D)[0, 1])
        return float(np.median(Ks))
    def analyse(name, f, J, s0, n=200000 if not QUICK else 20000, tr=2000):
        s = s0
        for _ in range(tr): s = f(s)
        d = len(s); Q = np.eye(d); S = np.zeros(d); xs = np.empty(n)
        for i in range(n):
            Q, Rm = np.linalg.qr(J(s) @ Q); S += np.log(np.abs(np.diag(Rm))); s = f(s); xs[i] = s[0]
        lam = sorted(S / n, reverse=True); h, _ = np.histogram(xs, bins=100, range=(xs.min(), xs.max())); e = n / 100
        return [name, f'{lam[0]:+.4f}', f'{lam[1]:+.4f}' if d > 1 else '–', f'{sum(l for l in lam if l > 0):.4f}', f'{zero_one(xs[::40][:5000]):.3f}',
                f'{((h - e) ** 2 / e).sum() / 99:.4g}', f'{np.corrcoef(xs[:-1], xs[1:])[0, 1]:+.4f}']
    t0 = time.time()
    rows = [analyse('Chebyshev (w = 10.3)', *m_cheb(10.3)), analyse('Hénon (1.4, 0.3)', *m_henon(1.4, 0.3)), analyse('Lozi (1.7, 0.5)', *m_lozi(1.7, 0.5)),
            analyse('2D-SLMM (1, 3)', *m_slmm(1, 3)), analyse('2D-LASM (0.9)', *m_lasm(0.9)), analyse('2D-LCM (5, 5.7, 5.3)', *m_lcm(5, 5.7, 5.3)),
            analyse('2D-LCM (working key)', *m_lcm(KE.a, KE.b, KE.w))]
    table(pd.DataFrame(rows, columns=['Map', 'λ₁', 'λ₂', 'KS', 'K', 'χ²/df', 'r₁']), 2, 'Chaotic performance indicators of six maps', f'{time.time() - t0:.0f} s')
    ks_nats = float(rows[-1][3]); print(f'entropy rate of the 2D-LCM at the working key: KS = {ks_nats:.4f} nats = {ks_nats / np.log(2):.2f} bits per iteration, '
                                        f'whereas 8 key bits are extracted per iteration (Eq. 20); see Sec. 5.7.6.5')

    # Table 3 and Fig. 11: robustness of chaos over each map's parameter range (rescaled to [0, 1])
    rows = []; fig, ax = plt.subplots(figsize=(8, 4.6))
    for k, (nm, f, pl, rg, s0, od) in enumerate(MAPS):
        P = np.linspace(*rg, 201)
        if nm == 'Chebyshev': P = P[P > 2]                       # the Chebyshev map is chaotic only for w > 2
        l1, _ = lyap_vec(f, P, s0, n=3000 if not QUICK else 1000, oned=od); neg = ~(l1 > 0)
        wins = int(np.sum(np.diff(neg.astype(int)) == 1) + neg[0])
        rows.append([nm, f'{pl} ∈ [{P[0]:g}, {P[-1]:g}]', f'{np.nanmin(l1):+.4f}', f'{100 * neg.mean():.1f} %', wins])
        ax.plot(np.linspace(0, 1, len(P)), l1, color=COL[k], lw=2.4 if 'LCM' in nm else 1.2, label=nm)
    T3ROWS = [list(r) for r in rows]
    ax.axhline(0, color='k', ls='--', lw=0.8); ax.set_ylim(-2, 4); ax.set_xlabel('control parameter, rescaled to [0, 1]'); ax.set_ylabel('$\\lambda_1$'); ax.legend(fontsize=7)
    table(pd.DataFrame(rows, columns=['Map', 'Parameter range', 'min λ₁', 'Range with λ₁ ≤ 0', 'Intervals with λ₁ ≤ 0']), 3,
          'Robustness of chaos over each map’s parameter range', 'computed from the same 201-point sweeps as Fig. 11 (the vertical axis of the figure is cut at -2); these are the values of Table 3 of the revised manuscript')
    lcm_min = float(rows[-1][2]); print(f'2D-LCM: min λ1 = {lcm_min:+.4f} and ½ ln b = {0.5 * np.log(5.7):.4f} (b = 5.7): since λ1 + λ2 = ln|b| and λ1 ≥ λ2, every map of the form (17) with |b| > 1 has λ1 ≥ ½ ln|b| > 0, '
                                       f'so its absence of non-chaotic windows is structural; the dissipative Hénon and Lozi maps (|b| < 1) have no such guarantee')
    savefig(fig, 11, 'largest_LE', 'Largest Lyapunov exponent of six maps against their rescaled control parameters')

    # Sec. 3.2: exponents at the working key, admissible-domain Lyapunov scan, equivalent keys
    l1, l2 = lyapunov(KE, n=200000); print(f'working key K_e: (lambda1, lambda2) = ({l1:+.4f}, {l2:+.4f}); sum = {l1 + l2:.4f} = ln b = {np.log(KE.b):.4f}')
    A_ = np.linspace(1, 20, 39 if not QUICK else 10); B_ = np.linspace(5, 25, 41 if not QUICK else 10); W_ = np.linspace(2, 10, 17 if not QUICK else 5); t0 = time.time()
    LY = np.array([(a, b, w, *lyapunov(Key(0.3, 0.6, a, b, w), n=5000, transient=1000)) for w in W_ for a in A_ for b in B_])
    np.save(f'{RES}/lyapunov_scan.npy', LY)
    SCAN = dict(points=len(LY), frac_l1_pos=float(np.mean(LY[:, 3] > 0)), frac_hyper=float(np.mean((LY[:, 3] > 0) & (LY[:, 4] > 0))), min_l1=float(LY[:, 3].min()))
    print(f"admissible domain (Eq. 18): {SCAN['points']:,} grid points, lambda1 > 0 at {100 * SCAN['frac_l1_pos']:.2f} % (min {SCAN['min_l1']:.3f}), "
          f"hyperchaos at {100 * SCAN['frac_hyper']:.1f} % ({time.time() - t0:.0f} s)")
    ks = lambda K, n=4096: p_bytes(lcm_sequence(K, n, nonce_of('eqkeys'))[2])
    r_ = np.random.default_rng(2026); cnt = 0; NK = 300 if not QUICK else 60
    for _ in range(NK):
        K = Key(r_.uniform(-1, 1), r_.uniform(-1, 1), r_.uniform(1, 20), r_.uniform(5, 25), r_.uniform(2, 10))
        cnt += np.array_equal(ks(K, 256), ks(K.replace(y0=K.y0 + 2), 256))
    print(f'equivalent keys: w -> -w identical: {np.array_equal(ks(KE), ks(KE.replace(w=-KE.w)))}; '
          f'y0 -> y0 + 2 identical in floating point for {cnt} of {NK} random admissible keys (paper: 224 of 300)')

## 6. Section 3.4 — NIST SP 800-22 (Tables 4–5)

In [ ]:
def nist_report(stsdir, seed, src, m=100, nbits=1_000_000):
    rng = np.random.default_rng(seed); data = bytearray()
    for k in range(m):
        nonce = rng.integers(0, 256, 16, dtype=np.uint8).tobytes()
        if src == 'lcm': _, _, p = lcm_sequence(KE, nbits // 8, nonce); data += p_bytes(p).tobytes()
        else: data += rng.integers(0, 256, nbits // 8, dtype=np.uint8).tobytes()
    open(f'{stsdir}/data/lcm_keystream.bin', 'wb').write(bytes(data))
    subprocess.run([f'{stsdir}/assess', str(nbits)], input=f'0\ndata/lcm_keystream.bin\n1\n0\n{m}\n1\n'.encode(), cwd=stsdir, stdout=subprocess.DEVNULL, check=False)
    rep = open(f'{stsdir}/experiments/AlgorithmTesting/finalAnalysisReport.txt').read()
    open(f'{RES}/nist_{src}{seed}.txt', 'w').write(rep); return rep
def parse_nist(rep):
    tests = {}; flags = []; nstat = 0
    for line in rep.splitlines():
        mm = re.match(r'\s*((?:\d+\s+){10})(\S+)\s+\*?\s*(\d+)/\s*(\d+)\s+\*?\s*(\S+)\s*$', line)
        if not mm: continue
        nstat += 1; pv = float(mm.group(2)); k, t, name = int(mm.group(3)), int(mm.group(4)), mm.group(5)
        e = tests.setdefault(name, dict(n=0, minprop=1.0, minp=1.0, eligible=t)); e['n'] += 1; e['minprop'] = min(e['minprop'], k / t)
        e['eligible'] = min(e['eligible'], t); e['minp'] = min(e['minp'], pv)
        bound = round((0.99 - 3 * np.sqrt(0.99 * 0.01 / t)) * t)
        if k < bound: flags.append(f'{name} proportion {k}/{t}')
        if pv < 1e-4: flags.append(f'{name} uniformity P = {pv:.6f}')
    return tests, flags, nstat
import re
if RUN['nist']:
    sts = os.path.abspath(os.path.join(OUT, '..', 'nist_sts'))
    if not os.path.exists(f'{sts}/sts/assess'):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/terrillmoore/NIST-Statistical-Test-Suite.git', sts], check=True)
        subprocess.run(['git', '-C', sts, 'checkout', '-q', '07cd57730402e9723b03f4775100773c36bfdc1b'], check=True)
        os.makedirs(f'{sts}/sts/obj', exist_ok=True)                                      # not tracked by git
        for t_ in ('Frequency', 'BlockFrequency', 'Runs', 'LongestRun', 'Rank', 'FFT', 'NonOverlappingTemplate', 'OverlappingTemplate', 'Universal',
                   'LinearComplexity', 'Serial', 'ApproximateEntropy', 'CumulativeSums', 'RandomExcursions', 'RandomExcursionsVariant'):
            os.makedirs(f'{sts}/sts/experiments/AlgorithmTesting/{t_}', exist_ok=True)   # what experiments/create-dir-script should create
        r_ = subprocess.run(['make', '-s'], cwd=f'{sts}/sts', capture_output=True, text=True)
        assert r_.returncode == 0 and os.path.exists(f'{sts}/sts/assess'), r_.stderr[-2000:]
    STS = f'{sts}/sts'; os.makedirs(f'{STS}/data', exist_ok=True); t0 = time.time()
    runs = [('2D-LCM, nonce seed 2026 (Table 4)', 2026, 'lcm'), ('2D-LCM, nonce seed 2027', 2027, 'lcm'), ('Reference: NumPy PCG64', 7, 'numpy')]
    reps = {}
    for lab, seed, src in runs:
        reps[lab] = nist_report(STS, seed, src); print(f'{lab}: done ({time.time() - t0:.0f} s)')
    tests, _, _ = parse_nist(reps[runs[0][0]]); rows = []
    for name, e in tests.items():
        bound = 0.99 - 3 * np.sqrt(0.99 * 0.01 / e['eligible']); kp = round(e['minprop'] * e['eligible']); kb = round(bound * e['eligible'])
        nlow = sum(1 for l in reps[runs[0][0]].splitlines() if l.rstrip().endswith(name) and re.match(r'\s*(?:\d+\s+){10}(0\.0000\d\d)', l))
        res = 'Pass' if (kp >= kb and e['minp'] >= 1e-4) else ((f'Pass (proportion); {nlow} of ' + str(e['n']) + ' uniformity values < 10⁻⁴') if kp >= kb else 'FAIL')
        rows.append([name, e['n'], e['eligible'], f"{kp}/{e['eligible']}", f"{kb}/{e['eligible']}", f"{e['minp']:.6f}", res])
    table(pd.DataFrame(rows, columns=['Test', 'Statistics', 'Sequences', 'Min. passing sequences', 'Required (STS)', 'Min. uniformity P-value', 'Result']), 4,
          'NIST SP 800-22 results for the 2D-LCM key-stream (100 sequences x 10^6 bits, α = 0.01)')
    rows = []
    for lab, seed, src in runs:
        _, flags, nstat = parse_nist(reps[lab])
        rows.append([lab, nstat, sum('proportion' in f for f in flags), sum('uniformity' in f for f in flags), '; '.join(flags) or 'none'])
    table(pd.DataFrame(rows, columns=['Run (100 x 10^6 bits each)', 'Statistics', 'Proportion flags', 'Uniformity flags', 'Flagged statistics']), 5,
          'Flags over three complete NIST SP 800-22 runs (α = 0.01)')

## 7. Section 4 — the proposed scheme (Fig. 12, Tables 6–7, embedding order)

In [ ]:
if RUN['scheme']:
    # Table 6: worked example of the embedding / extraction rule at L = 1
    c = np.array([0, -1, 3, -4, 1, 0, -2, 2, -1, 5]); bits = np.array([1, 0, 1, 1], np.uint8); L = 1
    cw, used = rdhei._expand(c.copy(), bits, L)
    bdec, rest, _ = rdhei._extract(cw, len(bits), L); crest = np.concatenate([rest, cw[len(rest):]])
    case, dec = [], []; k = 0
    for i, v in enumerate(c):
        if i >= used: case.append('–'); dec.append('–'); continue
        if -L <= v <= L - 1: case.append(f'exp. (b={bits[k]})'); dec.append(str(int(bdec[k]))); k += 1
        else: case.append('shift'); dec.append('–')
    cws = [str(int(x)) + (' (untouched)' if i >= used else '') for i, x in enumerate(cw)]
    t6 = pd.DataFrame([['c'] + [str(int(x)) for x in c], ['case'] + case, ["c′"] + cws, ['decoded b'] + dec, ['restored c'] + [str(int(x)) for x in crest]],
                      columns=[''] + [f'#{i + 1}' for i in range(len(c))])
    table(t6, 6, 'Worked example of the embedding and extraction rules at L = 1', f'bits {bits.tolist()}; restored == original: {np.array_equal(crest, c)}')
    t7 = pd.DataFrame([['L', 4, 'embedding level (1-15)'], ['D', 8, 'fall-off-boundary margin'], ['len_E', 16, 'length of B_E (0 if no block is excluded)'],
                       ['len_F', 24, 'length of B_F'], ['len_M', 24, 'length of the masked message m̃'], ['B_E', 'len_E', 'arithmetic-coded exclusion map (one symbol per 16 x 16 block)'],
                       ['B_F', 'len_F', 'arithmetic-coded fall-off-boundary corrections'], ['m̃', 'len_M', 'message masked with K_d']], columns=['Field', 'Bits', 'Content'])
    table(t7, 7, 'Layout of the embedded stream', f'header H = {HDR_BITS} bits (rdhei.HDR_BITS)')

    # ordering gain: complexity order from LL vs raster order (Sec. 4.3)
    orig = rdhei.embedding_order
    def raster(ll, bands='all3'):
        names = BAND_ORDER[bands]; nb = len(names); pos = np.arange(ll.size)
        return names, np.tile(np.arange(nb), len(pos)), np.repeat(pos, nb)
    rows = []; rng = np.random.default_rng(1)
    for d in list({id(x): x for x in [pick('Boat'), pick('Peppers'), pick('Baboon')]}.values()):
        I = d['I']
        for r in (0.05, 0.10, 0.20):
            n = int(r * I.size); m = rng.integers(0, 2, n).astype(np.uint8); out = {}
            for lab, fo in (('complexity', orig), ('raster', raster)):
                rdhei.embedding_order = fo; best = np.nan
                for L in range(1, 8):
                    for rho in RHOS:
                        try: Im, _ = embed(I, m, L, rho=rho); best = np.nanmax([best, psnr(I, Im)])
                        except (ValueError, RuntimeError): pass
                out[lab] = best
            rdhei.embedding_order = orig; rows.append([d['name'], f'{r:.2f}', f"{out['complexity']:.2f}", f"{out['raster']:.2f}", f"{out['complexity'] - out['raster']:+.2f}"])
    rdhei.embedding_order = orig
    og = pd.DataFrame(rows, columns=['Image', 'Net rate (bpp)', 'PSNR, LL-complexity order', 'PSNR, raster order', 'Gain (dB)']); gg = og['Gain (dB)'].astype(float)
    table(og, 'Sec4.3_ordering_gain', 'Embedding order: gain of the LL-complexity order over raster order (Sec. 4.3)',
          f'gain {gg.min():.1f}–{gg.max():.1f} dB, mean {gg.mean():.2f} dB')

In [ ]:
if RUN['scheme']:
    # Reviewer comment 1 (Sec. 4.2): the stored fall-off-boundary correction is a non-negative magnitude r = |I - I_p|;
    # the reconstruction direction follows from I_p = D (clamped up: I = D - r) or I_p = 255 - D (clamped down: I = 255 - D + r).
    sat = np.full((256, 256), 128, np.uint8); sat[:100] = 0; sat[150:] = 255
    sat[100:150] = np.random.default_rng(1).integers(0, 256, (50, 256)).astype(np.uint8)          # large black / white regions: frequent clamping
    rows = []; all_ok = True
    for nm, I in (('Boat', pick('Boat')['I']), ('synthetic saturated', sat)):
        for L in (1, 2, 3):
            D = 3 * L; Rm = rdhei.influence(np.ones((I.shape[0] // 2, I.shape[1] // 2), bool), I.shape)          # no excluded block
            Ip = np.where(Rm, np.clip(I.astype(np.int64), D, 255 - D), I.astype(np.int64))
            amb = Rm & ((Ip == D) | (Ip == 255 - D)); signed = (I.astype(np.int64) - Ip)[amb]; up = Ip[amb] == D
            BF, nA = rdhei.fob_map(I, Ip, Rm, D); Ir = rdhei.fob_restore(Ip, D, BF, Rm)
            ok = bool(np.array_equal(Ir, I)) and bool((signed[up] <= 0).all()) and bool((signed[~up] >= 0).all()) and (np.abs(signed).max(initial=0) <= D)
            all_ok &= ok
            rows.append([nm, L, D, int(amb.sum()), int((signed < 0).sum()), int(np.abs(signed).min(initial=0)), int(np.abs(signed).max(initial=0)), 'yes' if ok else 'NO'])
    table(pd.DataFrame(rows, columns=['Image', 'L', 'D = 3L', 'Ambiguous pixels', 'of which I - I_p < 0', 'min r', 'max r', 'r in {0..D}, direction from I_p, cover restored exactly']),
          'Sec4.2_fob_check', 'Fall-off-boundary correction r = |I - I_p| (Sec. 4.2, Algorithms 2 and 4)', 'a signed definition would be wrong for the pixels in the fifth column')
    FOB_CHECK_OK = all_ok

    # Reviewer comment 2 (Eq. 20): one key-byte conversion, floor(|p| * 2^32) mod 256, for the cipher, the round seeds, the data-hiding mask and the NIST sequences
    _, _, p = lcm_sequence(KE, 1_000_000, nonce_of('eq20'))
    b_fun = p_bytes(p)                                                                                    # the function used everywhere in the notebook
    b_np = (np.floor(np.abs(p) * 2.0 ** 32).astype(np.uint64) % 256).astype(np.uint8)                      # Eq. (20), NumPy
    b_py = np.array([int(math.floor(abs(v) * 4294967296.0)) % 256 for v in p[:100000]], np.uint8)        # Eq. (20), plain Python
    _, _, pd_ = lcm_sequence(KD, 4096, nonce_of('eq20'))
    mask_ok = bool(np.array_equal(rdhei.payload_mask(KD, 4096, nonce_of('eq20')), (b_ref := (np.floor(np.abs(pd_) * 2.0 ** 32).astype(np.uint64) % 256).astype(np.uint8)) & 1))
    cipher_ok = bool(np.array_equal(rdhei._cipher_material(KE, 64 * 64, nonce_of('eq20'))[0][0], (lambda q_: (np.floor(np.abs(q_[2][:64 * 64]) * 2.0 ** 32).astype(np.uint64) % 256).astype(np.uint8))(lcm_sequence(KE, 3 * 64 * 64 + 3, nonce_of('eq20')))))
    scale_exact = bool(np.all(np.abs(p) * 2.0 ** 32 == np.ldexp(np.abs(p), 32)))
    rows = [['p_bytes() equals floor(|p| 2^32) mod 256 (NumPy), 10^6 values', bool(np.array_equal(b_fun, b_np))],
            ['p_bytes() equals floor(|p| 2^32) mod 256 (plain Python), first 10^5 values', bool(np.array_equal(b_fun[:100000], b_py))],
            ['round-1 key-stream of the cipher equals the same conversion', cipher_ok],
            ['data-hiding mask equals the least significant bit of the same conversion of p\'', mask_ok],
            ['scaling by 2^32 is exact in binary64 (no rounding)', scale_exact], ['max |p| (< 1)', float(np.abs(p).max())]]
    table(pd.DataFrame(rows, columns=['Check (Eq. 20, Algorithm 3 steps 8-9, Eq. 22, Sec. 3.4)', 'Result']), 'Eq20_keybyte_check', 'Key-byte conversion of Eq. (20)', 'the same function also produces the NIST sequences of Section 6')
    KEYBYTE_CHECK_OK = bool(np.array_equal(b_fun, b_np) and np.array_equal(b_fun[:100000], b_py) and cipher_ok and mask_ok and scale_exact)

In [ ]:
FIG12_SRC = '"""Fig. 12: publication-quality block diagram of the proposed scheme, drawn for a 6.5 in print width (~8 pt text) and checked automatically:\n  * every text line lies inside its box (with padding),\n  * no connector passes through a box other than its source/target,\n  * no two connectors cross,\n  * every data dependency of Algorithms 1-2 is drawn, and nothing else.\nOutputs figs/fig7_block_diagram.pdf (vector) and .png (600 dpi)."""\nimport os, sys, itertools, numpy as np, matplotlib\nmatplotlib.use(\'Agg\'); import matplotlib.pyplot as plt\nfrom matplotlib.patches import FancyBboxPatch, Rectangle\nfrom PIL import Image\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\ntry:\n    from config import OUT as R, FIGS as F\nexcept Exception:\n    R, F = \'/home/claude/rdh/results\', \'/home/claude/rdh/figs\'\nplt.rcParams.update({\'font.family\': \'STIXGeneral\', \'mathtext.fontset\': \'stix\', \'font.size\': 8, \'pdf.fonttype\': 42, \'ps.fonttype\': 42})\n\nW, H = 7.16, 7.30\nFT, FB, FL, FC = 8.6, 8.1, 7.8, 7.9          # title, box body, wire label, caption (pt at 7.16 in; ~8 pt at the 6.5 in print width)\nCOL = dict(tr=\'#d9e6f6\', hide=\'#dcefd8\', fob=\'#e7dcf3\', chaos=\'#f6d6d6\', enc=\'#fbefc6\', rec=\'#e6e6e6\', band=\'#f9e2a8\', io=\'#ffffff\')\nfig = plt.figure(figsize=(W, H)); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis(\'off\')\nBOX = {}; TXT = []; WIRES = []\nLW = 0.8\ndef box(key, x, y, w, h, title, lines=(), fc=\'#fff\'):\n    BOX[key] = (x - w / 2, y - h / 2, x + w / 2, y + h / 2)\n    ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle=\'round,pad=0,rounding_size=0.04\', fc=fc, ec=\'#000\', lw=LW))\n    n = len(lines) + (1 if title else 0); dy = 0.148; y0 = y + (n - 1) * dy / 2; k = 0\n    if title: TXT.append((key, ax.text(x, y0, title, ha=\'center\', va=\'center\', fontsize=FT, weight=\'bold\'))); k = 1\n    for i, l in enumerate(lines): TXT.append((key, ax.text(x, y0 - (i + k) * dy, l, ha=\'center\', va=\'center\', fontsize=FB)))\ndef thumb(key, path, x, y, s, caption):\n    BOX[key] = (x - s / 2, y - s / 2, x + s / 2, y + s / 2)\n    ax.imshow(np.asarray(Image.open(path)), cmap=\'gray\', vmin=0, vmax=255, extent=(x - s / 2, x + s / 2, y - s / 2, y + s / 2), zorder=3, interpolation=\'lanczos\')\n    ax.add_patch(Rectangle((x - s / 2, y - s / 2), s, s, fill=False, ec=\'#000\', lw=0.6, zorder=4))\n    TXT.append((None, ax.text(x, y - s / 2 - 0.05, caption, ha=\'center\', va=\'top\', fontsize=FC, linespacing=1.15)))\ndef bands(key, x, y, s=0.29):\n    BOX[key] = (x - s - 0.03, y - s - 0.03, x + s + 0.03, y + s + 0.03)\n    for dx, dy, t in ((-1, 1, \'LL\'), (1, 1, \'HL\'), (-1, -1, \'LH\'), (1, -1, \'HH\')):\n        ax.add_patch(Rectangle((x + (dx - 1) * s / 2, y + (dy - 1) * s / 2), s, s, fc=\'#fff\' if t == \'LL\' else COL[\'band\'], ec=\'#000\', lw=0.6))\n        ax.text(x + dx * s / 2, y + dy * s / 2, t, ha=\'center\', va=\'center\', fontsize=FB, weight=\'bold\')\ndef wire(src, dst, pts, label=None, lxy=None, dashed=False, ha=\'left\'):\n    WIRES.append((src, dst, pts)); ls = (0, (3.5, 2.2)) if dashed else \'-\'\n    for a, b in zip(pts[:-2], pts[1:-1]): ax.plot([a[0], b[0]], [a[1], b[1]], color=\'#000\', lw=LW, ls=ls, solid_capstyle=\'butt\')\n    ax.annotate(\'\', xy=pts[-1], xytext=pts[-2], arrowprops=dict(arrowstyle=\'-|>\', lw=LW, color=\'#000\', mutation_scale=8, shrinkA=0, shrinkB=0, linestyle=ls))\n    if label: TXT.append((None, ax.text(*lxy, label, fontsize=FL, style=\'italic\', ha=ha, va=\'center\')))\ndef E(k, side, t=0.5):\n    x0, y0, x1, y1 = BOX[k]\n    return {\'l\': (x0, y0 + t * (y1 - y0)), \'r\': (x1, y0 + t * (y1 - y0)), \'t\': (x0 + t * (x1 - x0), y1), \'b\': (x0 + t * (x1 - x0), y0)}[side]\n\nX = dict(c0=0.40, c1=1.33, c2=2.36, c3=3.10, c4=4.32, c5=5.68, c6=6.74)\n# ================================================================ sender\nax.add_patch(Rectangle((0.05, 3.92), W - 0.10, 3.06, fill=False, ec=\'#444\', lw=0.7, ls=(0, (4, 3))))\nax.text(W / 2, 7.14, \'Content owner: embedding and encryption\', ha=\'center\', va=\'center\', fontsize=9.6, weight=\'bold\')\nr1, r2, r3 = 5.86, 4.86, 4.22\nthumb(\'cover\', f\'{R}/img/Boat_cover.png\', X[\'c0\'], r1, 0.56, \'Cover $I$\')\nbox(\'fob\', X[\'c1\'], r1, 1.20, 0.80, \'1. FOB pre-processing\', [\'margin $D$ (from $3L$)\', \'exclusion map $E$\', r\'clamp on $R$ $\\rightarrow B_F$\'], COL[\'fob\'])\nbox(\'iwt\', X[\'c2\'], r1, 0.62, 0.42, \'2. 5/3 IWT\', [], COL[\'tr\'])\nbands(\'bands\', X[\'c3\'], r1)\nbox(\'emb\', X[\'c4\'], r1, 1.46, 0.80, \'4. Expansion/shift\', [r"$c′=2c+b,\\ \\ -L\\leq c\\leq L-1$", r"$c′=c\\pm L$ otherwise", r"$H,B_E$ (level 1); $B_F,\\tilde m$ (level $L$)"], COL[\'hide\'])\nbox(\'iiwt\', X[\'c5\'], r1, 1.02, 0.62, \'5. Inverse IWT\', [r\'check $0\\leq I_m\\leq 255$\'], COL[\'rec\'])\n_ld = lambda p: np.asarray(Image.open(p))\n_I = _ld(f\'{R}/img/Boat_cover.png\').astype(float); _M = _ld(f\'{R}/img/Boat_marked.png\').astype(float)\n_psnr = 10 * np.log10(255 ** 2 / np.mean((_I - _M) ** 2))\nthumb(\'im\', f\'{R}/img/Boat_marked.png\', X[\'c6\'], r1, 0.52, \'Marked $I_m$\\n(%.2f dB)\' % _psnr)\nbox(\'ord\', X[\'c3\'], r2, 0.82, 0.50, \'3. Order\', [r\'from LL ($\\sigma,\\kappa$)\'], COL[\'hide\'])\nbox(\'msk\', X[\'c4\'], r2, 1.02, 0.50, \'Masking\', [r\'$\\tilde m = m\\oplus k_d$\'], COL[\'enc\'])\nbox(\'enc\', X[\'c5\'], r2, 1.18, 0.56, \'6. Encryption\', [r\'3 rounds ($+,\\oplus$), $\\pi_1,\\pi_2$\'], COL[\'enc\'])\nthumb(\'is\', f\'{R}/img/Boat_enc.png\', X[\'c6\'], r2 + 0.10, 0.52, r\'$I_s$ + nonce $N$\')\nbox(\'msg\', X[\'c3\'], r3, 0.82, 0.30, None, [\'message $m$\'], COL[\'io\'])\nbox(\'lcmd\', X[\'c4\'], r3, 1.20, 0.46, r\'2D-LCM $(K_d,N)$\', [r"$k_d=\\hat{p}′_n \\ \\mathrm{mod}\\ 2$"], COL[\'chaos\'])\nbox(\'lcme\', X[\'c5\'], r3, 1.34, 0.46, r\'2D-LCM $(K_e,N)$\', [r\'$\\hat p_n\\rightarrow k_r,s_r$;  $x_n\\rightarrow\\pi_1,\\pi_2$\'], COL[\'chaos\'])\n\nwire(\'cover\', \'fob\', [E(\'cover\', \'r\'), E(\'fob\', \'l\')])\nwire(\'fob\', \'iwt\', [E(\'fob\', \'r\'), E(\'iwt\', \'l\')], label=\'$I_p$\', lxy=((BOX[\'fob\'][2] + BOX[\'iwt\'][0]) / 2, r1 + 0.09), ha=\'center\')\nwire(\'iwt\', \'bands\', [E(\'iwt\', \'r\'), E(\'bands\', \'l\')])\nwire(\'bands\', \'emb\', [E(\'bands\', \'r\'), E(\'emb\', \'l\')])\nwire(\'emb\', \'iiwt\', [E(\'emb\', \'r\'), E(\'iiwt\', \'l\')])\nwire(\'iiwt\', \'im\', [E(\'iiwt\', \'r\'), E(\'im\', \'l\')])\nwire(\'bands\', \'ord\', [(X[\'c3\'] - 0.15, BOX[\'bands\'][1]), (X[\'c3\'] - 0.15, BOX[\'ord\'][3])], label=\'LL\', lxy=(X[\'c3\'] - 0.11, (BOX[\'bands\'][1] + BOX[\'ord\'][3]) / 2))\nxo = BOX[\'emb\'][0] + 0.10\nwire(\'ord\', \'emb\', [E(\'ord\', \'r\', 0.75), (xo, E(\'ord\', \'r\', 0.75)[1]), (xo, BOX[\'emb\'][1])])\nxm = BOX[\'ord\'][2] + 0.10\nwire(\'msg\', \'msk\', [E(\'msg\', \'r\'), (xm, E(\'msg\', \'r\')[1]), (xm, E(\'msk\', \'l\')[1]), E(\'msk\', \'l\')])\nwire(\'lcmd\', \'msk\', [E(\'lcmd\', \'t\'), E(\'msk\', \'b\')], label=r\'$k_d$\', lxy=(X[\'c4\'] + 0.05, (BOX[\'lcmd\'][3] + BOX[\'msk\'][1]) / 2))\nxmm = BOX[\'msk\'][0] + 0.62 * (BOX[\'msk\'][2] - BOX[\'msk\'][0])\nwire(\'msk\', \'emb\', [(xmm, BOX[\'msk\'][3]), (xmm, BOX[\'emb\'][1])], label=r\'$\\tilde m$\', lxy=(xmm + 0.05, (BOX[\'msk\'][3] + BOX[\'emb\'][1]) / 2))\nwire(\'iiwt\', \'enc\', [E(\'iiwt\', \'b\'), E(\'enc\', \'t\')], label=\'$I_m$\', lxy=(X[\'c5\'] + 0.05, (BOX[\'iiwt\'][1] + BOX[\'enc\'][3]) / 2))\nwire(\'lcme\', \'enc\', [E(\'lcme\', \'t\'), E(\'enc\', \'b\')], label=r\'$k_r,s_r,\\pi_r$\', lxy=(X[\'c5\'] + 0.05, (BOX[\'lcme\'][3] + BOX[\'enc\'][1]) / 2))\nwire(\'enc\', \'is\', [E(\'enc\', \'r\'), (BOX[\'is\'][0], r2)])\nyb = BOX[\'fob\'][3] + 0.14\nwire(\'fob\', \'emb\', [E(\'fob\', \'t\', 0.62), (E(\'fob\', \'t\', 0.62)[0], yb), (E(\'emb\', \'t\', 0.3)[0], yb), E(\'emb\', \'t\', 0.3)], dashed=True,\n     label=r\'$B_E,\\ B_F$ (auxiliary information)\', lxy=(2.75, yb + 0.08), ha=\'center\')\nyf = yb + 0.30\nwire(\'iiwt\', \'fob\', [E(\'iiwt\', \'t\', 0.5), (E(\'iiwt\', \'t\', 0.5)[0], yf), (E(\'fob\', \'t\', 0.3)[0], yf), E(\'fob\', \'t\', 0.3)], dashed=True,\n     label=r\'range violated: $D\\leftarrow D+1$, repeat\', lxy=(3.75, yf + 0.08), ha=\'center\')\n\n# ================================================================ receiver\nax.add_patch(Rectangle((0.05, 0.36), W - 0.10, 3.10, fill=False, ec=\'#444\', lw=0.7, ls=(0, (4, 3))))\nax.text(W / 2, 3.62, \'Receiver: decryption, extraction and recovery\', ha=\'center\', va=\'center\', fontsize=9.6, weight=\'bold\')\nq1, q2, q3 = 2.78, 1.70, 0.84\nthumb(\'ris\', f\'{R}/img/Boat_enc.png\', X[\'c0\'], q1, 0.56, r\'$I_s$, $N$\')\nbox(\'dec\', X[\'c1\'], q1, 1.20, 0.56, \'1. Decryption\', [r\'inverse rounds, $\\pi_2^{-1},\\pi_1^{-1}$\'], COL[\'enc\'])\nbox(\'riwt\', X[\'c2\'], q1, 0.62, 0.42, \'2. 5/3 IWT\', [], COL[\'tr\'])\nbands(\'rbands\', X[\'c3\'], q1)\nbox(\'ext\', X[\'c4\'], q1, 1.46, 0.80, \'4. Blind extraction\', [r"$b=c′\\ \\mathrm{mod}\\ 2,\\ \\ c=\\lfloor c′/2\\rfloor$", r"$c=c′\\mp L$ otherwise", r"$H,B_E$ (level 1); $B_F,\\tilde m$ (level $L$)"], COL[\'hide\'])\nbox(\'riiwt\', X[\'c5\'], q1, 1.02, 0.50, \'5. Inverse IWT\', [r\'restored $\\rightarrow I_p$\'], COL[\'rec\'])\nbox(\'rlcme\', X[\'c1\'], q2, 1.20, 0.46, r\'2D-LCM $(K_e,N)$\', [r\'$\\hat p_n\\rightarrow k_r,s_r$;  $x_n\\rightarrow\\pi_1,\\pi_2$\'], COL[\'chaos\'])\nbox(\'rord\', X[\'c3\'], q2, 0.82, 0.50, \'3. Order\', [r\'from LL ($\\sigma,\\kappa$)\'], COL[\'hide\'])\nbox(\'undo\', X[\'c5\'], q2, 1.18, 0.56, \'6. Undo FOB clamp\', [r\'corrections $B_F$ on $R$\'], COL[\'fob\'])\nthumb(\'rec\', f\'{R}/img/Boat_cover.png\', X[\'c6\'], q2, 0.52, \'Recovered $I$\\n(bit-exact)\')\nbox(\'rlcmd\', X[\'c3\'], q3, 1.20, 0.46, r\'2D-LCM $(K_d,N)$\', [r"$k_d=\\hat{p}′_n \\ \\mathrm{mod}\\ 2$"], COL[\'chaos\'])\nbox(\'unm\', X[\'c4\'] + 0.30, q3, 1.02, 0.50, \'Unmasking\', [r\'$m=\\tilde m\\oplus k_d$\'], COL[\'enc\'])\nbox(\'rmsg\', X[\'c6\'] - 0.06, q3, 0.80, 0.30, None, [\'message $m$\'], COL[\'io\'])\n# receiver rlcmd shifted left so the k_d arrow has room\nBOX_shift = None\n\nwire(\'ris\', \'dec\', [E(\'ris\', \'r\'), E(\'dec\', \'l\')])\nwire(\'dec\', \'riwt\', [E(\'dec\', \'r\'), E(\'riwt\', \'l\')], label=\'$I_m$\', lxy=((BOX[\'dec\'][2] + BOX[\'riwt\'][0]) / 2, q1 + 0.09), ha=\'center\')\nwire(\'riwt\', \'rbands\', [E(\'riwt\', \'r\'), E(\'rbands\', \'l\')])\nwire(\'rbands\', \'ext\', [E(\'rbands\', \'r\'), E(\'ext\', \'l\')])\nwire(\'ext\', \'riiwt\', [E(\'ext\', \'r\'), E(\'riiwt\', \'l\')])\nwire(\'rlcme\', \'dec\', [E(\'rlcme\', \'t\'), E(\'dec\', \'b\')], label=r\'$k_r,s_r,\\pi_r$\', lxy=(X[\'c1\'] + 0.05, (BOX[\'rlcme\'][3] + BOX[\'dec\'][1]) / 2))\nwire(\'rbands\', \'rord\', [(X[\'c3\'] - 0.15, BOX[\'rbands\'][1]), (X[\'c3\'] - 0.15, BOX[\'rord\'][3])], label=\'LL\', lxy=(X[\'c3\'] - 0.11, (BOX[\'rbands\'][1] + BOX[\'rord\'][3]) / 2))\nxo2 = BOX[\'ext\'][0] + 0.14\nwire(\'rord\', \'ext\', [E(\'rord\', \'r\'), (xo2, E(\'rord\', \'r\')[1]), (xo2, BOX[\'ext\'][1])])\nxmt = X[\'c4\'] + 0.30\nwire(\'ext\', \'unm\', [(xmt, BOX[\'ext\'][1]), (xmt, BOX[\'unm\'][3])], label=r\'$\\tilde m$\', lxy=(xmt + 0.05, (BOX[\'ext\'][1] + BOX[\'unm\'][3]) / 2))\nwire(\'rlcmd\', \'unm\', [E(\'rlcmd\', \'r\'), E(\'unm\', \'l\')], label=r\'$k_d$\', lxy=((BOX[\'rlcmd\'][2] + BOX[\'unm\'][0]) / 2, q3 + 0.09), ha=\'center\')\nwire(\'unm\', \'rmsg\', [E(\'unm\', \'r\'), E(\'rmsg\', \'l\')])\nwire(\'riiwt\', \'undo\', [E(\'riiwt\', \'b\'), E(\'undo\', \'t\')], label=\'$I_p$\', lxy=(X[\'c5\'] + 0.05, (BOX[\'riiwt\'][1] + BOX[\'undo\'][3]) / 2))\nxb = BOX[\'ext\'][2] - 0.10\nwire(\'ext\', \'undo\', [(xb, BOX[\'ext\'][1]), (xb, E(\'undo\', \'l\', 0.5)[1]), E(\'undo\', \'l\', 0.5)], dashed=True, label=\'$B_F$, $E$, $D$\', lxy=(xb + 0.05, (BOX[\'ext\'][1] + E(\'undo\', \'l\', 0.5)[1]) / 2 + 0.12), ha=\'left\')\nwire(\'undo\', \'rec\', [E(\'undo\', \'r\'), E(\'rec\', \'l\')])\n\n# ================================================================ legend\nitems = [(\'Transform\', COL[\'tr\']), (\'Data hiding / extraction\', COL[\'hide\']), (\'FOB handling\', COL[\'fob\']), (\'2D-LCM key-streams\', COL[\'chaos\']),\n         (\'Encryption / masking\', COL[\'enc\']), (\'Reconstruction\', COL[\'rec\'])]\nfig.canvas.draw(); _r = fig.canvas.get_renderer(); _inv = ax.transData.inverted()\ndef _w(t):\n    tt = ax.text(0, 0, t, fontsize=FC); bb = tt.get_window_extent(_r); tt.remove()\n    (a, _), (b, _) = _inv.transform([(bb.x0, 0), (bb.x1, 0)]); return b - a\nws = [0.2 + 0.05 + _w(t) for t, _ in items]; gap = (W - 0.2 - sum(ws)) / (len(items) - 1); xl = 0.1\nfor (t, c), wd in zip(items, ws):\n    ax.add_patch(Rectangle((xl, 0.10), 0.2, 0.13, fc=c, ec=\'#000\', lw=0.5)); TXT.append((None, ax.text(xl + 0.25, 0.165, t, va=\'center\', fontsize=FC)))\n    xl += wd + gap\n\n# ================================================================ automatic checks\nfig.canvas.draw(); rend = fig.canvas.get_renderer(); inv = ax.transData.inverted(); problems = []\nfor key, t in TXT:\n    bb = t.get_window_extent(rend); (x0, y0), (x1, y1) = inv.transform([(bb.x0, bb.y0), (bb.x1, bb.y1)])\n    if key is not None:\n        X0, Y0, X1, Y1 = BOX[key]\n        if not (x0 >= X0 + 0.02 and x1 <= X1 - 0.02 and y0 >= Y0 + 0.01 and y1 <= Y1 - 0.01): problems.append(f\'text overflows box {key}: "{t.get_text()}"\')\n    else:\n        for k, (X0, Y0, X1, Y1) in BOX.items():\n            if x0 < X1 and x1 > X0 and y0 < Y1 and y1 > Y0: problems.append(f\'label "{t.get_text()}" overlaps box {k}\')\n    if x0 < 0 or x1 > W or y0 < 0 or y1 > H: problems.append(f\'text outside canvas: {t.get_text()}\')\nlabels = [t for k, t in TXT if k is None]\nfor a, b in itertools.combinations(labels, 2):\n    A = a.get_window_extent(rend); B = b.get_window_extent(rend)\n    if A.overlaps(B): problems.append(f\'labels overlap: "{a.get_text()}" / "{b.get_text()}"\')\ndef seg_hits_box(p, q, bx, eps=1e-3):\n    X0, Y0, X1, Y1 = bx; X0 += eps; Y0 += eps; X1 -= eps; Y1 -= eps\n    if abs(p[0] - q[0]) < 1e-9:   # vertical\n        return X0 < p[0] < X1 and max(min(p[1], q[1]), Y0) < min(max(p[1], q[1]), Y1)\n    return Y0 < p[1] < Y1 and max(min(p[0], q[0]), X0) < min(max(p[0], q[0]), X1)\nsegs = []\nfor s, d, pts in WIRES:\n    for p, q in zip(pts[:-1], pts[1:]):\n        if not (abs(p[0] - q[0]) < 1e-9 or abs(p[1] - q[1]) < 1e-9): problems.append(f\'non-orthogonal segment {s}->{d}\')\n        for k, bx in BOX.items():\n            if k not in (s, d) and seg_hits_box(p, q, bx): problems.append(f\'wire {s}->{d} passes through {k}\')\n        segs.append((s, d, p, q))\nfor (s1, d1, p1, q1), (s2, d2, p2, q2) in itertools.combinations(segs, 2):\n    if (s1, d1) == (s2, d2): continue\n    v1 = abs(p1[0] - q1[0]) < 1e-9; v2 = abs(p2[0] - q2[0]) < 1e-9\n    if v1 == v2: continue\n    (vp, vq), (hp, hq) = ((p1, q1), (p2, q2)) if v1 else ((p2, q2), (p1, q1))\n    x = vp[0]; y = hp[1]\n    if min(hp[0], hq[0]) + 1e-6 < x < max(hp[0], hq[0]) - 1e-6 and min(vp[1], vq[1]) + 1e-6 < y < max(vp[1], vq[1]) - 1e-6:\n        problems.append(f\'wires cross: {s1}->{d1} and {s2}->{d2}\')\ndef on_boundary(pt, bx, eps=1e-6):\n    X0, Y0, X1, Y1 = bx; x, y = pt\n    return ((abs(x - X0) < eps or abs(x - X1) < eps) and Y0 - eps <= y <= Y1 + eps) or ((abs(y - Y0) < eps or abs(y - Y1) < eps) and X0 - eps <= x <= X1 + eps)\nfor s_, d_, pts in WIRES:\n    if not on_boundary(pts[0], BOX[s_]): problems.append(f\'wire {s_}->{d_} does not start on the source box\')\n    if not on_boundary(pts[-1], BOX[d_]): problems.append(f\'wire {s_}->{d_} does not end on the target box\')\n# conceptual completeness: dependencies of Algorithms 1-2 (source -> target)\nSPEC = {(\'cover\', \'fob\'), (\'fob\', \'iwt\'), (\'iwt\', \'bands\'), (\'bands\', \'emb\'), (\'bands\', \'ord\'), (\'ord\', \'emb\'), (\'fob\', \'emb\'), (\'msg\', \'msk\'),\n        (\'lcmd\', \'msk\'), (\'msk\', \'emb\'), (\'emb\', \'iiwt\'), (\'iiwt\', \'fob\'), (\'iiwt\', \'im\'), (\'iiwt\', \'enc\'), (\'lcme\', \'enc\'), (\'enc\', \'is\'),\n        (\'ris\', \'dec\'), (\'rlcme\', \'dec\'), (\'dec\', \'riwt\'), (\'riwt\', \'rbands\'), (\'rbands\', \'ext\'), (\'rbands\', \'rord\'), (\'rord\', \'ext\'),\n        (\'ext\', \'riiwt\'), (\'riiwt\', \'undo\'), (\'ext\', \'undo\'), (\'undo\', \'rec\'), (\'ext\', \'unm\'), (\'rlcmd\', \'unm\'), (\'unm\', \'rmsg\')}\ndrawn = {(s, d) for s, d, _ in WIRES}\nif drawn != SPEC: problems.append(f\'dependency mismatch: missing {SPEC - drawn}, extra {drawn - SPEC}\')\nos.makedirs(F, exist_ok=True)\nfig.savefig(f\'{F}/fig12_block_diagram.pdf\'); fig.savefig(f\'{F}/fig12_block_diagram.png\', dpi=600); fig.savefig(f\'{F}/fig12_block_diagram.tif\', dpi=600, pil_kwargs={\'compression\': \'tiff_lzw\'})\nprint(\'PROBLEMS:\' if problems else \'ALL CHECKS PASSED\', *problems, sep=\'\\n  \')\nsys.exit(1 if problems else 0)\n'

In [ ]:
if RUN['scheme']:
    # Fig. 12: block diagram, drawn with the marked / encrypted images of one test image
    d12 = pick('Boat'); I = d12['I']; caps = [max_net_capacity(I, L) for L in range(1, 11)]
    msg = SECRET if any(c['net'] >= SECRET.size for c in caps) else np.random.default_rng(0).integers(0, 2, I.size // 10).astype(np.uint8)
    L12 = next(L for L in range(1, 11) if caps[L - 1]['net'] >= len(msg))
    E12, Im12, _ = sender(I, msg, L12, KE, KD, nonce_of('fig12'), rho=caps[L12 - 1]['rho'])
    R12 = os.path.join(RES, 'fig12'); os.makedirs(f'{R12}/img', exist_ok=True)
    for tag, A in (('cover', I), ('marked', Im12), ('enc', E12)): Image.fromarray(A.astype(np.uint8)).save(f'{R12}/img/Boat_{tag}.png')
    src = FIG12_SRC.replace("from config import OUT as R, FIGS as F", "raise ImportError").replace("'/home/claude/rdh/results', '/home/claude/rdh/figs'", repr(R12) + ', ' + repr(FIGD))
    try:
        try: exec(compile(src, 'fig12', 'exec'), {'__file__': os.path.join(PKG, 'fig12.py'), '__name__': 'fig12'})
        except SystemExit as ex:                      # the drawing script ends with sys.exit(0) after its layout self-check
            if ex.code not in (0, None): raise RuntimeError('layout check failed')
        for ext in ('png', 'pdf', 'tif'):
            p = f'{FIGD}/fig12_block_diagram.{ext}'
            if os.path.exists(p): shutil.move(p, f'{FIGD}/Fig12_block_diagram.{ext}')
        SAVED.append(f'{FIGD}/Fig12_block_diagram.png'); print('Fig. 12: Block diagram of the proposed scheme')
        display(Image.open(f'{FIGD}/Fig12_block_diagram.png').resize((1400, int(1400 * Image.open(f'{FIGD}/Fig12_block_diagram.png').height / Image.open(f'{FIGD}/Fig12_block_diagram.png').width))))
    except (Exception, SystemExit) as ex: print('Fig. 12 could not be drawn:', repr(ex))

## 8. Section 5.2 — key space and key sensitivity (Tables 8–10, Eq. (30), Fig. 14)

In [ ]:
if RUN['keyspace']:
    ks = lambda K, n=64: p_bytes(lcm_sequence(K, n, nonce_of('keysens'))[2])
    # Table 8: smallest effective perturbation (in multiples of the ulp, doubling) over 200 random admissible keys
    rng = np.random.default_rng(7); NK = 200 if not QUICK else 40; minpert = {p: [] for p in ('x0', 'y0', 'a', 'b', 'w')}
    for _ in range(NK):
        K = Key(rng.uniform(-1, 1), rng.uniform(-1, 1), rng.uniform(1, 20), rng.uniform(5, 25), rng.uniform(2, 10)); base = ks(K)
        for p in minpert:
            v = getattr(K, p); k = 1; vv = v
            while k <= 1 << 20:
                vv = v + k * np.spacing(v)
                if not np.array_equal(base, ks(K.replace(**{p: vv}))): break
                k *= 2
            minpert[p].append(float(abs(vv - v)))
    RANGE = {'x0': (2, '[−1, 1)'), 'y0': (2, '[−1, 1)'), 'a': (19, '[1, 20]'), 'b': (20, '[5, 25]'), 'w': (8, '[2, 10]')}
    t8 = pd.DataFrame([[p, RANGE[p][1], f'{np.median(v):.2e}', f'{np.max(v):.2e}', f'{np.log2(RANGE[p][0] / np.max(v)):.1f}'] for p, v in minpert.items()],
                      columns=['Component', 'Admissible range', 'Median smallest effective change δ', f'Max over {NK} keys (δ used)', 'log₂(range/δ)'])
    table(t8, 8, f'Smallest effective perturbation of each key component over {NK} random admissible keys')
    frac = SCAN['frac_l1_pos'] if 'SCAN' in globals() else 1.0
    kbits = sum(np.log2(RANGE[p][0] / np.max(v)) for p, v in minpert.items()) + np.log2(frac)
    print(f'Eq. (30): effective key space = prod(range/delta) x f(lambda1 > 0) = 2^{kbits:.1f}  ->  about 2^{int(np.floor(kbits))}')
    table(pd.DataFrame([['Key space', f'2^{int(np.floor(kbits))}', '10⁵⁶ (≈ 2¹⁸⁶)', '10⁷⁵ (≈ 2²⁴⁹)', '10⁷⁵ (≈ 2²⁴⁹)']],
                       columns=['Scheme', 'Proposed', 'Chai et al. [60]', 'Ping et al. [61]', 'Wang et al. [62]']), 9, 'Key-space comparison',
          'Values of [60]-[62] as reported in those papers.')

    # Table 10: one-ulp key sensitivity on Baboon (65,536-bit secret, same nonce)
    dk = pick('Baboon'); I = dk['I']; N0 = nonce_of('keysens'); caps = [max_net_capacity(I, L) for L in range(1, 11)]
    msg = SECRET if any(c['net'] >= SECRET.size for c in caps) else np.random.default_rng(0).integers(0, 2, I.size // 10).astype(np.uint8)
    L = next(L for L in range(1, 11) if caps[L - 1]['net'] >= len(msg)); rho = caps[L - 1]['rho']
    E, Im, _ = sender(I, msg, L, KE, KD, N0, rho=rho); rows = []
    for p in ('x0', 'y0', 'a', 'b', 'w'):
        v = getattr(KE, p); K1 = KE.replace(**{p: float(np.nextafter(v, np.inf))}); E1, _, _ = sender(I, msg, L, K1, KD, N0, rho=rho)
        a_, b_ = npcr_uaci(E, E1); D1 = decrypt(E, K1, N0); na, _ = npcr_uaci(Im, D1); ps_ = psnr(Im, D1)
        rows.append([p, v, f'{np.spacing(v):.2e}', 'yes' if v + 1e-15 != v else 'no', f'{a_:.3f}', f'{b_:.3f}', 'inf' if np.isinf(ps_) else f'{ps_:.2f}', f'{na:.3f}'])
    table(pd.DataFrame(rows, columns=['Component', 'Value', '1 ulp', '+10⁻¹⁵ changes value?', 'NPCR enc. %', 'UACI enc. %', 'PSNR wrong-key dec. (dB)', 'NPCR wrong-key dec. %']),
          10, f'One-ulp key sensitivity ({dk["name"]}, {len(msg):,}-bit secret, same nonce, L = {L})',
          'A one-ulp change of x0 = 0.3 is absorbed by rounding in the first iteration; Table 8 therefore uses the smallest effective change.')

    # Fig. 14: Key1 = K_e, Key2 = a + 1 ulp, Key3 = b + 1 ulp
    N7 = bytes([7] * 16); K2 = KE.replace(a=float(np.nextafter(KE.a, 9))); K3 = KE.replace(b=float(np.nextafter(KE.b, 99)))
    E1, E2, E3 = (encrypt(I, K, N7) for K in (KE, K2, K3)); D1, D2, D3 = (decrypt(E1, K, N7) for K in (KE, K2, K3))
    fig, axs = plt.subplots(2, 5, figsize=(10, 4.3))
    panels = [(I, f'(a) Plain {dk["name"]}'), (E1, '(b) Enc. Key1'), (E2, '(c) Enc. Key2'), (E3, '(d) Enc. Key3'), (np.abs(E1.astype(int) - E2), '(e) |b - c|'),
              (np.abs(E1.astype(int) - E3), '(f) |b - d|'), (np.abs(E2.astype(int) - E3), '(g) |c - d|'), (D1, '(h) Dec. with Key1'), (D2, '(i) Dec. with Key2'), (D3, '(j) Dec. with Key3')]
    for ax, (A, t) in zip(axs.ravel(), panels): ax.imshow(A, cmap='gray', vmin=0, vmax=255); ax.set_title(t, fontsize=8); ax.axis('off')
    savefig(fig, 14, 'key_sensitivity', 'Key sensitivity with one-ulp key changes')
    print(f'NPCR(b,c) = {npcr_uaci(E1, E2)[0]:.3f} %, NPCR(b,d) = {npcr_uaci(E1, E3)[0]:.3f} %, NPCR(c,d) = {npcr_uaci(E2, E3)[0]:.3f} %; '
          f'PSNR of wrong-key decryptions: {psnr(I, D2):.2f} / {psnr(I, D3):.2f} dB; Key1 exact: {np.array_equal(D1, I)}')

## 9. Section 5.3 — capacity and overhead (Tables 12–13, Fig. 15)

In [ ]:
LMAX = 10 if not QUICK else 6
RATES = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70] if not QUICK else [0.01, 0.05, 0.10, 0.20, 0.30]
def run_capacity(d):
    I = d['I']; MN = I.size; rows = []
    for bands in ('all3', 'hh'):
        for L in range(1, LMAX + 1):
            c = max_net_capacity(I, L, bands)
            rows.append(dict(set=d['group'], image=d['name'], bands=bands, L=L, D=c['D'], rho=c['rho'], n_excl_blocks=c['n_excl_blocks'], len_E=c['len_E'],
                             n_amb=c['n_amb'], len_F=c['len_F'], header=HDR_BITS, gross_bits=c['gross'], net_bits=c['net'], gross_ceiling=c['gross_ceiling'],
                             net_bpp=c['net'] / MN, MN=MN, complexity=complexity(I)))
    return rows
if RUN['capacity'] or RUN['rate_distortion'] or RUN['comparison'] or RUN['security']:
    t0 = time.time(); CAP = pd.DataFrame(sum(pmap(run_capacity, IMGS), [])); CAP.to_csv(f'{RES}/capacity.csv', index=False)
    print(f'capacity: {len(IMGS)} images x {LMAX} levels x 2 band sets in {time.time() - t0:.0f} s')
    BEST = CAP.loc[CAP.groupby(['set', 'image', 'bands']).net_bits.idxmax()]; B3 = BEST[BEST.bands == 'all3']; BH = BEST[BEST.bands == 'hh']

In [ ]:
if RUN['capacity']:
    refn = [d['name'] for d in REF]; s5 = B3[B3.image.isin(refn) & B3.set.isin({d['group'] for d in REF})].sort_values('image')
    t12 = pd.DataFrame([[r.image, int(r.L), int(r.D), int(r.n_excl_blocks), int(r.len_E), int(r.n_amb), int(r.len_F), HDR_BITS,
                         f'{r.gross_bits / r.MN:.3f}', int(r.net_bits), f'{r.net_bpp:.3f}'] for r in s5.itertuples()],
                       columns=['Image', 'L*', 'D', 'Excl. blocks', 'B_E length (bits)', 'Ambig. pixels', 'B_F length (bits)', 'Header', 'Gross (bpp)', 'Net (bits)', 'Net (bpp)'])
    table(t12, 12, 'Overhead and gross/net embedding rate at the capacity-maximising level L*')
    fmt = lambda a: f'{a.mean():.3f} ± {a.std():.3f} [{a.min():.3f}, {a.max():.3f}]' if len(a) > 1 else f'{a.mean():.3f}'
    rows = []
    for g in GROUPS + (['All'] if len(GROUPS) > 1 else []):
        a = B3.net_bpp if g == 'All' else B3[B3.set == g].net_bpp; h = BH.net_bpp if g == 'All' else BH[BH.set == g].net_bpp
        rows.append([glabel(g), len(a), fmt(a), fmt(h), f'{a.mean() / h.mean():.2f}×' if h.mean() > 0 else '–'])
    table(pd.DataFrame(rows, columns=['Group', 'Images', 'Proposed HL+LH+HH', 'HH only', 'Ratio']), 13, 'Maximum net embedding rate (bpp) per image group: three-band versus HH-only embedding')
    print(f'mean maximum net rate over all {len(B3)} images: {B3.net_bpp.mean():.3f} ± {B3.net_bpp.std():.3f} bpp; '
          f'three-band / HH-only = {B3.net_bpp.mean() / BH.net_bpp.mean():.2f}; texture complexity {CAP.complexity.min():.2f}–{CAP.complexity.max():.2f}')
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.1)); c3 = CAP[CAP.bands == 'all3']
    for k, d in enumerate(REF[:8]):
        q = c3[(c3.image == d['name']) & (c3.set == d['group'])].sort_values('L'); axs[0].plot(q.L, q.net_bpp, marker=MK[k % 8], ms=4, color=COL[k % 8], label=d['name'])
    axs[0].set_xlabel('Embedding level L'); axs[0].set_ylabel('Net capacity (bpp)'); axs[0].legend(fontsize=7, ncol=2); axs[0].set_ylim(0, 0.8); axs[0].set_title('(a)')
    mm = c3.groupby(['set', 'image']).agg(net=('net_bpp', 'max'), cx=('complexity', 'first')).reset_index()
    for k, g in enumerate(GROUPS):
        q = mm[mm.set == g]; axs[1].scatter(q.cx, q.net, s=16, color=COL[k % 8], marker=MK[k % 8], label=f'{glabel(g)} ({len(q)})', edgecolor='white', linewidth=0.5)
    axs[1].set_xlabel('Mean absolute gradient of cover (texture complexity)'); axs[1].set_ylabel('Max. net capacity (bpp)'); axs[1].legend(fontsize=7); axs[1].set_title('(b)')
    savefig(fig, 15, 'net_capacity', 'Net capacity against the embedding level (a) and against texture complexity (b)')

## 10. Section 5.4 — rate–distortion with verified reversibility (Table 14, Figs. 16–17)

In [ ]:
def run_rd(d):
    I = d['I']; MN = I.size; rng = np.random.default_rng(zlib.crc32(f"{d['group']}/{d['name']}".encode())); rows = []
    for bands in ('all3', 'hh'):
        q = CAP[(CAP.set == d['group']) & (CAP.image == d['name']) & (CAP.bands == bands)].set_index('L').net_bits
        for r in RATES:
            n = int(round(r * MN)); fits = [L for L in q.index if q[L] >= n]
            if not fits: rows.append(dict(set=d['group'], image=d['name'], bands=bands, rate=r, L=np.nan, psnr=np.nan, ssim=np.nan, mse=np.nan, exact=np.nan)); continue
            best = None; msg = rng.integers(0, 2, n).astype(np.uint8)
            for L in fits[:2]:
                for rho in RHOS:
                    try: Im, info = embed(I, msg, L, bands, rho=rho)
                    except (ValueError, RuntimeError): continue
                    p = psnr(I, Im)
                    if best is None or p > best[0]: best = (p, L, Im)
            p, L, Im = best; m2, I2, _ = extract(Im, bands)
            rows.append(dict(set=d['group'], image=d['name'], bands=bands, rate=r, L=L, psnr=p, ssim=ssim(I, Im), mse=mse(I, Im),
                             exact=bool(np.array_equal(m2, msg) and np.array_equal(I2, I))))
    return rows
if RUN['rate_distortion'] or RUN['comparison']:
    t0 = time.time(); RD = pd.DataFrame(sum(pmap(run_rd, IMGS), [])); RD.to_csv(f'{RES}/rd.csv', index=False)
    RDv = RD[RD.psnr.notna()]; R3 = RDv[RDv.bands == 'all3']
    print(f'{len(RDv):,} rate-distortion runs in {time.time() - t0:.0f} s; bit-exact (message and cover): {int(RDv.exact.sum())}/{len(RDv)}')
if RUN['rate_distortion']:
    rows = []; ngr = {g: sum(d['group'] == g for d in IMGS) for g in GROUPS}
    for rate in [r for r in RATES if r <= 0.60]:            # the paper's table stops at 0.60 bpp (0.70 is still run and counted)
        row = [f'{rate:.2f}']
        for g in GROUPS:
            q = R3[(R3.set == g) & (R3.rate == rate)].psnr
            row.append('–' if len(q) == 0 else (f'{q.mean():.2f}' if len(q) == 1 else f'{q.mean():.2f} ± {q.std():.2f}') + ('' if len(q) == ngr[g] else f' (n={len(q)})'))
        if len(GROUPS) > 1:
            q = R3[R3.rate == rate].psnr; row.append('–' if len(q) == 0 else f'{q.mean():.2f}' + ('' if len(q) == len(IMGS) else f' (n={len(q)})'))
        if any(x != '–' for x in row[1:]): rows.append(row)
    table(pd.DataFrame(rows, columns=['Net rate (bpp)'] + [glabel(g) for g in GROUPS] + (['All'] if len(GROUPS) > 1 else [])), 14,
          'PSNR (dB) of the marked image at fixed net rates (mean ± standard deviation)', 'n = number of images that reach the rate.')
    fig, axs = plt.subplots(1, len(GROUPS), figsize=(2.6 * len(GROUPS) + 0.6, 2.9), sharey=True, squeeze=False); axs = axs[0]
    for ax, g in zip(axs, GROUPS):
        for k, (b, bl) in enumerate((('all3', 'HL+LH+HH (proposed)'), ('hh', 'HH only (ablation)'))):
            q = RDv[(RDv.set == g) & (RDv.bands == b)].groupby('rate').psnr.agg(['mean', 'std', 'count']); q = q[q['count'] >= max(1, 0.5 * q['count'].max())]
            ax.plot(q.index, q['mean'], marker=MK[k], ms=4, color=COL[k], label=bl); ax.fill_between(q.index, q['mean'] - q['std'].fillna(0), q['mean'] + q['std'].fillna(0), color=COL[k], alpha=0.15, lw=0)
        ax.set_title(f'{glabel(g)} ({ngr[g]})'); ax.set_xlabel('Net embedding rate (bpp)'); ax.set_xlim(0, max(RATES) + 0.02)
    axs[0].set_ylabel('PSNR of marked image (dB)'); axs[0].legend(loc='upper right', fontsize=7.5)
    savefig(fig, 16, 'rd_groups', 'PSNR of the marked image against the net rate per group: three-band versus HH-only')
    fig, ax = plt.subplots(figsize=(4.8, 3.3))
    for k, d in enumerate(REF[:10]):
        q = R3[(R3.image == d['name']) & (R3.set == d['group'])].sort_values('rate'); ax.plot(q.rate, q.psnr, marker=MK[k % 8], ms=4, color=COL[k % 8], label=d['name'])
    ax.set_xlabel('Net embedding rate (bpp)'); ax.set_ylabel('PSNR (dB)'); ax.legend(fontsize=7.5)
    savefig(fig, 17, 'rd_reference', 'Rate-distortion curves of the reference images')
    pv = RDv.pivot_table(index=['set', 'image', 'rate'], columns='bands', values='psnr').dropna()
    pv = pv[pv.index.get_level_values('rate').isin([0.05, 0.10, 0.15])]
    if len(pv): print(f'paired PSNR difference three-band - HH-only at 0.05-0.15 bpp: {(pv.all3 - pv.hh).mean():+.2f} dB (n = {len(pv)})')

## 11. Section 5.5 — comparison with related schemes (Tables 11, 15, 16, 17, Figs. 18–19)

In [ ]:
def run_baselines(d):
    I = d['I']; MN = I.size; rng = np.random.default_rng(zlib.crc32(d['name'].encode())); rows = []
    def proposed_at(n):
        msg = rng.integers(0, 2, n).astype(np.uint8); best = None
        q = CAP[(CAP.set == d['group']) & (CAP.image == d['name']) & (CAP.bands == 'all3')].set_index('L').net_bits
        for L in range(1, LMAX + 1):
            if q[L] < n: continue
            for rho in RHOS:
                try: Im, info = embed(I, msg, L, rho=rho)
                except (ValueError, RuntimeError): continue
                p = psnr(I, Im)
                if best is None or p > best[0]: best = (p, Im, msg)
            if best is not None: break
        if best is None: return None
        p, Im, msg = best; m2, I2, _ = extract(Im); return p, bool(np.array_equal(m2, msg) and np.array_equal(I2, I))
    for s in (8, 16, 32):
        n = (I.shape[0] // s) * (I.shape[1] // s); bits = rng.integers(0, 2, n)
        E = zh_encrypt(I, 11); Em = zh_embed(E, bits, s, 22); b1, R1, D1 = zhang_extract(Em, n, s, 11, 22)
        rows.append(dict(set=d['group'], image=d['name'], method='Zhang2011', param=f's={s}', rate=n / MN, psnr_dec=psnr(I, D1), ber=float((b1 != bits).mean()),
                         exact=bool(np.array_equal(R1, I) and (b1 == bits).all())))
        b2, R2, _ = hong_extract(Em, n, s, 11, 22)
        rows.append(dict(set=d['group'], image=d['name'], method='Hong2012', param=f's={s}', rate=n / MN, psnr_dec=psnr(I, D1), ber=float((b2 != bits).mean()),
                         exact=bool(np.array_equal(R2, I) and (b2 == bits).all())))
        pr = proposed_at(n)
        rows.append(dict(set=d['group'], image=d['name'], method='Proposed', param=f'matched s={s}', rate=n / MN, psnr_dec=pr[0] if pr else np.nan, ber=0.0 if pr else np.nan, exact=pr[1] if pr else np.nan))
    Ep, err, K, sv = pp_sender_prepare(I, 5); cap = pp_capacity(err, K); bits = rng.integers(0, 2, cap).astype(np.uint8)
    Em = pp_embed(Ep, err, K, sv, bits, 9); ext = pp_extract(Em, cap, 9); rec, dd = pp_recover(Em, cap, 5)
    rows.append(dict(set=d['group'], image=d['name'], method='Puteaux2018', param='max', rate=cap / MN, psnr_dec=psnr(I, dd), ber=float((ext != bits).mean()),
                     exact=bool(np.array_equal(rec, I) and np.array_equal(ext, bits))))
    b = B3[(B3.set == d['group']) & (B3.image == d['name'])].iloc[0]
    rows.append(dict(set=d['group'], image=d['name'], method='Proposed', param='max', rate=b.net_bpp, psnr_dec=np.nan, ber=np.nan, exact=np.nan))
    return rows
def run_maxrate(d):
    b = B3[(B3.set == d['group']) & (B3.image == d['name'])].iloc[0]; I = d['I']
    rng = np.random.default_rng(zlib.crc32(f"maxrate/{d['group']}/{d['name']}".encode()))
    m = rng.integers(0, 2, int(b.net_bits)).astype(np.uint8); nonce = bytes(rng.integers(0, 256, 16, dtype=np.uint8))
    E, Im, info = sender(I, m, int(b.L), KE, KD, nonce, rho=float(b.rho)); m2, I2, _, _, _ = receiver(E, KE, KD, nonce)
    return dict(set=d['group'], image=d['name'], L=int(b.L), net_bpp=b.net_bpp, psnr=psnr(I, Im), exact=bool(np.array_equal(m2, m) and np.array_equal(I2, I)))
if RUN['comparison']:
    t0 = time.time(); BL = pd.DataFrame(sum(pmap(run_baselines, IMGS), [])); BL.to_csv(f'{RES}/baselines.csv', index=False)
    MX = pd.DataFrame(pmap(run_maxrate, IMGS)); MX.to_csv(f'{RES}/maxrate.csv', index=False)
    print(f'baselines and maximum-rate runs in {time.time() - t0:.0f} s; proposed at maximum rate bit-exact: {int(MX.exact.sum())}/{len(MX)}')
    NI = len(IMGS); rows = []
    for s_ in (32, 16, 8):
        for meth, lab in (('Zhang2011', 'Zhang [14]'), ('Hong2012', 'Hong et al. [15]'), ('Proposed', 'Proposed')):
            q = BL[(BL.method == meth) & BL.param.str.contains(f's={s_}$')]; p = q.psnr_dec.astype(float)
            rows.append([f'{q.rate.mean():.5f}', lab, f'{p.mean():.2f} ± {p.std():.2f}' if len(p) > 1 else f'{p.mean():.2f}', f'{q.ber.astype(float).mean() * 100:.3f}',
                         f"{int((q.exact == True).sum())}/{len(q)}"])
    q = BL[BL.method == 'Puteaux2018']; dp = BL[(BL.method == 'Proposed') & (BL.param == 'max')]
    sd = lambda s: f' ± {s.std():.3f}' if len(s) > 1 else ''
    rows.append([f'{q.rate.mean():.3f}{sd(q.rate)} (max)', 'Puteaux & Puech [16]', f'{q.psnr_dec.astype(float).mean():.2f} (MSB plane carries data)', f'{q.ber.mean() * 100:.3f}', f"{int((q.exact == True).sum())}/{len(q)}"])
    rows.append([f'{dp.rate.mean():.3f}{sd(dp.rate)} (max)', 'Proposed', f'{MX.psnr.mean():.2f}' + (f' ± {MX.psnr.std():.2f}' if len(MX) > 1 else '') + ' (marked image)', '0.000', f'{int(MX.exact.sum())}/{len(MX)} (all runs)'])
    table(pd.DataFrame(rows, columns=['Net rate (bpp)', 'Method', 'PSNR directly decrypted (dB)', 'Bit error (%)', 'Exact (images)']), 15, f'Matched-payload comparison on {NI} images')

    # Table 11: net embedding rate of the proposed scheme and related schemes on the reference images
    REPORTED = {'Xiang & Luo [17]*': {'Airplane': '0.249', 'Baboon': '0.249', 'Boat': '0.249'}, 'Qin et al. [20]*': {'Airplane': '0.780'},
                'Qiu et al. [21]*': {'Airplane': '≥1.226', 'Baboon': '≥1.221', 'Boat': '≥1.221'}, 'Qiu et al. [2]*': {'Airplane': '2.582', 'Boat': '2.020', 'Peppers': '2.223'},
                'Yi & Zhou [18]*': {'Average': '2.203'}}
    names = [d['name'] for d in REF]; bl5 = BL[BL.image.isin(names)]
    def mrow(lab, get):
        v = [get(n) for n in names]; return [lab] + [f'{x:.3f}' for x in v] + [f'{np.mean(v):.3f}']
    rows = [mrow('Puteaux & Puech [16]', lambda n: bl5[(bl5.method == 'Puteaux2018') & (bl5.image == n)].rate.iloc[0]),
            mrow('Zhang [14]', lambda n: bl5[(bl5.method == 'Zhang2011') & (bl5.param == 's=8') & (bl5.image == n)].rate.iloc[0]),
            mrow('Hong et al. [15]', lambda n: bl5[(bl5.method == 'Hong2012') & (bl5.param == 's=8') & (bl5.image == n)].rate.iloc[0]),
            ['Meng et al. [29]†'] + ['≤0.250'] * (len(names) + 1)]
    for lab, vals in REPORTED.items():
        if any(n in vals for n in names) or 'Average' in vals:
            rows.append([lab] + [str(vals.get(n, '–')) for n in names] + [str(vals.get('Average', '–'))])
    rows.append(mrow('Proposed', lambda n: B3[B3.image == n].net_bpp.iloc[0]))
    table(pd.DataFrame(rows, columns=['Method'] + names + ['Average']), 11, 'Net embedding rate (bpp) of the proposed scheme and related schemes',
          '† bound: one bit per HH coefficient. * values reported in the cited papers for the same images (512 x 512), shown only for images present in the input.')

    # Table 16: qualitative comparison
    r25 = R3[R3.rate == 0.25].psnr if 'R3' in globals() else pd.Series(dtype=float)
    table(pd.DataFrame([['Zhang [14]', 'Joint', 'Yes (s ≤ 8)', 'Yes (s ≤ 8)', '1/s² (≤ 0.016)', f"≈ {BL[BL.method == 'Zhang2011'].psnr_dec.mean():.1f} dB"],
                        ['Hong et al. [15]', 'Joint', 'Yes (fewer)', 'Yes (fewer)', '1/s² (≤ 0.016)', f"≈ {BL[BL.method == 'Hong2012'].psnr_dec.mean():.1f} dB"],
                        ['Puteaux & Puech [16]', 'Separable', 'No', 'No', f'{q.rate.mean():.3f} bpp', f'{q.psnr_dec.mean():.1f} dB (exact after prediction)'],
                        ['Meng et al. [29]', "Joint (separable in the authors' terms)", 'No', 'No', 'HH only (see ablation)', 'marked image'],
                        ['Proposed', 'Joint', 'No', 'No', f'{dp.rate.mean():.3f} bpp (mean)',
                         f'marked image, {r25.mean():.1f} dB mean (min {r25.min():.1f} dB) at 0.25 bpp' if len(r25) else 'marked image']],
                       columns=['Method', 'Joint / separable', 'Extraction errors', 'Recovery errors', 'Max. net payload (this study)', 'Image seen by K_e holder before restoration']),
          16, 'Qualitative comparison of RDH-EI methods')

In [ ]:
if RUN['comparison']:
    # Fig. 19: PSNR of the image available to the K_e holder against the net rate
    fig, ax = plt.subplots(figsize=(5.4, 3.5))
    dd = R3.groupby('rate').agg(m=('psnr', 'mean'), n=('psnr', 'count')); dd = dd[dd.n >= max(1, len(IMGS) // 2)]
    pm = BL[(BL.method == 'Proposed') & BL.param.str.startswith('matched')].groupby('rate').psnr_dec.mean()
    xs = list(pm.index) + list(dd.index); ys = list(pm.values) + list(dd.m); o = np.argsort(xs)
    ax.plot(np.array(xs)[o], np.array(ys)[o], marker='o', ms=4, color=COL[0], label='Proposed (directly decrypted = marked image); exact in all runs')
    qz = BL[BL.method == 'Zhang2011'].groupby('rate').psnr_dec.mean(); ez = BL[BL.method == 'Zhang2011'].groupby('rate').ber.mean(); eh = BL[BL.method == 'Hong2012'].groupby('rate').ber.mean()
    ax.plot(qz.index, qz.values, marker='^', ms=5, ls=':', color=COL[2], label='Zhang [14] / Hong et al. [15] (same decrypted image)')
    for x, y in zip(qz.index, qz.values): ax.annotate(f'BER {100 * ez[x]:.1f}% (Z)\n{100 * eh[x]:.1f}% (H)', (x, y), textcoords='offset points', xytext=(0, -20), fontsize=6, ha='center', color='#555')
    pp = BL[BL.method == 'Puteaux2018']; ax.scatter(pp.rate.mean(), pp.psnr_dec.mean(), marker='v', s=30, color=COL[4], label='Puteaux & Puech [16] (before MSB prediction), exact after')
    ax.set_xscale('log'); ax.set_xlabel('Net embedding rate (bpp, log scale)'); ax.set_ylabel('PSNR of image available to K_e holder (dB)'); ax.set_ylim(0, 75); ax.legend(fontsize=6.5, loc='lower left')
    savefig(fig, 19, 'matched_comparison', f'PSNR of the image available to the holder of the encryption key against the net rate on {len(IMGS)} images')

    # Fig. 18: comparison with values TABULATED in the cited papers, our scheme run at exactly those payloads (bit-exact)
    XIANG = dict(bits=[4080, 8160, 12240, 16320, 24480, 32640, 36720, 48960, 65280], Airplane=[52.599, 49.431, 47.889, 46.888, 43.293, 42.071, 40.275, 38.572, 34.898],
                 Boat=[52.451, 49.540, 48.386, 44.722, 41.835, 39.33, 37.516, 34.383, 30.166], Baboon=[42.896, 39.835, 38.041, 36.879, 35.047, 33.541, 32.691, 30.576, 28.086])
    QIUD = dict(Airplane=[(11015, 59.52), (21377, 56.07), (41538, 52.25), (80651, 47.62), (160690, 41.70), (321421, 32.79)],
                Boat=[(10065, 54.49), (20196, 50.11), (40144, 45.63), (80137, 40.92), (160140, 34.84), (320057, 26.60)],
                Baboon=[(10072, 50.06), (20156, 46.21), (40082, 41.89), (80124, 36.37), (160051, 29.41), (320181, 21.65)])
    QIN = dict(Airplane=[(0.29, 51.41), (0.53, 47.79), (0.66, 45.18), (0.76, 43.05), (0.78, 41.50)])
    present = [n for n in ('Airplane', 'Boat', 'Baboon') if n.lower() in BYNAME and BYNAME[n.lower()]['I'].shape == (512, 512)]
    if not present: print('Fig. 18 needs Airplane, Boat or Baboon (512 x 512) among the input images: the cited papers tabulate values only for these images. Skipped.')
    else:
        def run_rep(I, nbits, name):
            m = np.random.default_rng(zlib.crc32(f'{name}{nbits}'.encode())).integers(0, 2, nbits).astype(np.uint8)
            N = np.random.default_rng(zlib.crc32(('rep' + name + str(nbits)).encode())).integers(0, 256, 16, dtype=np.uint8).tobytes()
            em = m ^ payload_mask(KD, nbits, N); best = None; feas = 0
            for L in range(1, 16):
                okL = False
                for rho in RHOS:
                    try: Im, info = embed(I, em, L, rho=rho)
                    except (ValueError, RuntimeError): continue
                    okL = True; p = psnr(I, Im)
                    if best is None or p > best[0]: best = (p, L, rho, Im)
                feas += okL
                if feas == 2: break
            if best is None: return None
            p, L, rho, Im = best; E = encrypt(Im, KE, N); m2, I2, *_ = receiver(E, KE, KD, N)
            return dict(psnr=p, L=L, exact=bool(np.array_equal(m2, m) and np.array_equal(I2, I)))
        rows = []
        for name in present:
            I = BYNAME[name.lower()]['I'].astype(np.int64)
            jobs = [('Xiang2018', b) for b in XIANG['bits']] + [('Qiu2020dual', b) for b, _ in QIUD[name][:5]] + ([('Qin2018', int(round(t * I.size))) for t, _ in QIN['Airplane']] if name == 'Airplane' else [])
            for src, b in jobs:
                r = run_rep(I, b, name); rows.append(dict(image=name, source=src, bits=b, rate=b / I.size, **(r or dict(psnr=np.nan, L=None, exact=None))))
        RM = pd.DataFrame(rows); RM.to_csv(f'{RES}/reported_match.csv', index=False)
        print('runs at the tabulated payloads, bit-exact:', int((RM.exact == True).sum()), '/', int(RM.psnr.notna().sum()))
        fig = plt.figure(figsize=(10.5, 6.6)); gs = fig.add_gridspec(2, 3, hspace=0.42, wspace=0.28); diffs = {}
        for k, img in enumerate(present):
            ax = fig.add_subplot(gs[0 if k < 2 else 1, k if k < 2 else 0])
            own = pd.concat([R3[R3.image == img][['rate', 'psnr']], RM[(RM.image == img) & RM.psnr.notna()][['rate', 'psnr']]]).groupby('rate').psnr.max().reset_index()
            ax.plot(own.rate, own.psnr, '-o', ms=2.8, lw=1.8, color=COL[0], label='Proposed (measured, exact)', zorder=5)
            ax.plot(np.array(XIANG['bits']) / 262144, XIANG[img], '--s', ms=4, mfc='white', lw=1, color=COL[7], label='Xiang & Luo [17] (reported)')
            q = np.array(QIUD[img]); ax.plot(q[:, 0] / 262144, q[:, 1], '--^', ms=4.5, mfc='white', lw=1, color=COL[2], label='Qiu et al. [21] (reported)')
            if img == 'Airplane': qi = np.array(QIN['Airplane']); ax.plot(qi[:, 0], qi[:, 1], '--D', ms=4, mfc='white', lw=1, color=COL[3], label='Qin et al. [20] (reported)')
            ax.set_title(f"({'abc'[k]}) {img}", fontsize=9.5); ax.set_xlabel('Embedding rate (bpp)'); ax.set_ylabel('PSNR (dB)'); ax.set_xlim(0, 1.3); ax.set_ylim(20, 62); ax.legend(fontsize=6.8)
            for src, pts in (('Xiang & Luo', list(zip(XIANG['bits'], XIANG[img]))), ('Qiu et al. [21]', QIUD[img]), ('Qin et al.', [(int(round(t * 262144)), p) for t, p in QIN.get(img, [])])):
                for b, p in pts:
                    r = RM[(RM.image == img) & (RM.bits == b)]
                    if len(r) and not np.isnan(r.psnr.iloc[0]): diffs.setdefault(src, []).append(float(r.psnr.iloc[0]) - p)
        ax = fig.add_subplot(gs[1, 1:])
        if 'airplane' in BYNAME:
            prop = B3[B3.image == 'Airplane'].net_bpp.max(); ppa = BL[(BL.method == 'Puteaux2018') & (BL.image == 'Airplane')].rate.max()
            bars = [('Zhang [14] / Hong et al. [15]', 1 / 64, 'measured'), ('Xiang & Luo [17]', 65280 / 262144, 'reported'), ('Meng et al. [29]', 0.25, 'bound'),
                    ('Proposed', prop, 'measured'), ('Qin et al. [20]', 0.78, 'reported'), ('Puteaux & Puech [16]', ppa, 'measured'),
                    ('Qiu et al. [21]', 321421 / 262144, 'reported (≥)'), ('Qiu et al. [2]', 676916 / 262144, 'reported')]
            for i, (n, v, kind) in enumerate(bars):
                c = COL[0] if n == 'Proposed' else '#8a8a8a'
                ax.bar(i, v, color=c if kind == 'measured' else 'white', edgecolor='#444', hatch=None if kind == 'measured' else ('...' if kind == 'bound' else '///'), lw=0.8)
                ax.text(i, v + 0.04, ('≥' if '≥' in kind else ('≤' if kind == 'bound' else '')) + f'{v:.3f}', ha='center', fontsize=7.5)
            ax.set_xticks(range(len(bars))); ax.set_xticklabels([n for n, _, _ in bars], rotation=18, ha='right', fontsize=7.5); ax.set_ylim(0, 2.9)
            ax.set_ylabel('Maximum embedding rate (bpp)'); ax.set_title('(d) Maximum embedding rate on Airplane', fontsize=9.5)
        else: ax.axis('off')
        savefig(fig, 18, 'comparison_reported', 'Comparison with related schemes at the payloads tabulated in the cited papers')
        for k_, v in diffs.items(): print(f'proposed - {k_}: {min(v):+.2f} to {max(v):+.2f} dB over {len(v)} matched payloads')

    # Table 17: run time and peak memory (reference images, proposed scheme at 0.10 bpp)
    rows = []
    for d in REF[:5]:
        I = d['I']; rng = np.random.default_rng(5); n = int(0.1 * I.size); msg = rng.integers(0, 2, n).astype(np.uint8); N0 = b'\x01' * 16
        q = CAP[(CAP.set == d['group']) & (CAP.image == d['name']) & (CAP.bands == 'all3')].set_index('L').net_bits
        L = 2 if q.get(2, 0) >= n else next((L for L in q.index if q[L] >= n), None)
        def prof(f):
            tracemalloc.start(); t = time.perf_counter(); out = f(); dt = time.perf_counter() - t; peak = tracemalloc.get_traced_memory()[1]; tracemalloc.stop(); return out, dt, peak / 2 ** 20
        if L is not None:
            (E, Im, info), t1, m1 = prof(lambda: sender(I, msg, L, KE, KD, N0)); _, t2, m2 = prof(lambda: receiver(E, KE, KD, N0))
            rows.append(dict(image=d['name'], method=f'Proposed (0.10 bpp, L = {L})', s=t1, r=t2, ms=m1, mr=m2))
            (_, a1, b1_), (_, a2, b2_) = prof(lambda: encrypt(Im, KE, N0)), prof(lambda: decrypt(E, KE, N0))
            rows.append(dict(image=d['name'], method='of which: 2D-LCM cipher', s=a1, r=a2, ms=b1_, mr=b2_))
        s = 8; nb_ = (I.shape[0] // s) * (I.shape[1] // s); bits = rng.integers(0, 2, nb_)
        Em, t1, m1 = prof(lambda: zh_embed(zh_encrypt(I, 11), bits, s, 22)); _, t2, m2 = prof(lambda: zhang_extract(Em, nb_, s, 11, 22))
        rows.append(dict(image=d['name'], method='Zhang [14] (s = 8)', s=t1, r=t2, ms=m1, mr=m2))
        _, t2, m2 = prof(lambda: hong_extract(Em, nb_, s, 11, 22)); rows.append(dict(image=d['name'], method='Hong et al. [15] (s = 8)', s=t1, r=t2, ms=m1, mr=m2))
        def ppsend():
            Ep, err, K, sv = pp_sender_prepare(I, 5); cap = pp_capacity(err, K); return pp_embed(Ep, err, K, sv, rng.integers(0, 2, cap).astype(np.uint8), 9), cap
        (Emp, cap), t1, m1 = prof(ppsend); _, t2, m2 = prof(lambda: (pp_extract(Emp, cap, 9), pp_recover(Emp, cap, 5)))
        rows.append(dict(image=d['name'], method='Puteaux and Puech [16] (max.)', s=t1, r=t2, ms=m1, mr=m2))
    rt = pd.DataFrame(rows); rt.to_csv(f'{RES}/runtime.csv', index=False); g_ = rt.groupby('method', sort=False).mean(numeric_only=True)
    table(pd.DataFrame([[m, f'{r.s:.3f}', f'{r.r:.3f}', f'{r.ms:.1f}', f'{r.mr:.1f}'] for m, r in g_.iterrows()],
                       columns=['Method', 'Sender (s)', 'Receiver (s)', 'Peak memory sender (MB)', 'Peak memory receiver (MB)']), 17,
          'Mean run time and peak memory (reference images; proposed scheme at 0.10 bpp)', 'Absolute times depend on the machine; unoptimised reference implementations.')

## 12. Sections 5.6–5.7 — quality, statistical security, key separation (Tables 18–22, Figs. 20–21)

In [ ]:
def run_security(d):
    I = d['I']; MN = I.size; g, name = d['group'], d['name']; isref = any(r['name'] == name and r['group'] == g for r in REF)
    q = CAP[(CAP.set == g) & (CAP.image == name) & (CAP.bands == 'all3')].sort_values('L')
    msg = SECRET if (g == 'SIPI5' or (INPUT_MODE != 'paper' and (q.net_bits >= SECRET.size).any())) else \
          np.random.default_rng(zlib.crc32(name.encode())).integers(0, 2, MN // 10).astype(np.uint8)
    cc = q[q.net_bits >= len(msg)]
    if not len(cc): return None
    L = int(cc.L.iloc[0]); rho = 2.0 if pd.isna(cc.rho.iloc[0]) else float(cc.rho.iloc[0]); N0 = nonce_of(g + name)
    E, Im, info = sender(I, msg, L, KE, KD, N0, rho=rho); m2, I2, Im2, _, _ = receiver(E, KE, KD, N0)
    RND = np.random.default_rng(zlib.crc32(('rnd' + name).encode())).integers(0, 256, I.shape).astype(np.uint8)   # truly random bytes, same size
    ok = bool(np.array_equal(m2, msg) and np.array_equal(I2, I))
    rng = np.random.default_rng(1); npc, uac = [], []
    for _ in range(10 if (g == 'SIPI5' or (INPUT_MODE != 'paper' and isref)) else 2):
        I1 = I.copy(); i, j = rng.integers(0, I.shape[0]), rng.integers(0, I.shape[1]); I1[i, j] = (int(I1[i, j]) + 1) % 256
        E1, _, _ = sender(I1, msg, L, KE, KD, N0, rho=rho); a, b = npcr_uaci(E, E1); npc.append(a); uac.append(b)
    E2, _, _ = sender(I, msg, L, KE, KD, nonce_of('fresh' + name), rho=rho); na, ua = npcr_uaci(E, E2)
    cp, ce = corr_all(I), corr_all(E)
    nc = float(np.sum(m2.astype(float) * msg) / np.sqrt(np.sum(msg.astype(float) ** 2) * np.sum(m2.astype(float) ** 2)))
    # key separation: K_e only / K_d only / both
    Nk = nonce_of('keysep' + name)      # key-separation experiment uses its own nonce (Table 22)
    Ek, _, _ = sender(I, msg, L, KE, KD, Nk, rho=rho); Imk = decrypt(Ek, KE, Nk); em, Ik, _ = extract(Imk)
    mk, Ib, _, _, _ = receiver(Ek, KE, KD, Nk); both_msg = bool(np.array_equal(mk, msg)); both_cover = bool(np.array_equal(Ib, I))
    try: extract(Ek); kd_only = 'extracts garbage'
    except Exception: kd_only = 'fails (no valid header)'
    if isref:
        os.makedirs(f'{RES}/img', exist_ok=True)
        for tag, A in (('cover', I), ('marked', Im), ('enc', E)): Image.fromarray(A.astype(np.uint8)).save(f'{RES}/img/{g}__{name}_{tag}.png')
    return dict(set=g, image=name, ref=isref, payload_bits=len(msg), L=L, psnr_marked=psnr(I, Im), ssim_marked=ssim(I, Im), mse_marked=mse(I, Im), nc=nc,
                mse_rec=mse(I, I2), ber=float(np.mean(m2 != msg)), exact=ok, H_plain=entropy(I), H_enc=entropy(E), H_local=local_entropy(E), chi2=chi2_hist(E),
                H_local_pos=local_entropy_random_pos(E), H_local_rand=local_entropy(RND), chi2_rand=chi2_hist(RND),
                **{f'c{k}_p': v for k, v in cp.items()}, **{f'c{k}_e': v for k, v in ce.items()},
                NPCR=float(np.mean(npc)), NPCR_min=float(np.min(npc)), UACI=float(np.mean(uac)), NPCR_nonce=na, UACI_nonce=ua,
                ke_psnr=psnr(I, Imk), ke_cover=bool(np.array_equal(Ik, I)), ke_ber=float(np.mean(em[:len(msg)] != msg)), kd_only=kd_only, both_msg=both_msg, both_cover=both_cover)
if RUN['security']:
    t0 = time.time(); SEC = pd.DataFrame([r for r in pmap(run_security, IMGS) if r]); SEC.to_csv(f'{RES}/security.csv', index=False)
    print(f'security run on {len(SEC)} images in {time.time() - t0:.0f} s; message and cover bit-exact: {int(SEC.exact.sum())}/{len(SEC)}')
    S5 = SEC[SEC.ref].copy(); msgbits = int(S5.payload_bits.iloc[0])
    rows = [[r.image, r.L, f'{r.psnr_marked:.2f}', f'{r.ssim_marked:.4f}', f'{r.mse_marked:.4f}', f'{r.nc:.4f}' + (' (bit-exact)' if r.exact else ''),
             f'{r.mse_rec:g}' + (' (PSNR = ∞)' if r.mse_rec == 0 else ''), f'{r.ber:g}'] for r in S5.itertuples()]
    rows.append(['Average', '', f'{S5.psnr_marked.mean():.2f}', f'{S5.ssim_marked.mean():.4f}', f'{S5.mse_marked.mean():.4f}', f'{S5.nc.mean():.4f}', f'{S5.mse_rec.mean():g}', f'{S5.ber.mean():g}'])
    table(pd.DataFrame(rows, columns=['Image', 'L', 'PSNR cover/marked (dB)', 'SSIM cover/marked', 'MSE cover/marked', 'NC secret', 'MSE cover/recovered', 'BER secret']),
          18, f'Marked-image quality and reversibility ({msgbits:,}-bit secret)')
    rows = [[r.image, f'{r.H_plain:.4f}', f'{r.H_enc:.4f}', f'{r.H_local:.4f}', f'{r.H_local_pos:.4f}', f'{r.chi2:.1f}'] for r in S5.itertuples()]
    rows.append(['Mean', f'{S5.H_plain.mean():.4f}', f'{S5.H_enc.mean():.4f}', f'{S5.H_local.mean():.4f}', f'{S5.H_local_pos.mean():.4f}', f'{S5.chi2.mean():.1f}'])
    table(pd.DataFrame(rows, columns=['Image', 'Entropy, plain', 'Entropy, encrypted', 'Local entropy, encrypted (non-overlapping blocks)',
                                      'Local entropy, encrypted (random block positions)', 'χ², encrypted']), 19,
          'Entropy and χ² statistic of the plain and encrypted images',
          'Local entropy: mean entropy of k = 30 random 44 x 44 blocks. The text of Sec. 5.7 specifies non-overlapping blocks; the column published in Table 19 '
          'was computed with random block positions (blocks may overlap), shown in the second local-entropy column. χ² critical value 293.25 (255 df, α = 0.05).')
    from scipy.stats import ks_2samp
    print(f'all {len(SEC)} images: local entropy {SEC.H_local.mean():.5f} ± {SEC.H_local.std():.5f} (ciphertexts) vs {SEC.H_local_rand.mean():.5f} ± {SEC.H_local_rand.std():.5f} '
          f'(random bytes), KS p = {ks_2samp(SEC.H_local, SEC.H_local_rand).pvalue:.2f}; chi2 below 293.25 for {int((SEC.chi2 < 293.25).sum())}/{len(SEC)} ciphertexts and '
          f'{int((SEC.chi2_rand < 293.25).sum())}/{len(SEC)} random images, KS p = {ks_2samp(SEC.chi2, SEC.chi2_rand).pvalue:.2f}')
    table(pd.DataFrame([[r.image] + [f'{getattr(r, c):.4f}' for c in ('cH_p', 'cV_p', 'cD_p', 'cH_e', 'cV_e', 'cD_e')] for r in S5.itertuples()],
                       columns=['Image', 'Plain H', 'Plain V', 'Plain D', 'Encrypted H', 'Encrypted V', 'Encrypted D']), 20, 'Correlation coefficients of adjacent pixels')
    rows = [[r.image, f'{r.NPCR:.3f}', f'{r.NPCR_min:.3f}', f'{r.UACI:.3f}', f'{r.NPCR_nonce:.3f}', f'{r.UACI_nonce:.3f}'] for r in S5.itertuples()]
    rows.append(['Mean', f'{S5.NPCR.mean():.3f}', f'{S5.NPCR_min.min():.3f}', f'{S5.UACI.mean():.3f}', f'{S5.NPCR_nonce.mean():.3f}', f'{S5.UACI_nonce.mean():.3f}'])
    table(pd.DataFrame(rows, columns=['Image', 'NPCR (mean)', 'NPCR (min)', 'UACI (mean)', 'NPCR, fresh nonce', 'UACI, fresh nonce']), 21, 'NPCR and UACI (%)',
          'Wu-Noonan-Agaian critical values for 512 x 512 (α = 0.05): NPCR ≥ 99.5893 %, UACI in [33.3730, 33.5541] %.')
    table(pd.DataFrame([[r.image, r.L, f'{r.ke_psnr:.2f}', 'yes' if r.ke_cover else 'no', f'{r.ke_ber:.3f}', r.kd_only, 'yes' if r.both_msg else 'no', 'yes' if r.both_cover else 'no']
                        for r in S5.itertuples()], columns=['Image', 'L', 'K_e only: marked-image PSNR (dB)', 'K_e only: cover exact', 'K_e only: message BER', 'K_d only',
                                                           'K_e + K_d: message exact', 'K_e + K_d: cover exact']), 22, f'Operations available to each receiver ({msgbits:,}-bit secret)')
    if len(GROUPS) > 1 or len(SEC) > len(S5):
        rows = []
        for g in GROUPS:
            q = SEC[SEC.set == g]; mm = lambda c, f='{:.4f}': f'{f.format(q[c].mean())} ({f.format(q[c].min())}–{f.format(q[c].max())})'
            rows.append([glabel(g), len(q), mm('H_enc'), mm('H_local'), mm('chi2', '{:.1f}'), f"{q[['cH_e', 'cV_e', 'cD_e']].abs().max(axis=1).mean():.4f}",
                         mm('NPCR', '{:.3f}'), mm('UACI', '{:.3f}'), f'{q.NPCR_nonce.mean():.3f} / {q.UACI_nonce.mean():.3f}', f'{int(q.exact.sum())}/{len(q)}'])
        table(pd.DataFrame(rows, columns=['Group', 'n', 'Entropy', 'Local entropy', 'χ²', 'max|r| H/V/D', 'NPCR % (1-pixel)', 'UACI % (1-pixel)', 'NPCR/UACI % (fresh nonce)', 'Exact']),
              'Sec5.7_all_images_security', 'Statistical security of all test images, per group (Sec. 5.7)')

In [ ]:
if RUN['security']:
    ld = lambda r, tag: np.asarray(Image.open(f"{RES}/img/{r.set}__{r.image}_{tag}.png"))
    rows5 = list(S5.itertuples())[:5]
    fig, axs = plt.subplots(len(rows5), 6, figsize=(9, 1.65 * len(rows5) + 0.3), squeeze=False)
    for i, r in enumerate(rows5):
        for k, tag in enumerate(('cover', 'marked', 'enc')):
            A = ld(r, tag); axs[i, 2 * k].imshow(A, cmap='gray', vmin=0, vmax=255); axs[i, 2 * k].axis('off')
            a = axs[i, 2 * k + 1]; a.bar(np.arange(256), np.bincount(A.ravel(), minlength=256), width=1.0, color=COL[0], linewidth=0)
            a.set_yticks([]); a.set_xlim(0, 255); a.grid(False); a.tick_params(labelsize=6)
            if i == 0: axs[0, 2 * k].set_title({'cover': 'Cover', 'marked': 'Marked', 'enc': 'Encrypted stego'}[tag], fontsize=9)
        axs[i, 0].text(-20, A.shape[0] / 2, r.image, rotation=90, va='center', ha='right', fontsize=9)
    savefig(fig, 20, 'histograms', 'Cover, marked and encrypted images and their histograms')
    DIRS = ((0, 1, 'H'), (1, 0, 'V'), (1, 1, 'D')); rng = np.random.default_rng(0)
    r_all = lambda A, dy, dx: np.corrcoef(A[:A.shape[0] - dy, :A.shape[1] - dx].ravel(), A[dy:, dx:].ravel())[0, 1]
    fig = plt.figure(figsize=(7.4, 1.0 * len(rows5) + 0.4)); gs = fig.add_gridspec(len(rows5), 8, wspace=0.3, hspace=0.25)
    for i, r in enumerate(rows5):
        Ic = ld(r, 'cover').astype(float); Ec = ld(r, 'enc').astype(float)
        for c, (A, t) in enumerate(((Ic, 'Cover'), (Ec, 'Encrypted'))):
            ax = fig.add_subplot(gs[i, c]); ax.imshow(A, cmap='gray', vmin=0, vmax=255); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            if i == 0: ax.set_title(t, fontsize=8)
            if c == 0: ax.set_ylabel(r.image, fontsize=8)
        for k, (A, col, lab) in enumerate(((Ic, COL[0], 'Cover'), (Ec, COL[1], 'Encrypted'))):
            for dI, (dy, dx, dl) in enumerate(DIRS):
                ii = rng.integers(0, A.shape[0] - dy, 5000); jj = rng.integers(0, A.shape[1] - dx, 5000); ax = fig.add_subplot(gs[i, 2 + 3 * k + dI])
                ax.scatter(A[ii, jj], A[ii + dy, jj + dx], s=0.15, color=col, alpha=0.6, lw=0, rasterized=True)
                ax.set_xlim(0, 255); ax.set_ylim(0, 255); ax.set_aspect('equal'); ax.set_xticks([0, 255]); ax.set_yticks([0, 255]); ax.tick_params(labelsize=5, length=1.5, pad=1); ax.grid(False)
                if dI > 0: ax.set_yticklabels([])
                if i < len(rows5) - 1: ax.set_xticklabels([])
                ax.set_title(f'r = {r_all(A, dy, dx):.4f}', fontsize=6, pad=2)
                if i == 0: ax.text(0.5, 1.32, f'{lab}: {dl}', transform=ax.transAxes, ha='center', fontsize=7.5)
    savefig(fig, 21, 'adjacent_pixel_correlation', 'Adjacent-pixel distribution of the cover and encrypted images (5,000 random pairs shown)')

## 13. Section 5.7.6 — chosen-plaintext attack, affinity and key reuse (Fig. 22)

In [ ]:
def legacy_encrypt(P, K): return encrypt(P, K, b'', legacy=True)
def cpa_attack(oracle, n_side, target_C):
    # equivalent-key recovery for E(P) = C3 P2 C2 P1 C1 P + b (mod 256), C = running-sum matrix, from n + 1 chosen images
    n = n_side * n_side; shp = (n_side, n_side); b = oracle(np.zeros(shp, np.uint8)).ravel().astype(np.int64)
    Cinv = lambda z: np.diff(np.concatenate([[0], z])) % 256; base = Cinv(b)
    pi1 = np.empty(n, np.int64); member = np.zeros(n, np.int64)
    for k in range(n):
        u = np.zeros(n, np.int64); u[k] = 1; P = np.diff(np.concatenate([[0], u])) % 256
        z = (Cinv(oracle(P.reshape(shp).astype(np.uint8)).ravel().astype(np.int64)) - base) % 256; S = z != 0; pi1[k] = n - S.sum(); member += S
    rank = lambda v: np.argsort(np.argsort(v, kind='stable'), kind='stable'); pi1 = rank(pi1); pi2inv = rank(member - 1)
    z = (Cinv(target_C.ravel().astype(np.int64)) - base) % 256; y = np.zeros(n, np.int64); y[pi2inv] = z
    y = np.diff(np.concatenate([[0], y])) % 256; x = y[pi1]; P = np.diff(np.concatenate([[0], x])) % 256
    return P.reshape(shp).astype(np.uint8), int(n + 1)
if RUN['attacks']:
    dT = pick('Boat'); res = {}; t0 = time.time(); FIGS22 = None
    for side in ((64, 128) if not QUICK else (32, 64)):
        T = np.asarray(Image.fromarray(dT['I']).resize((side, side), Image.LANCZOS))
        CT = legacy_encrypt(T, KE); P1, q = cpa_attack(lambda P: legacy_encrypt(P, KE), side, CT)
        cnt = [0]
        def oracle(P):
            cnt[0] += 1; return encrypt(P, KE, nonce_of(f'q{cnt[0]}'))
        CT2 = encrypt(T, KE, nonce_of('target')); P2, _ = cpa_attack(oracle, side, CT2)
        Nf = nonce_of('fixed'); CT3 = encrypt(T, KE, Nf); P3, _ = cpa_attack(lambda P: encrypt(P, KE, Nf), side, CT3)
        res[side] = dict(queries=q, legacy_exact=bool(np.array_equal(P1, T)), fresh_npcr=npcr_uaci(T, P2)[0], fresh_psnr=psnr(T, P2), reuse_psnr=psnr(T, P3), reuse_npcr=npcr_uaci(T, P3)[0])
        FIGS22 = (T, CT, P1, CT2, P2, q)
    rng = np.random.default_rng(3); aff = {'legacy': [], 'revised': []}
    for _ in range(20):
        P1_, P3_, Dd = (rng.integers(0, 256, (128, 128), dtype=np.uint8) for _ in range(3)); P2_ = P1_ - Dd; P4_ = P3_ - Dd
        for tag, f in (('legacy', lambda P: legacy_encrypt(P, KE)), ('revised', lambda P: encrypt(P, KE, b'N' * 16))):
            aff[tag].append(float(np.mean((f(P1_).astype(np.int64) - f(P2_)) % 256 == (f(P3_).astype(np.int64) - f(P4_)) % 256)))
    imgs = [d['I'] for d in REF[:5]]; leak = {'legacy': [], 'revised': []}
    for i in range(len(imgs)):
        for j in range(i + 1, len(imgs)):
            if imgs[i].shape != imgs[j].shape: continue
            for tag, f in (('legacy', lambda P, k: legacy_encrypt(P, KE)), ('revised', lambda P, k: encrypt(P, KE, nonce_of(f'm{k}')))):
                Ci, Cj, Ck, Cl = f(imgs[i], i), f(imgs[j], j), f(imgs[i] + 1, 100 + i), f(imgs[j] + 1, 200 + j)
                leak[tag].append(float(np.mean((Ck.astype(np.int64) - Ci) % 256 == (Cl.astype(np.int64) - Cj) % 256)))
    rows = [[f'{s} x {s}', f"{r['queries']:,}", 'yes' if r['legacy_exact'] else 'no', f"{r['fresh_npcr']:.2f} % / {r['fresh_psnr']:.2f} dB", f"{r['reuse_psnr']:.2f} dB"] for s, r in res.items()]
    table(pd.DataFrame(rows, columns=['Target', 'Chosen images (n + 1)', 'Fixed-key additive cipher recovered exactly', 'Proposed, fresh nonces: NPCR / PSNR vs target', 'Proposed, nonce reuse: PSNR']),
          'Sec5.7.6_cpa', 'Chosen-plaintext equivalent-key attack (Sec. 5.7.6)')
    print(f"affinity identity holds at {100 * np.mean(aff['legacy']):.2f} % of pixels (fixed-key additive cipher) vs {100 * np.mean(aff['revised']):.2f} % (proposed, same nonce; chance 0.39 %)")
    if leak['legacy']: print(f"multi-image, fixed key: identical ciphertext differences at {100 * np.mean(leak['legacy']):.2f} % (fixed-key) vs {100 * np.mean(leak['revised']):.2f} % (proposed)")
    T, CT, P1, CT2, P2, q = FIGS22; s = T.shape[0]
    fig, axs = plt.subplots(1, 5, figsize=(10, 2.3))
    for ax, (A, t) in zip(axs, [(T, f'(a) Target plaintext\n({s} x {s})'), (CT, '(b) Fixed-key additive\nvariant (affine)'), (P1, f'(c) Recovered by CPA\n({q:,} chosen images)'),
                                (CT2, '(d) Proposed cipher\n(fresh nonce)'), (P2, '(e) Same CPA against\nproposed cipher')]):
        ax.imshow(A, cmap='gray', vmin=0, vmax=255); ax.set_title(t, fontsize=7.5); ax.axis('off')
    savefig(fig, 22, 'cpa', f'Chosen-plaintext equivalent-key attack ({time.time() - t0:.0f} s)')

## 14. Section 5.7.7 — finite precision: cycle search

In [ ]:
if RUN['cycles']:
    rng = np.random.default_rng(11); keys = [KE] + [Key(rng.uniform(-1, 1), rng.uniform(-1, 1), rng.uniform(1, 20), rng.uniform(5, 25), rng.uniform(2, 10)) for _ in range(7)]
    rows = []; t0 = time.time()
    for K in keys:
        c64 = int(_lib.lcm_cycle(K.x0, K.y0, K.a, K.b, K.w, 2000, CYCLE_ITERATIONS)); c32 = int(_lib.lcm_cycle_f32(K.x0, K.y0, K.a, K.b, K.w, 2000, 1 << 34))
        rows.append([', '.join(f'{v:.4g}' for v in K.tup()), 'none found' if c64 == 0 else f'{c64:,}', f'{c32:,}' if c32 else 'none found']); print(rows[-1], f'{time.time() - t0:.0f} s', flush=True)
    table(pd.DataFrame(rows, columns=['Key (x0, y0, a, b, w)', f'Cycle length, float64 (searched {CYCLE_ITERATIONS:.3g} iterations)', 'Cycle length, float32']),
          'Sec5.7.7_cycles', 'Brent cycle detection on the exact state (Sec. 5.7.7)')
    c32s = [int(r[2].replace(',', '')) for r in rows if r[2] != 'none found']
    if c32s: print(f'single precision: periods {min(c32s):,}–{max(c32s):,}; one 512 x 512 image needs 3MN + 3 = {3 * 512 * 512 + 3:,} iterates')

## 15. Check against the published values (`'paper'` mode)

In [ ]:
if INPUT_MODE == 'paper':
    chk = []
    def ck(what, paper, got, tol=0.0):
        try: ok = abs(float(paper) - float(got)) <= tol + 1e-12
        except Exception: ok = str(paper) == str(got)
        chk.append([what, paper, got, 'match' if ok else 'DIFFERENT'])
    g = lambda name: globals().get(name)
    if g('B3') is not None:
        ck('Table 13: mean max net rate, all 79 (bpp)', 0.539, round(B3.net_bpp.mean(), 3)); ck('Table 13: std (bpp)', 0.164, round(B3.net_bpp.std(), 3))
        ck('Table 13: HH-only mean (bpp)', 0.170, round(BH.net_bpp.mean(), 3))
        for n, v in dict(Airplane=0.692, Baboon=0.426, Boat=0.525, Goldhill=0.590, Peppers=0.469).items(): ck(f'Table 12: net rate {n}', v, round(B3[(B3.set == 'SIPI5') & (B3.image == n)].net_bpp.iloc[0], 3))
    if g('RDv') is not None: ck('Sec. 5.4: rate-distortion runs, all bit-exact', '1,150 / 1,150', f'{int(RDv.exact.sum()):,} / {len(RDv):,}')
    if g('R3') is not None: ck('Table 14: PSNR at 0.10 bpp, all images (dB)', 51.77, round(R3[R3.rate == 0.10].psnr.mean(), 2))
    if g('MX') is not None: ck('Table 15: PSNR at maximum rate (dB)', 39.55, round(MX.psnr.mean(), 2))
    if g('BL') is not None: ck('Table 15: Puteaux & Puech max rate (bpp)', 0.952, round(BL[BL.method == 'Puteaux2018'].rate.mean(), 3))
    if g('S5') is not None:
        ck('Table 18: mean PSNR cover/marked (dB)', 43.88, round(S5.psnr_marked.mean(), 2)); ck('Table 19: mean entropy, encrypted', 7.9993, round(S5.H_enc.mean(), 4))
        ck('Table 19: mean local entropy (published column, random block positions)', 7.9029, round(S5.H_local_pos.mean(), 4)); ck('Table 19: mean χ²', 268.9, round(S5.chi2.mean(), 1))
        ck('Sec. 5.7: local entropy over 79 ciphertexts (non-overlapping)', 7.90271, round(SEC.H_local.mean(), 5))
        ck('Sec. 5.7: χ² below 293.25 (ciphertexts / random)', '75/79 / 78/79', f'{int((SEC.chi2 < 293.25).sum())}/79 / {int((SEC.chi2_rand < 293.25).sum())}/79')
        ck('Table 21: mean NPCR (%)', 99.611, round(S5.NPCR.mean(), 3)); ck('Table 21: mean UACI (%)', 33.466, round(S5.UACI.mean(), 3))
        ck('Table 22: mean message BER with K_e only', 0.502, round(S5.ke_ber.mean(), 3))
    if g('T3ROWS') is not None:
        exp3 = [(+0.6964, 0.0, 0), (-0.6029, 78.6, 3), (-0.3467, 79.6, 1), (-0.5835, 24.4, 14), (-3.6774, 42.8, 9), (+0.8702, 0.0, 0)]
        for (nm_, _, mn_, pc_, w_), (emn, epc, ew) in zip(T3ROWS, exp3):
            ck(f'Table 3 (revised): min λ1, {nm_}', emn, float(mn_), 0.01); ck(f'Table 3 (revised): % range with λ1 <= 0, {nm_}', epc, float(pc_.split()[0]), 0.2); ck(f'Table 3 (revised): intervals, {nm_}', ew, int(w_))
    if g('FOB_CHECK_OK') is not None: ck('Sec. 4.2: correction r = |I - I_p|, direction from I_p, exact restoration', True, FOB_CHECK_OK)
    if g('KEYBYTE_CHECK_OK') is not None: ck('Eq. (20): one key-byte conversion everywhere', True, KEYBYTE_CHECK_OK)
    if g('kbits') is not None: ck('Eq. (30): effective key space (bits)', 255, int(np.floor(kbits)))
    if g('SCAN') is not None: ck('Sec. 3.2.2: Lyapunov scan points', 27183, SCAN['points']); ck('Sec. 3.2.2: hyperchaotic fraction (%)', 18.8, round(100 * SCAN['frac_hyper'], 1))
    if g('res') is not None and 128 in res: ck('Sec. 5.7.6: CPA queries, 128 x 128', 16385, res[128]['queries'])
    table(pd.DataFrame(chk, columns=['Quantity', 'Paper', 'This run', 'Status']), 'Check_against_paper', 'Reproduced values against the published values')
else: print("not in 'paper' mode: nothing to compare")

## 16. All outputs

In [ ]:
print(f'{len(SAVED)} files written to {OUT}:')
for f in SAVED: print('  ', os.path.relpath(f, OUT))
zipf = shutil.make_archive(OUT, 'zip', OUT); print('archive:', zipf)
if IN_COLAB:
    try:
        from google.colab import files as _f; _f.download(zipf)
    except Exception: pass